In [ ]:
# ══════════════════════════════════════════════
# PRELUDE — run this cell first
# Added by fix pass. Everything below your own code is unchanged.
# ══════════════════════════════════════════════

import numpy as np
import pandas as pd
import yfinance as yf
import warnings
from datetime import datetime, timezone

class Stale(RuntimeError): pass
class Unconverged(RuntimeError): pass
class TooFewObs(RuntimeError): pass


def safe_at(obj, i=-1, col=None):
    """float() on a 1-element Series is deprecated. Handles yfinance MultiIndex columns."""
    s = obj[col] if col is not None else obj
    if isinstance(s, pd.DataFrame):
        s = s.iloc[:, 0]
    a = np.asarray(s.dropna()).ravel()
    if a.size == 0:
        raise Stale("empty series")
    return float(a[i])


def safe_last(obj, col=None):
    return safe_at(obj, -1, col)


def unmute_convergence():
    """Let convergence failures through. They were being swallowed by filterwarnings('ignore')."""
    try:
        import statsmodels.tools.sm_exceptions as _s
        for _n in ("ConvergenceWarning", "EstimationWarning", "ValueWarning"):
            if hasattr(_s, _n):
                warnings.filterwarnings("always", category=getattr(_s, _n))
    except Exception:
        pass
    try:
        from arch.utility.exceptions import DataScaleWarning
        warnings.filterwarnings("always", category=DataScaleWarning)
    except Exception:
        pass


# ── CONTRACT: pin which gold series this notebook uses ──────────────
#   'front_intraday' = GC=F 30-min  (the contract you trade — DEFAULT)
#   'front_daily'    = GC=F daily   (often prints spot, not the future)
#   'spot'           = XAUUSD
LIVE_CONTRACT = "front_intraday"


def get_ctx(contract=LIVE_CONTRACT):
    if contract == "front_intraday":
        gc_df = yf.download("GC=F", period="5d", interval="30m", progress=False)
    elif contract == "front_daily":
        gc_df = yf.download("GC=F", period="1mo", interval="1d", progress=False)
    elif contract == "spot":
        gc_df = yf.download("XAUUSD=X", period="5d", interval="30m", progress=False)
    else:
        raise ValueError(contract)

    gc = safe_last(gc_df, "Close")
    gld = safe_last(yf.download("GLD", period="5d", progress=False), "Close")

    def _s(t, d):
        try:
            return safe_last(yf.download(t, period="5d", progress=False), "Close")
        except Exception:
            return d

    if not 1000 < gc < 12000:
        raise Stale(f"GC={gc:,.2f} implausible — bad fetch")
    if not 100 < gld < 1200:
        raise Stale(f"GLD={gld:,.2f} implausible — bad fetch")
    ratio = gc / gld
    if not 9.5 <= ratio <= 12.5:
        raise Stale(f"GLD->gold ratio {ratio:.3f}x outside [9.5, 12.5] — "
                    f"prices are from different dates")

    # known failure point: GC=F daily and 30m disagree
    try:
        _d = safe_last(yf.download("GC=F", period="1mo", interval="1d", progress=False), "Close")
        _i = safe_last(yf.download("GC=F", period="5d", interval="30m", progress=False), "Close")
        if abs(_d - _i) / _i > 0.005:
            print(f"  !! GC=F daily {_d:,.2f} vs 30m {_i:,.2f} — ${abs(_d-_i):,.1f} apart.")
            print(f"     Using '{contract}'. VERIFY THE SETTLE IN QUANTOWER.")
    except Exception:
        pass

    ctx = dict(gc=gc, gld=gld, ratio=ratio, contract=contract,
               asof=str(gc_df.index[-1]),
               vix=_s("^VIX", np.nan), gvz=_s("^GVZ", np.nan), rf=_s("^IRX", 4.0) / 100)
    print(f"  contract : {contract}")
    print(f"  GC {gc:>10,.2f}   GLD {gld:>8,.2f}   ratio {ratio:.4f}x   <-- NOT 10.0")
    print(f"  VIX {ctx['vix']:.2f}   GVZ {ctx['gvz']:.1f}   rf {ctx['rf']:.2%}   as of {ctx['asof']}")
    return ctx


def need_obs(n, floor, label=""):
    if n < floor:
        raise TooFewObs(f"{label}: {n} observations, need >= {floor}. Do not report this fit.")


def need_fresh(as_of, days=7, label="field"):
    age = (datetime.now(timezone.utc).date() - datetime.fromisoformat(as_of).date()).days
    if age > days:
        raise Stale(f"{label} written {as_of} ({age}d ago) — EXPIRED. Rewrite or delete it.")
    return True


def need_converged(res, states=None, label="model"):
    conv = getattr(res, "converged", None)
    if conv is None and isinstance(getattr(res, "mle_retvals", None), dict):
        conv = res.mle_retvals.get("converged", True)
    if conv is False:
        raise Unconverged(f"{label}: optimiser did not converge. Not a regime classification.")
    if states is not None:
        u, c = np.unique(np.asarray(states), return_counts=True)
        if len(u) < 2:
            raise Unconverged(f"{label}: {len(u)} distinct state — a flat line, not regimes.")
        if c.min() / c.sum() < 0.05:
            raise Unconverged(f"{label}: minority state is {c.min()/c.sum():.1%} — "
                              f"outlier detector, not a regime model.")


def kelly_cap(f, frac=0.25, cap=0.20):
    out = float(np.clip(f * frac, -cap, cap))
    if abs(f) > 1:
        print(f"  ! raw f*={f:.2f} implies {f*100:.0f}% of capital (small-sample artefact). "
              f"Using {out:.1%}.")
    return out


LIVE_CTX   = get_ctx()
LIVE_RATIO = LIVE_CTX["ratio"]   # use instead of 10
LIVE_GC    = LIVE_CTX["gc"]

# NOTE: named LIVE_* on purpose — GOLD is your hex colour in 15 cells,
#       and RATIO / CONTRACT are already used in cells 44-45.


In [ ]:

# ══════════════════════════════════════════════════════════════════════
# COT AUTO-FETCH — CFTC Disaggregated, Futures-and-Options COMBINED
# Source: https://www.cftc.gov/MarketReports/CommitmentsofTraders/index.htm
#
# NOTE: your old code pulled f_year.txt from fut_disagg_* = FUTURES ONLY,
#       while every dashboard header said "Options and Futures Combined".
#       This uses com_disagg_* -> c_year.txt = actually COMBINED.
#
# The annual zip is rewritten by CFTC every Friday ~15:30 ET, so pulling
# the current year's file always gives the newest report. No manual step.
# ══════════════════════════════════════════════════════════════════════

import io, os, zipfile, requests
import numpy as np
import pandas as pd
from datetime import datetime, timedelta, timezone
from pathlib import Path

COT_CACHE = Path.home() / ".cot_cache"
COT_CACHE.mkdir(exist_ok=True)

# Disaggregated Futures-and-Options Combined, annual history
COT_HIST_URL = "https://www.cftc.gov/files/dea/history/com_disagg_txt_{year}.zip"
# Current-week snapshot (headerless; used only as a freshness cross-check)
COT_CURRENT_URL = "https://www.cftc.gov/dea/newcot/c_disagg.txt"

CFTC_CODES = {
    "gold":        "088691",
    "silver":      "084691",
    "copper":      "085692",
    "wti_crude":   "067651",
    "natgas":      "023651",
    "corn":        "002602",
    "platinum":    "076651",
    "palladium":   "075651",
}

_COT_UA = {"User-Agent": "Mozilla/5.0 (research; contact: elena)"}


def _cot_expected_report_date(now=None):
    """
    COT is Tuesday data released Friday 15:30 ET. Returns the latest
    Tuesday that should be published by now.
    """
    now = now or datetime.now(timezone.utc)
    et = now - timedelta(hours=4)                    # ET approx (EDT)
    days_since_fri = (et.weekday() - 4) % 7
    last_fri = (et - timedelta(days=days_since_fri)).replace(
        hour=15, minute=30, second=0, microsecond=0)
    if et < last_fri:
        last_fri -= timedelta(days=7)
    return (last_fri - timedelta(days=3)).date()     # the Tuesday it covers


def _cot_download_year(year, force=False):
    """Download and cache one year of combined disaggregated data."""
    cache = COT_CACHE / f"com_disagg_{year}.parquet"
    stamp = COT_CACHE / f"com_disagg_{year}.stamp"

    # current year: refresh if cache older than 12h; past years never change
    fresh = False
    if cache.exists() and not force:
        if year < datetime.now().year:
            fresh = True
        elif stamp.exists():
            age_h = (datetime.now().timestamp() - float(stamp.read_text())) / 3600
            fresh = age_h < 12
    if fresh:
        return pd.read_parquet(cache)

    url = COT_HIST_URL.format(year=year)
    r = requests.get(url, timeout=60, headers=_COT_UA)
    r.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        names = z.namelist()
        inner = next((n for n in names if n.lower().endswith((".txt", ".csv"))), None)
        if inner is None:
            raise RuntimeError(f"{year}: no txt/csv in zip — got {names}")
        if inner.lower().startswith("f_"):
            raise RuntimeError(
                f"{year}: zip contains '{inner}' (f_ = FUTURES ONLY). "
                f"Expected 'c_year.txt' from com_disagg_*. Wrong URL.")
        with z.open(inner) as fh:
            df = pd.read_csv(fh, low_memory=False)

    df.columns = [c.strip() for c in df.columns]
    cache.parent.mkdir(exist_ok=True)
    df.to_parquet(cache, index=False)
    stamp.write_text(str(datetime.now().timestamp()))
    print(f"    downloaded {year}  ({inner}, {len(df):,} rows)")
    return df


def _cot_date_col(df):
    for c in ("Report_Date_as_YYYY-MM-DD", "Report_Date_as_MM_DD_YYYY",
              "As_of_Date_In_Form_YYMMDD", "Report_Date"):
        if c in df.columns:
            return c
    raise KeyError(f"no date column found in {list(df.columns)[:12]}")


def _cot_parse_dates(s, col):
    if "YYMMDD" in col:
        return pd.to_datetime(s.astype(str).str.zfill(6), format="%y%m%d", errors="coerce")
    return pd.to_datetime(s, errors="coerce")


def fetch_cot(market="gold", years=3, force=False, verbose=True):
    """
    Returns a tidy weekly DataFrame for one market, newest last:

        date, open_interest,
        prod_long, prod_short, prod_net,
        swap_long, swap_short, swap_net,
        mm_long,   mm_short,   mm_net,
        other_net, comm_net (prod+swap),
        mm_index, comm_index   (156-week rolling 0-100 Williams index)

    Raises on anything implausible rather than returning zeros.
    """
    code = CFTC_CODES.get(market.lower(), market)
    this_year = datetime.now().year
    frames = []
    if verbose:
        print(f"  CFTC Disaggregated · Futures-and-Options COMBINED · code {code}")

    for y in range(this_year - years + 1, this_year + 1):
        try:
            frames.append(_cot_download_year(y, force=force))
        except Exception as e:
            print(f"    ! {y}: {type(e).__name__}: {e}")
    if not frames:
        raise RuntimeError("no COT data retrieved — check network / CFTC availability")

    raw = pd.concat(frames, ignore_index=True)

    code_col = next((c for c in ("CFTC_Contract_Market_Code", "CFTC_Contract_Market_Code_Quotes")
                     if c in raw.columns), None)
    if code_col is None:
        raise KeyError("no CFTC_Contract_Market_Code column")

    m = raw[raw[code_col].astype(str).str.strip().str.zfill(6) == code].copy()
    if m.empty:
        names = raw["Market_and_Exchange_Names"].dropna().unique()[:5]
        raise ValueError(f"code {code} not found. Sample markets: {list(names)}")

    dcol = _cot_date_col(m)
    m["date"] = _cot_parse_dates(m[dcol], dcol)
    m = m.dropna(subset=["date"]).sort_values("date")
    m = m.drop_duplicates(subset=["date"], keep="last")

    def col(*cands):
        for c in cands:
            if c in m.columns:
                return pd.to_numeric(m[c], errors="coerce")
        raise KeyError(f"none of {cands} present")

    out = pd.DataFrame({
        "date":          m["date"].values,
        "open_interest": col("Open_Interest_All"),
        "prod_long":     col("Prod_Merc_Positions_Long_All"),
        "prod_short":    col("Prod_Merc_Positions_Short_All"),
        "swap_long":     col("Swap_Positions_Long_All"),
        "swap_short":    col("Swap__Positions_Short_All", "Swap_Positions_Short_All"),
        "mm_long":       col("M_Money_Positions_Long_All"),
        "mm_short":      col("M_Money_Positions_Short_All"),
        "other_long":    col("Other_Rept_Positions_Long_All"),
        "other_short":   col("Other_Rept_Positions_Short_All"),
    }).reset_index(drop=True)

    out["prod_net"]  = out.prod_long  - out.prod_short
    out["swap_net"]  = out.swap_long  - out.swap_short
    out["mm_net"]    = out.mm_long    - out.mm_short
    out["other_net"] = out.other_long - out.other_short
    out["comm_net"]  = out.prod_net + out.swap_net     # producers + swap dealers

    w = min(156, len(out))
    for c in ("mm_net", "comm_net", "swap_net"):
        lo = out[c].rolling(w, min_periods=20).min()
        hi = out[c].rolling(w, min_periods=20).max()
        out[c.replace("_net", "_index")] = 100 * (out[c] - lo) / (hi - lo).replace(0, np.nan)

    # ---- validation: fail loud rather than score a broken parse ----------
    last = out.iloc[-1]
    if last.open_interest < 10_000:
        raise ValueError(f"open interest {last.open_interest:,.0f} implausible — bad parse")
    for f in ("prod_net", "swap_net", "mm_net"):
        if last[f] == 0:
            raise ValueError(f"{f} parsed as exactly 0 — column mismatch, not flat positioning")

    expected = _cot_expected_report_date()
    lag = (expected - last.date.date()).days
    if lag > 7:
        print(f"    !! newest report {last.date.date()} but {expected} expected "
              f"({lag}d stale). CFTC may be delayed, or cache is stale — force=True to refresh.")
    elif verbose:
        print(f"    latest report : {last.date.date()}  (current)")

    if verbose:
        print(f"    rows          : {len(out)}  [{out.date.min().date()} -> {out.date.max().date()}]")
        print(f"    OI            : {last.open_interest:>10,.0f}")
        print(f"    Managed Money : {last.mm_net:>+10,.0f}   index {last.mm_index:5.1f}/100")
        print(f"    Swap Dealers  : {last.swap_net:>+10,.0f}   index {last.swap_index:5.1f}/100")
        print(f"    Prod/Merch    : {last.prod_net:>+10,.0f}")
        print(f"    Commercial    : {last.comm_net:>+10,.0f}   index {last.comm_index:5.1f}/100")

    return out


def cot_signal(df, extreme_lo=20, extreme_hi=80):
    """Williams rule: need BOTH commercial and managed-money at an extreme."""
    r = df.iloc[-1]
    c, m = r.comm_index, r.mm_index
    if np.isnan(c) or np.isnan(m):
        return "INSUFFICIENT HISTORY", 0
    if c >= extreme_hi and m <= extreme_lo:
        return "BULLISH — commercials long, specs washed out", +1
    if c <= extreme_lo and m >= extreme_hi:
        return "BEARISH — commercials short, specs crowded", -1
    return f"NEUTRAL — no extreme (comm {c:.0f}, mm {m:.0f})", 0


In [ ]:
%matplotlib inline
!pip install yfinance scipy statsmodels matplotlib numpy pandas --quiet


Gold Weekly Prep — Sunday Check
¶
Organized to mirror 
gold_weekly_prep_checklist.svg
: Tier 1 Positioning -> Tier 2 Macro Triggers -> Tier 3 Options/Derivatives -> Tier 4 Technical Structure -> Wild Cards -> Additional Research.

Tier 1 - Positioning
¶
Who is where, and how extreme

In [ ]:
# =============================================================================
# COT MULTI-MARKET DASHBOARD  |  Gold Context Suite — auto-updated
# All data: Options and Futures Combined, latest available CFTC report
# Source: CFTC Commitments of Traders Reports (Socrata API, live)
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import rcParams
import warnings
warnings.filterwarnings('ignore'); unmute_convergence()

rcParams.update({
    'figure.facecolor':'#0d0d0d','axes.facecolor':'#111111','axes.edgecolor':'#2a2a2a',
    'axes.labelcolor':'#cccccc','axes.grid':True,'grid.color':'#1e1e1e',
    'grid.linewidth':0.5,'xtick.color':'#888888','ytick.color':'#888888',
    'text.color':'#cccccc','legend.facecolor':'#1a1a1a','legend.edgecolor':'#333333',
    'font.family':'monospace','font.size':9,
})
GOLD='#C9A84C'; GREEN='#2ecc71'; RED='#e74c3c'; BLUE='#3498db'; GREY='#444444'
ORANGE='#e67e22'; CYAN='#1abc9c'

# =============================================================================
# DATA — auto-fetched from CFTC Commitments of Traders (live)
#   - Disaggregated F&O Combined (Gold/Silver/Copper/Crude) : dataset kh3c-gbw2
#   - TFF Financial F&O Combined (USD/Bonds/FX/Equities)    : dataset yw9f-hn96
# Falls back to last-known cached snapshot (26 May 2026) if offline.
# =============================================================================

import urllib.request, json as _json

CFTC_DISAGG = 'kh3c-gbw2'
CFTC_TFF    = 'yw9f-hn96'

def _cftc_fetch(dataset, code, n=2):
    """Pull the latest n COT reports for a given contract code. Returns None on failure."""
    url = (f"https://publicreporting.cftc.gov/resource/{dataset}.json"
           f"?cftc_contract_market_code={code}"
           f"&$order=report_date_as_yyyy_mm_dd%20DESC&$limit={n}")
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=20) as resp:
            data = _json.load(resp)
            return data if data else None
    except Exception as e:
        print(f"  [WARN] CFTC live fetch failed for code {code}: {e}")
        return None

def _disagg_rows(code, fallback):
    recs = _cftc_fetch(CFTC_DISAGG, code)
    if not recs:
        return fallback
    rows = []
    try:
        for r in recs:
            rows.append((
                r['report_date_as_yyyy_mm_dd'][:10],
                int(float(r.get('m_money_positions_long_all', 0))),
                int(float(r.get('m_money_positions_short_all', 0))),
                int(float(r.get('swap_positions_long_all', 0))),
                int(float(r.get('swap__positions_short_all', 0))),
                int(float(r.get('prod_merc_positions_long_all', 0))),
                int(float(r.get('prod_merc_positions_short_all', 0))),
                int(float(r.get('open_interest_all', 0))),
            ))
    except (KeyError, ValueError, TypeError):
        return fallback
    return rows if rows else fallback

def _tff_rows(code, fallback):
    recs = _cftc_fetch(CFTC_TFF, code)
    if not recs:
        return fallback
    rows = []
    try:
        for r in recs:
            rows.append((
                r['report_date_as_yyyy_mm_dd'][:10],
                int(float(r.get('lev_money_positions_long', 0))),
                int(float(r.get('lev_money_positions_short', 0))),
                int(float(r.get('asset_mgr_positions_long', 0))),
                int(float(r.get('asset_mgr_positions_short', 0))),
                int(float(r.get('dealer_positions_long', 0))),
                int(float(r.get('dealer_positions_short', 0))),
                int(float(r.get('open_interest_all', 0))),
            ))
    except (KeyError, ValueError, TypeError):
        return fallback
    return rows if rows else fallback

# ── CFTC contract market codes ───────────────────────────────────────────────
CODE_GOLD   = '088691'   # COMEX Gold
CODE_SILVER = '084691'   # COMEX Silver
CODE_COPPER = '085692'   # COMEX Copper
CODE_CRUDE  = '06741Q'   # ICE WTI 1st line
CODE_USD    = '098662'   # ICE USD Index
CODE_B10Y   = '043602'   # CBT 10Y Note
CODE_B30Y   = '020604'   # CBT Ultra 30Y Bond
CODE_EUR    = '099741'   # CME Euro FX
CODE_JPY    = '097741'   # CME Japanese Yen
CODE_SP500  = '13874A'   # CME S&P500 Consolidated
CODE_NASDAQ = '20974A'   # CME Nasdaq-100 Consolidated

# ── Cached fallback snapshot (26 May 2026 report — used only if API call fails) ──
gold_data = _disagg_rows(CODE_GOLD, [
    ('2026-05-26', 124534, 27603,  29746, 189568,  15824,  38531, 455126),
    ('2026-05-05', 124667, 29003,  28715, 200373,  18823,  41173, 526987),
])
silver_data = _disagg_rows(CODE_SILVER, [
    ('2026-05-26', 17279,  7035, 19729, 42080,  2299, 22042, 120844),
    ('2026-05-05', 16203,  5262, 19629, 43049,  3107, 20578, 117570),
])
copper_data = _disagg_rows(CODE_COPPER, [
    ('2026-05-26', 87565, 15591, 53688, 27659, 25579, 136060, 293364),
    ('2026-05-05', 80296, 17152, 51098, 21481, 25776, 127147, 248308),
])
crude_data = _disagg_rows(CODE_CRUDE, [
    ('2026-05-26',      0,     0,  1173, 13821, 21101,  8049,  41813),
    ('2026-05-05',  53816, 29352,     0,     0,     0,     0,  217942),
])
usd_data = _tff_rows(CODE_USD, [
    ('2026-05-26', 11352, 23925, 18222,  2053,  3791, 10773, 42581),
    ('2026-05-05',  7403, 13261, 12659,  2084,  3503, 10473, 32684),
])
bond10y_data = _tff_rows(CODE_B10Y, [
    ('2026-05-26',  397559, 2312319, 2935568,  884590, 139988,  615181, 7343368),
    ('2026-05-05',  371240, 2333193, 3145515,   996018, 217342,  584429, 6579709),
])
bond30y_data = _tff_rows(CODE_B30Y, [
    ('2026-05-26',   83389,  953834, 1597085,  530680,  26747,  218757, 2538106),
    ('2026-05-05',  108502,  406760, 1078137,  644600,  22866,  251056, 2057727),
])
eurofx_data = _tff_rows(CODE_EUR, [
    ('2026-05-26',  87284, 113595, 465802, 167674,  50744, 368492, 962447),
    ('2026-05-05',   1037,    686,   6506,  12651,   3454,   2763,  20946),
])
jpy_data = _tff_rows(CODE_JPY, [
    ('2026-05-26',  73803, 160052,  70293, 126569, 118779,  33933, 469765),
    ('2026-05-05',  78592, 139932,  69505,  80158,  72780,  44648, 387769),
])
sp500_data = _tff_rows(CODE_SP500, [
    ('2026-05-26', 152250,  599720, 1210008, 200994, 166668,  843674, 3122371),
    ('2026-05-05', 139233,  536054, 1206217, 195775, 126998,  837397, 2929948),
])
nasdaq_data = _tff_rows(CODE_NASDAQ, [
    ('2026-05-26',  36055, 105230, 118313,  32808,  62040,  93577, 380759),
    ('2026-05-05',  54097,  87178, 117165,  32634,  53694, 115788, 332505),
])

# =============================================================================
# BUILD DATAFRAMES
# =============================================================================

def make_disagg(rows):
    df = pd.DataFrame(rows, columns=['date','mm_long','mm_short','swap_long','swap_short','prod_long','prod_short','oi'])
    df['date'] = pd.to_datetime(df['date'])
    df.sort_values('date', inplace=True)
    df.set_index('date', inplace=True)
    df['mm_net']       = df['mm_long']   - df['mm_short']
    df['swap_net']     = df['swap_long'] - df['swap_short']
    df['prod_net']     = df['prod_long'] - df['prod_short']
    df['mm_net_pct']   = df['mm_net']   / df['oi'] * 100
    df['swap_net_pct'] = df['swap_net'] / df['oi'] * 100
    df['prod_net_pct'] = df['prod_net'] / df['oi'] * 100
    return df

def make_financial(rows):
    df = pd.DataFrame(rows, columns=['date','lev_long','lev_short','asset_long','asset_short','dealer_long','dealer_short','oi'])
    df['date'] = pd.to_datetime(df['date'])
    df.sort_values('date', inplace=True)
    df.set_index('date', inplace=True)
    df['lev_net']        = df['lev_long']    - df['lev_short']
    df['asset_net']      = df['asset_long']  - df['asset_short']
    df['dealer_net']     = df['dealer_long'] - df['dealer_short']
    df['lev_net_pct']    = df['lev_net']    / df['oi'] * 100
    df['asset_net_pct']  = df['asset_net']  / df['oi'] * 100
    df['dealer_net_pct'] = df['dealer_net'] / df['oi'] * 100
    return df

gold   = make_disagg(gold_data)
silver = make_disagg(silver_data)
copper = make_disagg(copper_data)
crude  = make_disagg(crude_data)
usd    = make_financial(usd_data)
b10y   = make_financial(bond10y_data)
b30y   = make_financial(bond30y_data)
eurofx = make_financial(eurofx_data)
jpy    = make_financial(jpy_data)
sp500  = make_financial(sp500_data)
nasdaq = make_financial(nasdaq_data)

# =============================================================================
# LATEST SNAPSHOTS
# =============================================================================
g  = gold.iloc[-1]
si = silver.iloc[-1]
cu = copper.iloc[-1]
cr = crude.iloc[-1]
us = usd.iloc[-1]
b1 = b10y.iloc[-1]
b3 = b30y.iloc[-1]
eu = eurofx.iloc[-1]
jp = jpy.iloc[-1]
sp = sp500.iloc[-1]
nq = nasdaq.iloc[-1]

# =============================================================================
# SIGNAL FUNCTIONS
# =============================================================================

def gold_signal(mm_pct, swap_pct):
    if mm_pct > 20 and swap_pct < -20: return ('CROWDED LONG  ⚠  flush risk',         RED)
    if mm_pct < 5  and swap_pct > -5:  return ('WASHED OUT   ✦  contrarian bull',      GREEN)
    if mm_pct > 15:                    return ('MM elevated — watch for squeeze',       ORANGE)
    if swap_pct < -25:                 return ('Swap very short — covering = bullish',  CYAN)
    return ('Neutral positioning',                                                       GREY)

def silver_signal(mm_pct, gold_mm_pct):
    if mm_pct > 5 and gold_mm_pct > 5: return ('Both metals bid — PM conviction ✦',    GREEN)
    if mm_pct < 2:                     return ('Silver lagging — weak PM confirmation', ORANGE)
    return ('Silver positioning neutral',                                                GREY)

def copper_signal(mm_pct):
    if mm_pct > 20: return ('Strong risk-on — safe haven suppressed ⚠',   ORANGE)
    if mm_pct < 5:  return ('Weak growth bid — stagflation risk → gold +', GREEN)
    return ('Copper neutral',                                                GREY)

def crude_signal(prod_pct):
    # ICE WTI 1st line: MM=0, use prod net as directional proxy
    if prod_pct > 10: return ('Prod net long crude — supply confidence',          GREY)
    if prod_pct < -5: return ('Prod net short crude — deflation/demand risk ⚠',  ORANGE)
    return ('Crude positioning neutral',                                            GREY)

def usd_signal(lev_pct):
    if lev_pct < -10: return ('Lev funds NET SHORT USD  ✦  gold tailwind', GREEN)
    if lev_pct >  10: return ('Lev funds net long USD  ⚠  gold headwind',  RED)
    return ('USD neutral',                                                   GREY)

def bond_signal(lev_pct, label='Bond'):
    if lev_pct < -20: return (f'{label} lev heavily short → rate fear → gold ⚠', RED)
    if lev_pct >  5:  return (f'{label} lev net long → rate calm → neutral',      GREY)
    return (f'{label} positioning mixed',                                           GREY)

def euro_signal(lev_pct, asset_pct):
    if lev_pct > 5 and asset_pct > 5:  return ('Both funds long EUR → USD weak → gold +',  GREEN)
    if lev_pct < -5 and asset_pct < 0: return ('Funds short EUR → USD strong → gold −',    RED)
    return ('EUR positioning mixed',                                                          GREY)

def jpy_signal(lev_pct):
    if lev_pct < -10: return ('Funds net SHORT JPY — carry on → risk-on ⚠',    ORANGE)
    if lev_pct >  5:  return ('Funds net long JPY — risk-off → gold aligned',   GREEN)
    return ('JPY positioning neutral',                                             GREY)

def equity_signal(lev_pct, label='Equity'):
    if lev_pct < -10: return (f'{label} lev funds net short → rotation to gold ✦',   GREEN)
    if lev_pct < -5:  return (f'{label} lev funds reducing longs → mild risk-off',    ORANGE)
    if lev_pct >  5:  return (f'{label} lev funds long → risk-on, gold neglected ⚠', ORANGE)
    return (f'{label} positioning neutral',                                             GREY)

gold_sig, gold_col = gold_signal(float(g['mm_net_pct']),    float(g['swap_net_pct']))
si_sig,   si_col   = silver_signal(float(si['mm_net_pct']), float(g['mm_net_pct']))
cu_sig,   cu_col   = copper_signal(float(cu['mm_net_pct']))
cr_sig,   cr_col   = crude_signal(float(cr['prod_net_pct']))
usd_sig,  usd_col  = usd_signal(float(us['lev_net_pct']))
b10_sig,  b10_col  = bond_signal(float(b1['lev_net_pct']), '10Y')
b30_sig,  b30_col  = bond_signal(float(b3['lev_net_pct']), '30Y')
eu_sig,   eu_col   = euro_signal(float(eu['lev_net_pct']), float(eu['asset_net_pct']))
jp_sig,   jp_col   = jpy_signal(float(jp['lev_net_pct']))
sp_sig,   sp_col   = equity_signal(float(sp['lev_net_pct']), 'SP500')
nq_sig,   nq_col   = equity_signal(float(nq['lev_net_pct']), 'NQ100')

# =============================================================================
# PRINT REPORT
# =============================================================================
W = 72
date_str = gold.index[-1].strftime('%d %b %Y')

def section(title, lines, signal):
    print(f"\n  {title}")
    print(f"  {'─'*65}")
    for lbl, val in lines:
        print(f"  {lbl:<22}: {val}")
    print(f"\n  Signal ► {signal}")

print("=" * W)
print(f"  COT MULTI-MARKET GOLD CONTEXT DASHBOARD  |  {date_str}")
print(f"  Options and Futures Combined")
print("=" * W)

section("GOLD (COMEX GC — Disaggregated O+F)", [
    ("OI",           f"{int(g['oi']):,} contracts"),
    ("MM Net",       f"{int(g['mm_net']):+,}  ({float(g['mm_net_pct']):+.1f}% OI)"),
    ("Swap Net",     f"{int(g['swap_net']):+,}  ({float(g['swap_net_pct']):+.1f}% OI)"),
    ("Producer Net", f"{int(g['prod_net']):+,}  ({float(g['prod_net_pct']):+.1f}% OI)"),
], gold_sig)

section("SILVER (COMEX SI — Disaggregated O+F)", [
    ("OI",       f"{int(si['oi']):,}"),
    ("MM Net",   f"{int(si['mm_net']):+,}  ({float(si['mm_net_pct']):+.1f}% OI)"),
    ("Swap Net", f"{int(si['swap_net']):+,}  ({float(si['swap_net_pct']):+.1f}% OI)"),
], si_sig)

section("COPPER (COMEX HG — Disaggregated O+F)", [
    ("OI",       f"{int(cu['oi']):,}"),
    ("MM Net",   f"{int(cu['mm_net']):+,}  ({float(cu['mm_net_pct']):+.1f}% OI)"),
    ("Swap Net", f"{int(cu['swap_net']):+,}  ({float(cu['swap_net_pct']):+.1f}% OI)"),
], cu_sig)

section("CRUDE WTI 1st Line (ICE — Disaggregated O+F)", [
    ("OI",           f"{int(cr['oi']):,}"),
    ("MM Net",       f"{int(cr['mm_net']):+,}  ({float(cr['mm_net_pct']):+.1f}% OI)"),
    ("Swap Net",     f"{int(cr['swap_net']):+,}  ({float(cr['swap_net_pct']):+.1f}% OI)"),
    ("Producer Net", f"{int(cr['prod_net']):+,}  ({float(cr['prod_net_pct']):+.1f}% OI)"),
], cr_sig)

section("USD INDEX — ICE (TFF O+F)", [
    ("OI",            f"{int(us['oi']):,}"),
    ("Lev Funds Net", f"{int(us['lev_net']):+,}  ({float(us['lev_net_pct']):+.1f}% OI)"),
    ("Asset Mgr Net", f"{int(us['asset_net']):+,}  ({float(us['asset_net_pct']):+.1f}% OI)"),
    ("Dealer Net",    f"{int(us['dealer_net']):+,}  ({float(us['dealer_net_pct']):+.1f}% OI)"),
], usd_sig)

section("US 10Y NOTE — CBT (TFF O+F)", [
    ("OI",            f"{int(b1['oi']):,}"),
    ("Lev Funds Net", f"{int(b1['lev_net']):+,}  ({float(b1['lev_net_pct']):+.1f}% OI)"),
    ("Asset Mgr Net", f"{int(b1['asset_net']):+,}  ({float(b1['asset_net_pct']):+.1f}% OI)"),
    ("Dealer Net",    f"{int(b1['dealer_net']):+,}  ({float(b1['dealer_net_pct']):+.1f}% OI)"),
], b10_sig)

section("ULTRA UST 30Y BOND — CBT (TFF O+F)", [
    ("OI",            f"{int(b3['oi']):,}"),
    ("Lev Funds Net", f"{int(b3['lev_net']):+,}  ({float(b3['lev_net_pct']):+.1f}% OI)"),
    ("Asset Mgr Net", f"{int(b3['asset_net']):+,}  ({float(b3['asset_net_pct']):+.1f}% OI)"),
    ("Dealer Net",    f"{int(b3['dealer_net']):+,}  ({float(b3['dealer_net_pct']):+.1f}% OI)"),
], b30_sig)

section("EURO FX — CME (TFF O+F)", [
    ("OI",            f"{int(eu['oi']):,}"),
    ("Lev Funds Net", f"{int(eu['lev_net']):+,}  ({float(eu['lev_net_pct']):+.1f}% OI)"),
    ("Asset Mgr Net", f"{int(eu['asset_net']):+,}  ({float(eu['asset_net_pct']):+.1f}% OI)"),
    ("Dealer Net",    f"{int(eu['dealer_net']):+,}  ({float(eu['dealer_net_pct']):+.1f}% OI)"),
], eu_sig)

section("JAPANESE YEN — CME (TFF O+F)", [
    ("OI",            f"{int(jp['oi']):,}"),
    ("Lev Funds Net", f"{int(jp['lev_net']):+,}  ({float(jp['lev_net_pct']):+.1f}% OI)"),
    ("Asset Mgr Net", f"{int(jp['asset_net']):+,}  ({float(jp['asset_net_pct']):+.1f}% OI)"),
    ("Dealer Net",    f"{int(jp['dealer_net']):+,}  ({float(jp['dealer_net_pct']):+.1f}% OI)"),
], jp_sig)

section("S&P 500 Consolidated — CME (TFF O+F)", [
    ("OI",            f"{int(sp['oi']):,}"),
    ("Lev Funds Net", f"{int(sp['lev_net']):+,}  ({float(sp['lev_net_pct']):+.1f}% OI)"),
    ("Asset Mgr Net", f"{int(sp['asset_net']):+,}  ({float(sp['asset_net_pct']):+.1f}% OI)"),
    ("Dealer Net",    f"{int(sp['dealer_net']):+,}  ({float(sp['dealer_net_pct']):+.1f}% OI)"),
], sp_sig)

section("NASDAQ-100 Consolidated — CME (TFF O+F)", [
    ("OI",            f"{int(nq['oi']):,}"),
    ("Lev Funds Net", f"{int(nq['lev_net']):+,}  ({float(nq['lev_net_pct']):+.1f}% OI)"),
    ("Asset Mgr Net", f"{int(nq['asset_net']):+,}  ({float(nq['asset_net_pct']):+.1f}% OI)"),
    ("Dealer Net",    f"{int(nq['dealer_net']):+,}  ({float(nq['dealer_net_pct']):+.1f}% OI)"),
], nq_sig)

# =============================================================================
# MACRO CONFLUENCE SUMMARY
# =============================================================================
print(f"\n{'='*W}")
print(f"  MACRO CONFLUENCE SUMMARY  |  Gold Directional Bias")
print(f"{'='*W}")

factors = [
    ('✦' if float(g['mm_net_pct']) < 15 else '⚠',
     f"Gold MM {float(g['mm_net_pct']):+.1f}% OI — {'not crowded, room to build' if float(g['mm_net_pct']) < 15 else 'elevated, flush risk'}",
     GREEN if float(g['mm_net_pct']) < 15 else ORANGE),

    ('✦' if float(g['swap_net_pct']) < -25 else '–',
     f"Swap dealers {float(g['swap_net_pct']):+.1f}% OI — {'extreme short, structural support' if float(g['swap_net_pct']) < -25 else 'moderate short'}",
     GREEN if float(g['swap_net_pct']) < -25 else GREY),

    ('✦' if float(si['mm_net_pct']) > 5 else '⚠',
     f"Silver MM {float(si['mm_net_pct']):+.1f}% OI — {'confirms PM bid' if float(si['mm_net_pct']) > 5 else 'lagging, weak PM confirmation'}",
     GREEN if float(si['mm_net_pct']) > 5 else ORANGE),

    ('⚠' if float(cu['mm_net_pct']) > 20 else ('✦' if float(cu['mm_net_pct']) < 5 else '–'),
     f"Copper MM {float(cu['mm_net_pct']):+.1f}% OI — {'risk-on suppresses gold' if float(cu['mm_net_pct']) > 20 else ('stagflation narrative' if float(cu['mm_net_pct']) < 5 else 'neutral')}",
     ORANGE if float(cu['mm_net_pct']) > 20 else (GREEN if float(cu['mm_net_pct']) < 5 else GREY)),

    ('✦' if float(us['lev_net_pct']) < -10 else ('⚠' if float(us['lev_net_pct']) > 10 else '–'),
     f"USD lev funds {float(us['lev_net_pct']):+.1f}% OI — {'short dollar = gold tailwind' if float(us['lev_net_pct']) < -10 else ('long dollar = gold headwind' if float(us['lev_net_pct']) > 10 else 'neutral')}",
     GREEN if float(us['lev_net_pct']) < -10 else (RED if float(us['lev_net_pct']) > 10 else GREY)),

    ('✦' if float(eu['lev_net_pct']) > 5 else ('⚠' if float(eu['lev_net_pct']) < -5 else '–'),
     f"EUR lev funds {float(eu['lev_net_pct']):+.1f}% OI — {'long EUR = weak USD = gold +' if float(eu['lev_net_pct']) > 5 else ('short EUR = strong USD = gold −' if float(eu['lev_net_pct']) < -5 else 'neutral')}",
     GREEN if float(eu['lev_net_pct']) > 5 else (RED if float(eu['lev_net_pct']) < -5 else GREY)),

    ('✦' if float(jp['lev_net_pct']) > 5 else ('⚠' if float(jp['lev_net_pct']) < -10 else '–'),
     f"JPY lev funds {float(jp['lev_net_pct']):+.1f}% OI — {'long JPY = risk-off aligned' if float(jp['lev_net_pct']) > 5 else ('short JPY = carry on, risk appetite' if float(jp['lev_net_pct']) < -10 else 'neutral')}",
     GREEN if float(jp['lev_net_pct']) > 5 else (ORANGE if float(jp['lev_net_pct']) < -10 else GREY)),

    ('⚠' if float(b1['lev_net_pct']) < -20 else '–',
     f"10Y lev funds {float(b1['lev_net_pct']):+.1f}% OI — {'massively short bonds = rate fear' if float(b1['lev_net_pct']) < -20 else 'moderate bond short'}",
     RED if float(b1['lev_net_pct']) < -20 else GREY),

    ('⚠' if float(b3['lev_net_pct']) < -20 else '–',
     f"30Y lev funds {float(b3['lev_net_pct']):+.1f}% OI — {'massively short bonds = rate fear' if float(b3['lev_net_pct']) < -20 else 'moderate bond short'}",
     RED if float(b3['lev_net_pct']) < -20 else GREY),

    ('✦' if float(sp['lev_net_pct']) < -10 else ('⚠' if float(sp['lev_net_pct']) < -5 else '–'),
     f"SP500 lev funds {float(sp['lev_net_pct']):+.1f}% OI — {'net short = rotation to gold' if float(sp['lev_net_pct']) < -10 else ('reducing longs = mild risk-off' if float(sp['lev_net_pct']) < -5 else 'risk-on, gold neglected')}",
     GREEN if float(sp['lev_net_pct']) < -10 else (ORANGE if float(sp['lev_net_pct']) < -5 else GREY)),

    ('✦' if float(nq['lev_net_pct']) < -10 else ('⚠' if float(nq['lev_net_pct']) < -5 else '–'),
     f"NQ100 lev funds {float(nq['lev_net_pct']):+.1f}% OI — {'net short = rotation to gold' if float(nq['lev_net_pct']) < -10 else ('reducing longs = mild risk-off' if float(nq['lev_net_pct']) < -5 else 'risk-on, gold neglected')}",
     GREEN if float(nq['lev_net_pct']) < -10 else (ORANGE if float(nq['lev_net_pct']) < -5 else GREY)),
]

bull = sum(1 for f in factors if f[2] == GREEN)
bear = sum(1 for f in factors if f[2] == RED)
warn = sum(1 for f in factors if f[2] == ORANGE)

for icon, text, _ in factors:
    print(f"  {icon}  {text}")

print(f"\n  Score  :  {bull} bullish  |  {warn} caution  |  {bear} bearish  (of {len(factors)} factors)")
overall = 'BULLISH LEAN' if bull >= 5 else 'BEARISH LEAN' if bear >= 3 else 'MIXED / NEUTRAL'
print(f"  Bias   :  {overall}")
print("=" * W)

# =============================================================================
# CHART  —  3 rows x 4 cols + confluence scorecard
# =============================================================================
fig = plt.figure(figsize=(20, 15))
fig.patch.set_facecolor('#0d0d0d')
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.65, wspace=0.4)
fig.suptitle(f'COT MULTI-MARKET DASHBOARD  |  Gold Context Suite  |  {date_str}  |  O+F Combined',
             fontsize=12, color=GOLD, fontfamily='monospace', y=0.99)

def mini_bar(ax, labels, values, title):
    cols = [GREEN if v > 0 else RED for v in values]
    x = np.arange(len(labels))
    ax.bar(x, values, color=cols, width=0.45)
    ax.axhline(0, color=GREY, lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=7.5)
    ax.set_title(title, fontsize=8.5, color='#bbbbbb', pad=3)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1f}%'))
    rng = max(abs(v) for v in values) if any(values) else 1
    for i, (v, c) in enumerate(zip(values, cols)):
        ax.text(i, v + (rng * 0.06 if v >= 0 else -rng * 0.14),
                f'{v:.1f}%', ha='center', fontsize=7, color=c)

# Row 0: commodities
ax = fig.add_subplot(gs[0, 0])
mini_bar(ax, ['MM', 'Swap', 'Prod'],
         [float(g['mm_net_pct']), float(g['swap_net_pct']), float(g['prod_net_pct'])],
         'GOLD  (% OI)')

ax = fig.add_subplot(gs[0, 1])
mini_bar(ax, ['MM', 'Swap', 'Prod'],
         [float(si['mm_net_pct']), float(si['swap_net_pct']), float(si['prod_net_pct'])],
         'SILVER  (% OI)')

ax = fig.add_subplot(gs[0, 2])
mini_bar(ax, ['MM', 'Swap', 'Prod'],
         [float(cu['mm_net_pct']), float(cu['swap_net_pct']), float(cu['prod_net_pct'])],
         'COPPER  (% OI)')

ax = fig.add_subplot(gs[0, 3])
mini_bar(ax, ['MM', 'Swap', 'Prod'],
         [float(cr['mm_net_pct']), float(cr['swap_net_pct']), float(cr['prod_net_pct'])],
         'CRUDE WTI ICE  (% OI)')

# Row 1: currencies + SP500
ax = fig.add_subplot(gs[1, 0])
mini_bar(ax, ['Lev', 'Asset', 'Dealer'],
         [float(us['lev_net_pct']), float(us['asset_net_pct']), float(us['dealer_net_pct'])],
         'USD INDEX  (% OI)')

ax = fig.add_subplot(gs[1, 1])
mini_bar(ax, ['Lev', 'Asset', 'Dealer'],
         [float(eu['lev_net_pct']), float(eu['asset_net_pct']), float(eu['dealer_net_pct'])],
         'EURO FX  (% OI)')

ax = fig.add_subplot(gs[1, 2])
mini_bar(ax, ['Lev', 'Asset', 'Dealer'],
         [float(jp['lev_net_pct']), float(jp['asset_net_pct']), float(jp['dealer_net_pct'])],
         'JAPANESE YEN  (% OI)')

ax = fig.add_subplot(gs[1, 3])
mini_bar(ax, ['Lev', 'Asset', 'Dealer'],
         [float(sp['lev_net_pct']), float(sp['asset_net_pct']), float(sp['dealer_net_pct'])],
         'S&P 500  (% OI)')

# Row 2: bonds + nasdaq + confluence scorecard
ax = fig.add_subplot(gs[2, 0])
mini_bar(ax, ['Lev', 'Asset', 'Dealer'],
         [float(b1['lev_net_pct']), float(b1['asset_net_pct']), float(b1['dealer_net_pct'])],
         '10Y T-NOTE  (% OI)')

ax = fig.add_subplot(gs[2, 1])
mini_bar(ax, ['Lev', 'Asset', 'Dealer'],
         [float(b3['lev_net_pct']), float(b3['asset_net_pct']), float(b3['dealer_net_pct'])],
         'ULTRA 30Y BOND  (% OI)')

ax = fig.add_subplot(gs[2, 2])
mini_bar(ax, ['Lev', 'Asset', 'Dealer'],
         [float(nq['lev_net_pct']), float(nq['asset_net_pct']), float(nq['dealer_net_pct'])],
         'NASDAQ-100  (% OI)')

# Confluence scorecard
ax_score = fig.add_subplot(gs[2, 3])
ax_score.axis('off')
ax_score.set_facecolor('#111111')
col_map = {GREEN:'#2ecc71', RED:'#e74c3c', ORANGE:'#e67e22', GREY:'#666666', CYAN:'#1abc9c'}
ax_score.text(0.03, 0.98, 'CONFLUENCE', transform=ax_score.transAxes,
              fontsize=9, color=GOLD, va='top', fontfamily='monospace', fontweight='bold')
for i, (icon, text, col) in enumerate(factors):
    short_text = text[:38] + '..' if len(text) > 40 else text
    ax_score.text(0.03, 0.88 - i * 0.073, f'{icon} {short_text}',
                  transform=ax_score.transAxes, fontsize=6.5,
                  color=col_map.get(col, '#cccccc'), va='top', fontfamily='monospace')
ax_score.text(0.03, 0.02,
              f'B:{bull}  C:{warn}  Bear:{bear}  →  {overall}',
              transform=ax_score.transAxes, fontsize=8, color=GOLD,
              va='bottom', fontfamily='monospace', fontweight='bold')

# ── Inline display ─────────────────────────────────────
import io as _io, IPython.display as _ipyd
_buf = _io.BytesIO()
plt.savefig(_buf, dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.savefig('cot_multi_market_dashboard.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
_buf.seek(0)
_ipyd.display(_ipyd.Image(_buf.read()))
plt.close()
print(f"\n✓ Saved: cot_multi_market_dashboard.png")
print(f"  All data: Options and Futures Combined, {date_str}")


Large Spec Index
 is read directly from the COT dashboard above: the 
MM Net % OI
 figure for gold, flagged by 
gold_signal()
 as 
CROWDED LONG
 (near max long -> flush risk) or 
WASHED OUT
 (contrarian bull) - this is the Large Spec extremity check from the checklist.

In [ ]:
# =============================================================================
# GLD OPTIONS OI  |  Put/Call Ratio + Gamma Wall Detection
# =============================================================================

import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib import rcParams
import warnings; warnings.filterwarnings('ignore'); unmute_convergence()

rcParams.update({
    'figure.facecolor':'#0d0d0d','axes.facecolor':'#111111','axes.edgecolor':'#2a2a2a',
    'axes.labelcolor':'#cccccc','axes.grid':True,'grid.color':'#1e1e1e',
    'grid.linewidth':0.5,'xtick.color':'#888888','ytick.color':'#888888',
    'text.color':'#cccccc','legend.facecolor':'#1a1a1a','legend.edgecolor':'#333333',
    'font.family':'monospace','font.size':9,
})
GOLD='#C9A84C'; GREEN='#2ecc71'; RED='#e74c3c'; BLUE='#3498db'; PURPLE='#9b59b6'

print("Fetching GLD spot price ...")
gld_ticker = yf.Ticker("GLD")

# GLD = ~1/10 of gold price. Gold futures (GC=F) gives the proper $/oz price.
gold_spot_raw = yf.download('GC=F', period='5d', auto_adjust=True, progress=False)['Close']
gold_spot = float(gold_spot_raw.dropna().iloc[-1])

gld_px_raw = yf.download('GLD', period='5d', auto_adjust=True, progress=False)['Close']
gld_spot = float(gld_px_raw.dropna().iloc[-1])   # GLD ETF share price (~$420s)

print(f"  Gold (XAU/USD futures) : ${gold_spot:,.2f} per troy oz")
print(f"  GLD ETF share price    : ${gld_spot:.2f}  (≈ 1/10 oz, normal)")
print(f"  Gold futures           : ${LIVE_GC:,.2f}  (ratio {LIVE_RATIO:.4f}x)")

# Options chain uses GLD share price as the strike reference
spot = gld_spot

exps = gld_ticker.options
target_exps = exps[:4]

all_calls, all_puts = [], []
for exp in target_exps:
    try:
        chain = gld_ticker.option_chain(exp)
        c = chain.calls[['strike','openInterest','volume','impliedVolatility']].copy()
        p = chain.puts [['strike','openInterest','volume','impliedVolatility']].copy()
        c['expiry'] = exp; p['expiry'] = exp
        all_calls.append(c); all_puts.append(p)
        print(f"  ✓ Loaded expiry {exp}")
    except Exception as e:
        print(f"  ⚠ Skipped {exp}: {e}")

if not all_calls:
    raise RuntimeError("No options data returned — market may be closed or yfinance rate-limited.")

calls = pd.concat(all_calls).fillna(0)
puts  = pd.concat(all_puts).fillna(0)

# Aggregate OI by strike
call_oi = calls.groupby('strike')['openInterest'].sum()
put_oi  = puts.groupby('strike') ['openInterest'].sum()
all_strikes = sorted(set(call_oi.index) | set(put_oi.index))
call_oi = call_oi.reindex(all_strikes, fill_value=0)
put_oi  = put_oi.reindex(all_strikes,  fill_value=0)

# Filter ±20% around GLD spot
lo, hi = spot * 0.80, spot * 1.20
mask   = [lo <= s <= hi for s in all_strikes]
strikes_f = np.array(all_strikes)[mask]
call_f    = call_oi.values[mask]
put_f     = put_oi.values[mask]

# Ratios
total_call_oi  = float(calls['openInterest'].sum())
total_put_oi   = float(puts['openInterest'].sum())
total_call_vol = float(calls['volume'].sum())
total_put_vol  = float(puts['volume'].sum())
pc_oi  = total_put_oi  / (total_call_oi  + 1e-10)
pc_vol = total_put_vol / (total_call_vol + 1e-10)

# Top gamma walls = strikes with highest combined OI near spot
net_oi = call_f + put_f
top_idx = np.argsort(net_oi)[-5:][::-1]
gamma_walls = strikes_f[top_idx]

print(f"\n  ─── Options Market Summary ──────────────────────────────")
print(f"  GLD Spot               : ${spot:.2f}")
print(f"  Gold Futures Spot      : ${gold_spot:,.2f}")
print(f"  Put/Call OI Ratio      : {pc_oi:.3f}  {'(bearish skew)' if pc_oi > 1.1 else '(bullish skew)' if pc_oi < 0.8 else '(neutral)'}")
print(f"  Put/Call Vol Ratio     : {pc_vol:.3f}")
print(f"  Top Gamma Walls (GLD$) : {[f'${g:.0f}' for g in gamma_walls[:3]]}")
print(f"  Top Gamma Walls (Gold) : {[f'${g*LIVE_RATIO:,.0f}' for g in gamma_walls[:3]]}  (approx)")

# ── Chart ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle(
    f'GLD Options  //  OI by Strike  |  Gamma Walls  |  Gold=${gold_spot:,.0f}  GLD=${gld_spot:.2f}',
    fontsize=11, color=GOLD, fontfamily='monospace', y=0.98)

ax1, ax2 = axes
w = (strikes_f[1] - strikes_f[0]) * 0.38 if len(strikes_f) > 1 else 0.5

ax1.bar(strikes_f - w/2, call_f, width=w, color=GREEN, alpha=0.75, label='Call OI')
ax1.bar(strikes_f + w/2, put_f,  width=w, color=RED,   alpha=0.75, label='Put OI')
ax1.axvline(spot, color=GOLD, lw=1.5, ls='--', label=f'GLD Spot ${spot:.1f}')
for i, gw in enumerate(gamma_walls[:3]):
    ax1.axvline(gw, color=PURPLE, lw=0.9, ls=':', alpha=0.9,
                label=f'Gamma wall ${gw:.0f}' if i == 0 else '')
ax1.set_title(
    f'Call vs Put OI by Strike (±20% moneyness)  |  Purple dashed = top gamma walls',
    fontsize=9, color='#aaa', pad=3)
ax1.set_ylabel('Open Interest', fontsize=8)
ax1.set_xlabel(f'GLD Strike ($)  [multiply by ~10 for gold $/oz equivalent]', fontsize=8)
ax1.legend(fontsize=7)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))

# Net OI (call - put)
net_signed = call_f - put_f
ax2.bar(strikes_f, net_signed, width=w * 1.5,
        color=[GREEN if v > 0 else RED for v in net_signed], alpha=0.7)
ax2.axhline(0, color='#444', lw=0.8)
ax2.axvline(spot, color=GOLD, lw=1.5, ls='--', label=f'GLD Spot ${spot:.1f}')
ax2.set_title(
    f'Net Dealer Gamma Pressure (Call−Put OI)  |  P/C OI={pc_oi:.2f}  P/C Vol={pc_vol:.2f}',
    fontsize=9, color='#aaa', pad=3)
ax2.set_ylabel('Net OI  (Call − Put)', fontsize=8)
ax2.set_xlabel('GLD Strike ($)', fontsize=8)
ax2.legend(fontsize=7)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))

interp = ('Bearish skew — downside hedging dominant' if pc_oi > 1.1
          else 'Bullish skew — calls dominant' if pc_oi < 0.8
          else 'Balanced — no strong directional bias in options market')
fig.text(0.5, 0.01, f'► {interp}', ha='center', fontsize=9, color=GOLD, fontfamily='monospace')

plt.tight_layout(rect=[0, 0.03, 1, 0.96])
# ── Inline display ─────────────────────────────────────
import io as _io, IPython.display as _ipyd
_buf = _io.BytesIO()
plt.savefig(_buf, dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.savefig('gld_options_oi.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
_buf.seek(0)
_ipyd.display(_ipyd.Image(_buf.read()))
plt.close()
print("✓ Saved: gld_options_oi.png")


Tier 2 - Macro Triggers
¶
What's on the calendar, and what's the regime

In [ ]:
# =============================================================================
# WEEKLY MACRO CALENDAR  |  CPI/PCE/PPI · FOMC · NFP / jobless claims
# Auto-flags the high-impact US macro events for the current week
# =============================================================================
import pandas as pd

today       = pd.Timestamp.now().normalize()
week_start  = today - pd.Timedelta(days=today.dayofweek)   # Monday
week_end    = week_start + pd.Timedelta(days=6)            # Sunday

# Confirmed 2026 FOMC decision days (2nd day of each 2-day meeting)
FOMC_2026 = pd.to_datetime([
    '2026-01-28', '2026-03-18', '2026-04-29', '2026-06-17',
    '2026-07-29', '2026-09-16', '2026-10-28', '2026-12-09',
])

def first_friday(d):
    first = d.replace(day=1)
    return first + pd.Timedelta(days=(4 - first.weekday()) % 7)

events = []

for d in FOMC_2026:
    if week_start <= d <= week_end:
        events.append((d, 'FOMC decision (Fed funds rate)', 'HIGH'))

nfp = first_friday(week_start)
if week_start <= nfp <= week_end:
    events.append((nfp, 'NFP + unemployment rate', 'HIGH'))

# Weekly jobless claims — every Thursday
claims = week_start + pd.Timedelta(days=3)
events.append((claims, 'Initial jobless claims', 'LOW'))

# CPI/PCE/PPI cluster typically lands ~10th-15th of the month
if week_start.day <= 15 and week_end.day >= 10:
    events.append((week_start, 'CPI / PPI / PCE window (~10th-15th of month)', 'MEDIUM'))

print("=" * 60)
print(f"  WEEKLY MACRO CALENDAR  |  {week_start.date()} - {week_end.date()}")
print("=" * 60)
for d, label, importance in sorted(events, key=lambda x: x[0]):
    print(f"  [{importance:<6}] {pd.Timestamp(d).strftime('%a %d %b')}  -  {label}")
if not events:
    print("  No flagged high-impact US macro events this week.")

print("\n  Manual cross-checks:")
print("   - Exact CPI/PCE/PPI release dates: bls.gov / bea.gov release schedules")
print("   - Fed speakers: federalreserve.gov/newsevents/speeches.htm")
print("   - Geopolitical calendar: scan overnight headlines Sun evening before open")


In [ ]:
# =============================================================================
# MACRO TRIGGER DASHBOARD  |  Real Rates · DXY · Yield Curve · Correlations
# =============================================================================

import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import rcParams
import warnings; warnings.filterwarnings('ignore'); unmute_convergence()

rcParams.update({
    'figure.facecolor':'#0d0d0d','axes.facecolor':'#111111','axes.edgecolor':'#2a2a2a',
    'axes.labelcolor':'#cccccc','axes.grid':True,'grid.color':'#1e1e1e',
    'grid.linewidth':0.5,'xtick.color':'#888888','ytick.color':'#888888',
    'text.color':'#cccccc','legend.facecolor':'#1a1a1a','legend.edgecolor':'#333333',
    'font.family':'monospace','font.size':9,
})
GOLD='#C9A84C'; GREEN='#2ecc71'; RED='#e74c3c'; BLUE='#3498db'
PURPLE='#9b59b6'; GREY='#444444'; ORANGE='#e67e22'

print("Fetching macro data — downloading each ticker separately for clean mapping ...")

def safe_download(ticker, period='2y', label=None):
    """Download a single ticker and return a clean named Series."""
    raw = yf.download(ticker, period=period, auto_adjust=True, progress=False)['Close']
    raw = raw.squeeze()  # ensure Series not DataFrame
    raw.name = label or ticker
    return raw.ffill().dropna()

gold   = safe_download('GC=F',       label='Gold')
dxy    = safe_download('DX-Y.NYB',   label='DXY')
ust10y = safe_download('^TNX',       label='UST10Y')   # in % (e.g. 4.3 = 4.3%)
ust3m  = safe_download('^IRX',       label='UST3M')    # in % (e.g. 5.2 = 5.2%)
vix    = safe_download('^VIX',       label='VIX')

# Align on common dates
raw = pd.concat([gold, dxy, ust10y, ust3m, vix], axis=1).ffill().dropna()
raw.columns = ['Gold','DXY','UST10Y','UST3M','VIX']

print(f"  ✓ {len(raw)} trading days loaded  [{raw.index[0].date()} → {raw.index[-1].date()}]")

# Sanity check prints
print(f"\n  Sanity check (latest values):")
print(f"    Gold    : ${raw['Gold'].iloc[-1]:,.2f}   ← should be ~$3,000+")
print(f"    DXY     : {raw['DXY'].iloc[-1]:.2f}      ← should be ~95–115")
print(f"    10Y     : {raw['UST10Y'].iloc[-1]:.2f}%  ← should be ~3–5%")
print(f"    3M      : {raw['UST3M'].iloc[-1]:.2f}%   ← should be ~4–5%")
print(f"    VIX     : {raw['VIX'].iloc[-1]:.1f}      ← should be ~12–40")

# Log returns
rets   = np.log(raw / raw.shift(1)).dropna()
gold_r = rets['Gold']

# Rolling 63-day correlations
corr_dxy = gold_r.rolling(63).corr(rets['DXY'])
corr_ust = gold_r.rolling(63).corr(rets['UST10Y'])
corr_vix = gold_r.rolling(63).corr(rets['VIX'])

# Yield curve: 10Y minus 3M (both already in %)
curve = raw['UST10Y'] - raw['UST3M']

# DXY 1-month momentum
dxy_mom_1m = raw['DXY'].pct_change(21) * 100

latest = raw.iloc[-1]
print(f"\n  ─── Macro Snapshot [{raw.index[-1].date()}] ───────────────────")
print(f"  Gold           : ${float(latest['Gold']):,.2f}")
print(f"  DXY            : {float(latest['DXY']):.2f}  (1m chg {safe_at(dxy_mom_1m, -1):+.2f}%)")
print(f"  10Y Yield      : {float(latest['UST10Y']):.2f}%")
print(f"  3M Yield       : {float(latest['UST3M']):.2f}%")
print(f"  Yield Curve    : {safe_at(curve, -1):+.2f}bp  (10Y−3M)  {'[INVERTED]' if curve.iloc[-1] < 0 else '[NORMAL]'}")
print(f"  VIX            : {float(latest['VIX']):.1f}")
print(f"\n  ─── 63-Day Rolling Correlations with Gold ──────────────")
print(f"  Gold / DXY     : {safe_at(corr_dxy, -1):+.3f}  {'(normal inverse)' if corr_dxy.iloc[-1] < 0 else '⚠ POSITIVE — unusual decoupling'}")
print(f"  Gold / UST10Y  : {safe_at(corr_ust, -1):+.3f}")
print(f"  Gold / VIX     : {safe_at(corr_vix, -1):+.3f}  {'(fear bid active)' if corr_vix.iloc[-1] > 0.15 else '(no fear premium)'}")

# ── Dashboard ─────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle('MACRO TRIGGER DASHBOARD  //  Gold | DXY · Rates · Yield Curve · Correlations',
             fontsize=12, color=GOLD, fontfamily='monospace', y=0.98)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.35,
                        left=0.07, right=0.97, top=0.94, bottom=0.05)
start = raw.index[-504]  # 2yr

# P1: Gold price (wide)
ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(raw.loc[start:, 'Gold'], color=GOLD, lw=1.2)
ax1.set_title('Gold Futures (XAU/USD)  —  GC=F', fontsize=9, color='#aaa', pad=3)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))

# P2: DXY
ax2 = fig.add_subplot(gs[0, 2])
ax2.plot(raw.loc[start:, 'DXY'], color=BLUE, lw=1.0)
ax2.set_title('DXY Dollar Index', fontsize=9, color='#aaa', pad=3)

# P3: 10Y yield
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(raw.loc[start:, 'UST10Y'], color=RED, lw=1.0)
ax3.set_title('US 10Y Treasury Yield (%)', fontsize=9, color='#aaa', pad=3)

# P4: Yield curve
ax4 = fig.add_subplot(gs[1, 1])
c = curve.loc[start:]
ax4.plot(c, color=ORANGE, lw=1.0)
ax4.axhline(0, color=GREY, lw=0.8, ls='--')
ax4.fill_between(c.index, 0, c, where=(c > 0), color=GREEN, alpha=0.15, label='Normal')
ax4.fill_between(c.index, 0, c, where=(c < 0), color=RED,   alpha=0.15, label='Inverted')
ax4.set_title('Yield Curve (10Y − 3M, %)', fontsize=9, color='#aaa', pad=3)
ax4.legend(fontsize=7)

# P5: VIX
ax5 = fig.add_subplot(gs[1, 2])
ax5.plot(raw.loc[start:, 'VIX'], color=PURPLE, lw=1.0)
ax5.axhline(20, color=GREY, lw=0.5, ls=':')
ax5.axhline(30, color=RED,  lw=0.5, ls=':')
ax5.set_title('VIX  |  20=caution  30=fear', fontsize=9, color='#aaa', pad=3)

# P6: Rolling correlations (full width)
ax6 = fig.add_subplot(gs[2, :])
ax6.plot(corr_dxy.loc[start:], color=BLUE,   lw=1.0, label='Gold / DXY')
ax6.plot(corr_ust.loc[start:], color=RED,    lw=1.0, label='Gold / 10Y Yield')
ax6.plot(corr_vix.loc[start:], color=GOLD,   lw=1.0, label='Gold / VIX')
ax6.axhline(0,    color=GREY, lw=0.8, ls='--')
ax6.axhline(0.3,  color=GREY, lw=0.4, ls=':')
ax6.axhline(-0.3, color=GREY, lw=0.4, ls=':')
ax6.fill_between(corr_vix.loc[start:].index, 0.3,
                 corr_vix.loc[start:].clip(lower=0.3),
                 color=GOLD, alpha=0.1, label='VIX fear bid zone')
ax6.set_title('63-Day Rolling Correlations with Gold', fontsize=9, color='#aaa', pad=3)
ax6.legend(fontsize=8, loc='upper right')
ax6.set_ylim(-1, 1)

# ── Inline display ─────────────────────────────────────
import io as _io, IPython.display as _ipyd
_buf = _io.BytesIO()
plt.savefig(_buf, dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.savefig('macro_triggers.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
_buf.seek(0)
_ipyd.display(_ipyd.Image(_buf.read()))
plt.close()
print("✓ Saved: macro_triggers.png")


In [ ]:
# =============================================================================
# CB DEMAND & GEOPOLITICAL RISK PROXY  |  GLD flow trend + Gold-VIX co-movement
# =============================================================================
import yfinance as yf
import pandas as pd

print("Fetching GLD flow and gold/VIX co-movement proxies...")
gld = yf.download('GLD', period='3mo', interval='1d', auto_adjust=True, progress=False)
vix = yf.download('^VIX', period='3mo', interval='1d', auto_adjust=True, progress=False)
gc  = yf.download('GC=F', period='3mo', interval='1d', auto_adjust=True, progress=False)

for _d in (gld, vix, gc):
    _d.columns = [c[0] if isinstance(c, tuple) else c for c in _d.columns]

if len(gld) >= 25 and len(vix) >= 5 and len(gc) >= 5:
    gld_vol_chg = gld['Volume'].iloc[-5:].mean() / gld['Volume'].iloc[-25:-5].mean() - 1
    vix_chg     = vix['Close'].iloc[-1] / vix['Close'].iloc[-5] - 1
    gold_chg    = gc['Close'].iloc[-1] / gc['Close'].iloc[-5] - 1

    print(f"\n  GLD 5d avg volume vs prior 20d avg : {gld_vol_chg:+.1%}")
    print(f"  VIX 5d change                      : {vix_chg:+.1%}")
    print(f"  Gold 5d change                     : {gold_chg:+.1%}")
    print()

    if gld_vol_chg > 0.25 and gold_chg > 0:
        print("  Signal -> Rising GLD volume + rising gold = inflow/accumulation,"
              " consistent with CB or institutional demand")
    if vix_chg > 0.15 and gold_chg > 0:
        print("  Signal -> VIX spiking with gold = geopolitical/risk-off demand for gold")
    if vix_chg > 0.15 and gold_chg <= 0:
        print("  Signal -> VIX spiking but gold flat/down = risk-off favouring cash/USD, not gold yet")
    if abs(gld_vol_chg) < 0.10 and abs(vix_chg) < 0.05:
        print("  Signal -> Quiet flows - no acute CB demand or geo-risk signal this week")
else:
    print("  Insufficient data returned - check connection.")

print("\n  Manual checks:")
print("   - World Gold Council monthly central bank purchase reports (gold.org)")
print("   - Active geopolitical flashpoints: scan major newswires for escalation/de-escalation")


Tier 3 - Options & Derivatives
¶
Where the market is pinned or coiled

In [ ]:
# =============================================================================
# GOLD VOLATILITY SURFACE  |  GVZ · IV Term Structure · Vol Risk Premium
# =============================================================================

import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.interpolate import CubicSpline
from matplotlib import rcParams
import warnings; warnings.filterwarnings('ignore'); unmute_convergence()

rcParams.update({
    'figure.facecolor':'#0d0d0d','axes.facecolor':'#111111','axes.edgecolor':'#2a2a2a',
    'axes.labelcolor':'#cccccc','axes.grid':True,'grid.color':'#1e1e1e',
    'grid.linewidth':0.5,'xtick.color':'#888888','ytick.color':'#888888',
    'text.color':'#cccccc','legend.facecolor':'#1a1a1a','legend.edgecolor':'#333333',
    'font.family':'monospace','font.size':9,
})
GOLD='#C9A84C'; GREEN='#2ecc71'; RED='#e74c3c'; BLUE='#3498db'; GREY='#444444'

print("Fetching GVZ and gold data ...")

def safe_series(ticker, period='2y', label=None):
    s = yf.download(ticker, period=period, auto_adjust=True, progress=False)['Close'].squeeze()
    s.name = label or ticker
    return s.ffill().dropna()

gvz     = safe_series('^GVZ',  label='GVZ')
gold_px = safe_series('GC=F',  label='Gold')

# Realised volatility (annualised %)
gold_ret = np.log(gold_px / gold_px.shift(1)).dropna()
rvol_10  = gold_ret.rolling(10).std() * np.sqrt(252) * 100
rvol_21  = gold_ret.rolling(21).std() * np.sqrt(252) * 100
rvol_63  = gold_ret.rolling(63).std() * np.sqrt(252) * 100

# Align GVZ with realised vol
gvz_aligned = gvz.reindex(rvol_21.index).ffill().dropna()
common_idx  = gvz_aligned.index.intersection(rvol_21.index)
gvz_a       = gvz_aligned.loc[common_idx]
rvol_21_a   = rvol_21.loc[common_idx]
vrp         = gvz_a - rvol_21_a   # vol risk premium

# Extract latest scalars safely
latest_gvz  = safe_at(gvz_a, -1)
latest_rvol = safe_at(rvol_21_a, -1)
latest_vrp  = safe_at(vrp, -1)

gvz_mean    = float(gvz_a.rolling(252).mean().iloc[-1])
gvz_std     = float(gvz_a.rolling(252).std().iloc[-1])
gvz_z       = (latest_gvz - gvz_mean) / (gvz_std + 1e-5)

# IV term structure from GLD options
print("  Building IV term structure from GLD options chain ...")
gld_tk  = yf.Ticker("GLD")
gld_spot_raw = yf.download('GLD', period='5d', auto_adjust=True, progress=False)['Close'].squeeze()
gld_spot = float(gld_spot_raw.dropna().iloc[-1])

expirations = gld_tk.options[:6]
term = []
for exp in expirations:
    try:
        chain   = gld_tk.option_chain(exp)
        calls   = chain.calls.copy()
        puts    = chain.puts.copy()
        calls['dist'] = (calls['strike'] - gld_spot).abs()
        puts ['dist'] = (puts ['strike'] - gld_spot).abs()
        atm_c_iv = float(calls.nsmallest(2,'dist')['impliedVolatility'].mean())
        atm_p_iv = float(puts.nsmallest(2,'dist') ['impliedVolatility'].mean())
        atm_iv   = (atm_c_iv + atm_p_iv) / 2 * 100
        dte      = (pd.Timestamp(exp) - pd.Timestamp('today')).days
        if dte > 0 and atm_iv > 0:
            term.append({'expiry': exp, 'dte': int(dte), 'atm_iv': float(atm_iv)})
    except Exception as e:
        print(f"    ⚠ {exp}: {e}")

term_df = pd.DataFrame(term).dropna().sort_values('dte').reset_index(drop=True)
print(f"  ✓ {len(term_df)} valid expirations")

# Interpolated term structure curve
has_curve = len(term_df) >= 3
if has_curve:
    cs = CubicSpline(term_df['dte'].values, term_df['atm_iv'].values)
    dte_interp = np.linspace(term_df['dte'].min(), term_df['dte'].max(), 300)
    iv_interp  = cs(dte_interp)
    front_iv   = safe_at(term_df['atm_iv'], 0)
    back_iv    = safe_at(term_df['atm_iv'], -1)
    ts_shape   = 'Contango (normal backwardation in vol — calm near-term)' if back_iv > front_iv else 'Backwardation ⚠ — near-term fear elevated'
else:
    ts_shape = 'Insufficient expirations for curve'

print(f"\n  ─── Volatility Summary ─────────────────────────────────")
print(f"  GVZ (Implied Vol)    : {latest_gvz:.1f}%")
print(f"  Realised Vol 21d     : {latest_rvol:.1f}%")
print(f"  Vol Risk Premium     : {latest_vrp:+.1f}%  {'(options expensive)' if latest_vrp > 2 else '(options cheap)' if latest_vrp < -2 else '(fair value)'}")
print(f"  GVZ z-score (1yr)    : {gvz_z:+.2f}  {'⚠ Fear spike' if gvz_z > 1.5 else '✦ Suppressed — watch for vol expansion' if gvz_z < -1.0 else 'Normal range'}")
if has_curve: print(f"  Term Structure       : {ts_shape}")

# ── Dashboard ─────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 11))
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle('GOLD VOLATILITY SURFACE  //  GVZ · Realised Vol · IV Term Structure · VRP',
             fontsize=11, color=GOLD, fontfamily='monospace', y=0.98)

gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.3,
                        left=0.07, right=0.97, top=0.94, bottom=0.06)
start = gvz_a.index[-504] if len(gvz_a) > 504 else gvz_a.index[0]

# P1: GVZ vs realised vol
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(gvz_a.loc[start:],      color=RED,   lw=1.0, label='GVZ (Implied)')
ax1.plot(rvol_21_a.loc[start:],  color=GREEN, lw=1.0, label='RVol 21d')
ax1.plot(rvol_63.reindex(gvz_a.index).loc[start:], color=BLUE, lw=0.8, ls='--', label='RVol 63d')
ax1.set_title('GVZ vs Realised Volatility (%)', fontsize=9, color='#aaa', pad=3)
ax1.set_ylabel('Annualised Vol %', fontsize=8)
ax1.legend(fontsize=7)

# P2: Vol risk premium
ax2 = fig.add_subplot(gs[0, 1])
v = vrp.loc[start:]
ax2.fill_between(v.index, 0, v, where=(v > 0), color=RED,   alpha=0.3, label='IV expensive (sell vol)')
ax2.fill_between(v.index, 0, v, where=(v < 0), color=GREEN, alpha=0.3, label='IV cheap (buy vol)')
ax2.axhline(0, color=GREY, lw=0.8)
ax2.plot(v, color=GOLD, lw=0.7, alpha=0.7)
ax2.set_title(f'Vol Risk Premium  (GVZ − RVol21d)  |  latest={latest_vrp:+.1f}%',
              fontsize=9, color='#aaa', pad=3)
ax2.set_ylabel('VRP %', fontsize=8)
ax2.legend(fontsize=7)

# P3: IV term structure
ax3 = fig.add_subplot(gs[1, 0])
if has_curve:
    ax3.plot(dte_interp, iv_interp, color=GOLD, lw=1.5, label='Interpolated')
if not term_df.empty:
    ax3.scatter(term_df['dte'].values, term_df['atm_iv'].values,
                color='white', s=35, zorder=5, label='Market ATM IV')
    for _, row in term_df.iterrows():
        ax3.annotate(row['expiry'], (row['dte'], row['atm_iv']),
                     textcoords='offset points', xytext=(0, 6),
                     fontsize=6, color='#888', ha='center')
ax3.set_title(f'ATM IV Term Structure (GLD)  |  {ts_shape}', fontsize=8, color='#aaa', pad=3)
ax3.set_xlabel('Days to Expiry', fontsize=8)
ax3.set_ylabel('ATM IV (%)', fontsize=8)
ax3.legend(fontsize=7)

# P4: GVZ z-score
ax4 = fig.add_subplot(gs[1, 1])
gvz_z_series = (gvz_a - gvz_a.rolling(252).mean()) / (gvz_a.rolling(252).std() + 1e-5)
z = gvz_z_series.loc[start:]
ax4.plot(z, color=BLUE, lw=0.9, label='GVZ z-score')
ax4.axhline(0,    color=GREY,  lw=0.8, ls='--')
ax4.axhline(1.5,  color=RED,   lw=0.6, ls=':')
ax4.axhline(-1.5, color=GREEN, lw=0.6, ls=':')
ax4.fill_between(z.index, 1.5, z.clip(lower=1.5),  color=RED,   alpha=0.15, label='Fear spike')
ax4.fill_between(z.index, z.clip(upper=-1.5), -1.5, color=GREEN, alpha=0.15, label='Complacency zone')
ax4.set_title(f'GVZ z-Score (1yr rolling)  |  current={gvz_z:+.2f}',
              fontsize=9, color='#aaa', pad=3)
ax4.legend(fontsize=7)
ax4.set_ylim(-4, 4)

# ── Inline display ─────────────────────────────────────
import io as _io, IPython.display as _ipyd
_buf = _io.BytesIO()
plt.savefig(_buf, dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.savefig('gold_vol_surface.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
_buf.seek(0)
_ipyd.display(_ipyd.Image(_buf.read()))
plt.close()
print("✓ Saved: gold_vol_surface.png")


Gamma concentration - dealer GEX pipeline
¶

In [ ]:
# ============================================================
# CELL 1 — IMPORTS & CONFIGURATION
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from scipy.stats import norm
from scipy.interpolate import griddata
import yfinance as yf
import warnings
warnings.filterwarnings('ignore'); unmute_convergence()

# ─── Style ────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#020818',
    'axes.facecolor':   '#020818',
    'axes.edgecolor':   '#2a3550',
    'axes.labelcolor':  '#c8d0e0',
    'text.color':       '#c8d0e0',
    'xtick.color':      '#7a8aaa',
    'ytick.color':      '#7a8aaa',
    'grid.color':       '#1a2540',
    'grid.linewidth':   0.5,
    'font.family':      'monospace',
    'axes.titlecolor':  '#e8c840',
})

GOLD  = '#e8c840'
SILVER= '#8aa0cc'
GREEN = '#2ecc8a'
RED   = '#e85540'
DIM   = '#3a4a70'
BG    = '#020818'

# ─── Configuration ────────────────────────────────────────
# Replace with your live GC front-month price
GC_FUTURES_PRICE = LIVE_GC   # live from PRELUDE (was hardcoded 4725.0 from 9 May)

# Ticker for options chain proxy
# GLD = $10 per oz equivalent (GC / 10), IAU = $1 per oz (GC / 100)
OPTIONS_TICKER    = 'GLD'
PRICE_MULTIPLIER  = LIVE_RATIO  # live ratio (was 10.0; actual ~10.9-11.1)
RISK_FREE_RATE    = 0.053  # current SOFR approx

print(f"[CONFIG] GC Futures Reference Price: ${GC_FUTURES_PRICE:,.1f}")
print(f"[CONFIG] Options Proxy: {OPTIONS_TICKER} | Multiplier: {PRICE_MULTIPLIER}x")
print(f"[CONFIG] Risk-Free Rate: {RISK_FREE_RATE*100:.2f}%")


In [ ]:
# ============================================================
# CELL 2 — DATA FETCH: OPTIONS CHAIN + GREEKS ENGINE
# ============================================================

def fetch_options_chain(ticker_sym: str, multiplier: float) -> pd.DataFrame:
    """
    Pull full options chain, compute all standard + exotic greeks.
    Returns a unified DataFrame for calls and puts.
    """
    t = yf.Ticker(ticker_sym)
    spot_proxy = t.history(period='1d')['Close'].iloc[-1]
    spot_gc    = spot_proxy * multiplier  # back to GC-equivalent price
    
    print(f"[FETCH] {ticker_sym} spot: ${spot_proxy:.2f}  |  GC-equiv: ${spot_gc:.2f}")
    print(f"[FETCH] Using configured GC price: ${GC_FUTURES_PRICE:.2f}")
    
    expirations = t.options
    print(f"[FETCH] Available expirations: {len(expirations)}")
    for i, e in enumerate(expirations[:8]):
        print(f"        [{i}] {e}")
    
    all_frames = []
    
    for exp in expirations[:6]:  # fetch first 6 expirations
        try:
            chain = t.option_chain(exp)
            
            for side, df in [('call', chain.calls), ('put', chain.puts)]:
                df = df.copy()
                df['expiration']  = pd.to_datetime(exp)
                df['side']        = side
                df['spot_proxy']  = spot_proxy
                df['spot_gc']     = GC_FUTURES_PRICE
                df['dte']         = (pd.to_datetime(exp) - pd.Timestamp.now()).days
                df['strike_gc']   = df['strike'] * multiplier
                
                # Clean IV — yfinance sometimes returns NaN or 0
                df['iv'] = df['impliedVolatility'].replace(0, np.nan).fillna(0.20)
                df['iv'] = df['iv'].clip(0.01, 2.0)
                
                all_frames.append(df)
        except Exception as e:
            print(f"[WARN] Failed to fetch {exp}: {e}")
    
    raw = pd.concat(all_frames, ignore_index=True)
    raw['T'] = (raw['dte'] / 365.0).clip(lower=1/365)  # time in years, min 1 day
    
    print(f"\n[FETCH] Total option rows: {len(raw)}")
    return raw

raw_chain = fetch_options_chain(OPTIONS_TICKER, PRICE_MULTIPLIER)


In [ ]:
# ============================================================
# CELL 3 — BLACK-SCHOLES GREEKS ENGINE
# ============================================================

def bs_greeks(S, K, T, r, sigma, flag='call'):
    """
    Full Black-Scholes greeks including vanna, charm, vomma, speed.
    S     = spot price
    K     = strike
    T     = time to expiry (years)
    r     = risk-free rate
    sigma = implied vol
    flag  = 'call' or 'put'
    """
    if T <= 0 or sigma <= 0:
        return {g: 0.0 for g in ['delta','gamma','theta','vega','vanna','charm','vomma','speed','iv']}
    
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    
    Nd1  = norm.cdf(d1)
    Nd2  = norm.cdf(d2)
    nd1  = norm.pdf(d1)
    
    phi  = 1 if flag == 'call' else -1
    
    delta = phi * Nd1 if flag == 'call' else phi * Nd1 + 1  # put delta is Nd1-1
    if flag == 'put':
        delta = Nd1 - 1
    else:
        delta = Nd1
    
    gamma  = nd1 / (S * sigma * np.sqrt(T))
    
    # Vega: per 1% change in vol → divide by 100
    vega   = S * nd1 * np.sqrt(T) / 100
    
    # Theta: per calendar day
    theta_call = (-(S * nd1 * sigma) / (2*np.sqrt(T)) - r*K*np.exp(-r*T)*Nd2) / 365
    theta_put  = (-(S * nd1 * sigma) / (2*np.sqrt(T)) + r*K*np.exp(-r*T)*(1-Nd2)) / 365
    theta      = theta_call if flag == 'call' else theta_put
    
    # Vanna: dDelta/dVol = dVega/dS
    vanna  = -nd1 * d2 / sigma
    
    # Charm: dDelta/dTime (delta decay)
    charm_call = -nd1 * (2*r*T - d2*sigma*np.sqrt(T)) / (2*T*sigma*np.sqrt(T))
    charm_put  = charm_call
    charm      = charm_call  # same formula, sign embedded in use
    
    # Vomma: dVega/dVol (vol convexity)
    vomma  = vega * d1 * d2 / sigma
    
    # Speed: dGamma/dS
    speed  = -gamma/S * (d1/(sigma*np.sqrt(T)) + 1)
    
    return {
        'delta': delta,
        'gamma': gamma,
        'theta': theta,
        'vega':  vega,
        'vanna': vanna,
        'charm': charm,
        'vomma': vomma,
        'speed': speed,
    }


# ─── Vectorised apply ──────────────────────────────────────
def compute_greeks_df(df: pd.DataFrame) -> pd.DataFrame:
    records = []
    for _, row in df.iterrows():
        g = bs_greeks(
            S     = float(row['spot_gc']),
            K     = float(row['strike_gc']),
            T     = float(row['T']),
            r     = RISK_FREE_RATE,
            sigma = float(row['iv']),
            flag  = row['side']
        )
        g['iv'] = row['iv']
        records.append(g)
    
    greeks_df = pd.DataFrame(records)
    return pd.concat([df.reset_index(drop=True), greeks_df], axis=1)


print("[GREEKS] Computing Black-Scholes greeks for all strikes/expirations...")
chain = compute_greeks_df(raw_chain)
print(f"[GREEKS] Done. Shape: {chain.shape}")
print(chain[['expiration','side','strike_gc','dte','iv','delta','gamma','vega','vanna']].head(10).to_string())


In [ ]:
# ============================================================
# CELL 4 — DEALER GAMMA EXPOSURE (GEX) ENGINE
# ============================================================
"""
Dealer GEX Framework:
─────────────────────
Dealers are short gamma when retail/funds are net long options.
- Dealer long call  → dealer short gamma (delta-hedging SELLS into rallies)
- Dealer long put   → dealer short gamma (delta-hedging BUYS into drops)
- Dealer short call → dealer long gamma  (stabilising)
- Dealer short put  → dealer long gamma  (stabilising)

Convention used here (most common in prop/HF desks):
  GEX per strike = OI × Gamma × contract_size × spot²/100
  Sign: calls = positive (dealers long), puts = negative (dealers short)
  
  Aggregate GEX > 0 → dealers long gamma → mean-reverting price action
  Aggregate GEX < 0 → dealers short gamma → trending/volatile price action
"""

CONTRACT_SIZE = 100  # GLD = 100 shares/contract; GC = 100 oz

def compute_gex(df: pd.DataFrame, contract_size: int = CONTRACT_SIZE) -> pd.DataFrame:
    df = df.copy()
    S = GC_FUTURES_PRICE
    
    # GEX per contract row (in dollar terms)
    df['gex_raw'] = (
        df['openInterest'].fillna(0) *
        df['gamma'] *
        contract_size *
        (S ** 2) / 100
    )
    
    # Dealer sign: calls dealers are effectively long (positive GEX)
    # puts dealers are effectively short (negative GEX)
    df['gex'] = np.where(df['side'] == 'call', df['gex_raw'], -df['gex_raw'])
    
    # Dollar delta: OI × delta × contract_size × spot
    df['dollar_delta'] = (
        df['openInterest'].fillna(0) *
        df['delta'].abs() *
        contract_size *
        S
    )
    df['dollar_delta'] = np.where(df['side'] == 'call', df['dollar_delta'], -df['dollar_delta'])
    
    # Dollar vanna (important near vol events)
    df['dollar_vanna'] = (
        df['openInterest'].fillna(0) *
        df['vanna'] *
        contract_size
    )
    
    # Dollar charm (delta decay per day — drives hedging flows into expiry)
    df['dollar_charm'] = (
        df['openInterest'].fillna(0) *
        df['charm'] *
        contract_size
    )
    
    return df


chain = compute_gex(chain)
print("[GEX] Dealer positioning computed.")

# ─── Aggregate by strike (all expirations combined) ────────
by_strike = (
    chain.groupby('strike_gc')[['gex','dollar_delta','openInterest','dollar_vanna','dollar_charm']]
    .sum()
    .reset_index()
    .sort_values('strike_gc')
)

# Focus on strikes within ±15% of spot
spot = GC_FUTURES_PRICE
mask = (by_strike['strike_gc'] >= spot * 0.85) & (by_strike['strike_gc'] <= spot * 1.15)
by_strike_near = by_strike[mask].copy()

total_gex = by_strike_near['gex'].sum()
gex_sign  = "LONG GAMMA (mean-reverting)" if total_gex > 0 else "SHORT GAMMA (trending/volatile)"
print(f"\n[GEX] Net Dealer GEX (±15% from spot): ${total_gex/1e6:.1f}M")
print(f"[GEX] Dealer Regime: {gex_sign}")


In [ ]:
# ============================================================
# CELL 5 — OPEN INTEREST STRUCTURE BY EXPIRATION
# ============================================================

oi_by_exp = (
    chain.groupby(['expiration','side'])['openInterest']
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)
oi_by_exp['expiration_str'] = oi_by_exp['expiration'].dt.strftime('%b %d')
oi_by_exp['dte']            = (oi_by_exp['expiration'] - pd.Timestamp.now()).dt.days
oi_by_exp['put_call_ratio'] = (oi_by_exp.get('put', 0) / oi_by_exp.get('call', 1)).round(2)

print("[OI] Open Interest by Expiration:")
print(oi_by_exp[['expiration_str','dte','call','put','put_call_ratio']].to_string(index=False))

# ─── Key strike OI concentration ──────────────────────────
calls_near = chain[(chain['side']=='call') & mask].groupby('strike_gc')['openInterest'].sum().sort_values(ascending=False)
puts_near  = chain[(chain['side']=='put')  & mask].groupby('strike_gc')['openInterest'].sum().sort_values(ascending=False)

print(f"\n[OI] Top 5 Call Strikes (near spot ±15%):")
print(calls_near.head(5).to_string())
print(f"\n[OI] Top 5 Put Strikes (near spot ±15%):")
print(puts_near.head(5).to_string())


In [ ]:
# ============================================================
# CELL 6 — DASHBOARD PLOT 1: GEX BY STRIKE + DOLLAR DELTA
# ============================================================

fig, axes = plt.subplots(2, 1, figsize=(16, 11), facecolor=BG)
fig.suptitle(
    f'GOLD DEALER POSITIONING DASHBOARD  |  GC Ref: ${GC_FUTURES_PRICE:,.0f}',
    fontsize=14, color=GOLD, fontweight='bold', y=0.98
)

# ─── TOP: GEX by strike ───────────────────────────────────
ax1 = axes[0]
strikes_plot = by_strike_near['strike_gc'].values
gex_vals     = by_strike_near['gex'].values / 1e6  # in $M

colors_gex = [GREEN if v >= 0 else RED for v in gex_vals]
x_pos      = np.arange(len(strikes_plot))

bars = ax1.bar(x_pos, gex_vals, color=colors_gex, alpha=0.85, width=0.7, linewidth=0)
ax1.axhline(0, color=DIM, linewidth=0.8, linestyle='--')
ax1.axvline(
    x=np.searchsorted(strikes_plot, spot),
    color=GOLD, linewidth=1.2, linestyle=':', alpha=0.7,
    label=f'Spot ${spot:,.0f}'
)

# Shade total GEX regime
regime_color = GREEN if total_gex > 0 else RED
ax1.set_facecolor(BG)
ax1.set_title(
    f'NET DEALER GAMMA EXPOSURE (GEX) BY STRIKE  |  Total: ${total_gex/1e6:.1f}M  [{gex_sign}]',
    color=regime_color, fontsize=10, pad=8
)

# Label every 5th strike
step = max(1, len(strikes_plot) // 15)
ax1.set_xticks(x_pos[::step])
ax1.set_xticklabels([f'${s:,.0f}' for s in strikes_plot[::step]], rotation=45, fontsize=8)
ax1.set_ylabel('GEX ($M)', color=GOLD, fontsize=9)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.0f}M'))
ax1.grid(axis='y', alpha=0.3)
ax1.legend(fontsize=8, facecolor='#0a1020', edgecolor=DIM)

# Annotate top 3 positive and negative GEX strikes
gex_series = pd.Series(gex_vals, index=strikes_plot)
top3_pos   = gex_series.nlargest(3)
top3_neg   = gex_series.nsmallest(3)
for k, v in pd.concat([top3_pos, top3_neg]).items():
    xi = np.where(strikes_plot == k)[0]
    if len(xi):
        ax1.annotate(
            f'${k:,.0f}', xy=(xi[0], v),
            xytext=(xi[0], v + (0.8 if v >= 0 else -0.8)),
            fontsize=7, ha='center', color=GOLD, alpha=0.9
        )

# ─── BOTTOM: Dollar Delta by strike ───────────────────────
ax2 = axes[1]
delta_vals = by_strike_near['dollar_delta'].values / 1e6

colors_dd = [GREEN if v >= 0 else RED for v in delta_vals]
ax2.bar(x_pos, delta_vals, color=colors_dd, alpha=0.75, width=0.7, linewidth=0)
ax2.axhline(0, color=DIM, linewidth=0.8, linestyle='--')
ax2.axvline(
    x=np.searchsorted(strikes_plot, spot),
    color=GOLD, linewidth=1.2, linestyle=':', alpha=0.7
)
ax2.set_facecolor(BG)
ax2.set_title('DOLLAR DELTA EXPOSURE BY STRIKE  (+ = net long, − = net short)', color=SILVER, fontsize=10, pad=8)
ax2.set_xticks(x_pos[::step])
ax2.set_xticklabels([f'${s:,.0f}' for s in strikes_plot[::step]], rotation=45, fontsize=8)
ax2.set_ylabel('Dollar Delta ($M)', color=GOLD, fontsize=9)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.0f}M'))
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.97])
# ── Inline display ─────────────────────────────────────
import io as _io, IPython.display as _ipyd
_buf = _io.BytesIO()
plt.savefig(_buf, dpi=150, bbox_inches='tight', facecolor=BG)
plt.savefig('gex_dashboard.png', dpi=150, bbox_inches='tight', facecolor=BG)
_buf.seek(0)
_ipyd.display(_ipyd.Image(_buf.read()))
plt.close()
print("[PLOT] GEX Dashboard saved.")


In [ ]:
# ============================================================
# CELL 7 — DASHBOARD PLOT 2: OI HEATMAP + PUT/CALL BY EXPIRY
# ============================================================

fig2, axes2 = plt.subplots(1, 2, figsize=(18, 8), facecolor=BG)
fig2.suptitle('OPEN INTEREST STRUCTURE  |  GOLD OPTIONS', fontsize=13, color=GOLD, fontweight='bold')

# ─── Rebuild mask against current chain index ─────────────
spot       = GC_FUTURES_PRICE
chain_near = chain.loc[
    (chain['strike_gc'] >= spot * 0.85) &
    (chain['strike_gc'] <= spot * 1.15)
].copy()

# ─── LEFT: OI Heatmap (strike × expiration) ───────────────
ax3 = axes2[0]

pivot_all = (
    chain_near
    .groupby(['expiration', 'strike_gc'])['openInterest']
    .sum()
    .unstack(fill_value=0)
)
pivot_all.index = pivot_all.index.strftime('%b %d')

vmax = np.percentile(pivot_all.values, 98)

im = ax3.imshow(
    pivot_all.values,
    aspect='auto',
    cmap='YlOrRd',
    vmin=0, vmax=vmax,
    origin='upper'
)
ax3.set_title('OI HEATMAP  (strike × expiration)', color=SILVER, fontsize=10)
ax3.set_yticks(range(len(pivot_all.index)))
ax3.set_yticklabels(pivot_all.index, fontsize=8)
ax3.set_xlabel('Strike ($GC-equiv)', color=GOLD, fontsize=9)

ncols  = len(pivot_all.columns)
step_c = max(1, ncols // 10)
ax3.set_xticks(range(0, ncols, step_c))
ax3.set_xticklabels(
    [f'${s:,.0f}' for s in pivot_all.columns[::step_c]],
    rotation=45, fontsize=7
)

spot_col_idx = int(np.searchsorted(pivot_all.columns.values, spot))
ax3.axvline(x=spot_col_idx, color=GOLD, linewidth=1.5, linestyle='--', alpha=0.8, label='Spot')
ax3.legend(fontsize=8, facecolor='#0a1020', edgecolor=DIM)

cbar = plt.colorbar(im, ax=ax3, fraction=0.03, pad=0.02)
cbar.set_label('Open Interest', color=SILVER, fontsize=8)
cbar.ax.yaxis.set_tick_params(color=SILVER, labelsize=7)

# ─── RIGHT: Put/Call OI by expiration ─────────────────────
ax4 = axes2[1]

if 'call' in oi_by_exp.columns and 'put' in oi_by_exp.columns:
    n_exp     = len(oi_by_exp)
    y_pos     = np.arange(n_exp)
    call_vals = oi_by_exp['call'].values / 1000
    put_vals  = oi_by_exp['put'].values  / 1000

    ax4.barh(y_pos,       call_vals, color=GREEN, alpha=0.8, label='Calls', height=0.4)
    ax4.barh(y_pos + 0.4, put_vals,  color=RED,   alpha=0.8, label='Puts',  height=0.4)

    ax4.set_yticks(y_pos + 0.2)
    ax4.set_yticklabels(oi_by_exp['expiration_str'], fontsize=9)
    ax4.set_xlabel('Open Interest (000s)', color=GOLD, fontsize=9)
    ax4.set_title('CALL vs PUT OI BY EXPIRATION', color=SILVER, fontsize=10)
    ax4.axvline(0, color=DIM, linewidth=0.5)
    ax4.legend(facecolor='#0a1020', edgecolor=DIM, fontsize=9)
    ax4.set_facecolor(BG)

    for i, row in oi_by_exp.reset_index(drop=True).iterrows():
        pcr     = row.get('put_call_ratio', 0)
        max_bar = max(call_vals[i], put_vals[i])
        ax4.annotate(
            f'PCR:{pcr:.2f}',
            xy=(max_bar + 0.5, i + 0.2),
            fontsize=7, color=GOLD, va='center'
        )
else:
    ax4.text(0.5, 0.5, 'Insufficient OI data', ha='center', va='center',
             transform=ax4.transAxes, color=SILVER)

ax4.set_facecolor(BG)

plt.tight_layout(rect=[0, 0, 1, 0.96])
# ── Inline display ─────────────────────────────────────
import io as _io, IPython.display as _ipyd
_buf = _io.BytesIO()
plt.savefig(_buf, dpi=150, bbox_inches='tight', facecolor=BG)
plt.savefig('oi_structure.png', dpi=150, bbox_inches='tight', facecolor=BG)
_buf.seek(0)
_ipyd.display(_ipyd.Image(_buf.read()))
plt.close()
print("[PLOT] OI Structure saved.")


In [ ]:
# ============================================================
# CELL 8 — DASHBOARD PLOT 3: VANNA + CHARM EXPOSURE
# ============================================================

fig3, axes3 = plt.subplots(1, 2, figsize=(18, 7), facecolor=BG)
fig3.suptitle('SECOND-ORDER DEALER FLOWS  |  VANNA & CHARM', fontsize=13, color=GOLD, fontweight='bold')

# ─── Rebuild near-spot filter inline (safe index alignment) ─
vanna_by_strike = (
    chain.loc[
        (chain['strike_gc'] >= spot * 0.85) &
        (chain['strike_gc'] <= spot * 1.15)
    ]
    .groupby('strike_gc')[['dollar_vanna', 'dollar_charm']]
    .sum()
    .reset_index()
    .sort_values('strike_gc')
)

strikes_v = vanna_by_strike['strike_gc'].values
x_v       = np.arange(len(strikes_v))
step_v    = max(1, len(strikes_v) // 15)

# ─── LEFT: Vanna ──────────────────────────────────────────
ax5 = axes3[0]
vanna_vals = vanna_by_strike['dollar_vanna'].values

ax5.bar(x_v, vanna_vals,
        color=[GREEN if v >= 0 else RED for v in vanna_vals],
        alpha=0.8, width=0.7, linewidth=0)
ax5.axhline(0, color=DIM, linewidth=0.8)
ax5.axvline(
    x=int(np.searchsorted(strikes_v, spot)),
    color=GOLD, linewidth=1.2, linestyle=':', alpha=0.7,
    label=f'Spot ${spot:,.0f}'
)
ax5.set_facecolor(BG)
ax5.set_title(
    'VANNA EXPOSURE BY STRIKE\n(IV move \u2192 dealer delta hedging)',
    color=SILVER, fontsize=9
)
ax5.set_xticks(x_v[::step_v])
ax5.set_xticklabels([f'${s:,.0f}' for s in strikes_v[::step_v]], rotation=45, fontsize=8)
ax5.set_ylabel('Dollar Vanna', color=GOLD, fontsize=9)
ax5.grid(axis='y', alpha=0.3)
ax5.legend(fontsize=8, facecolor='#0a1020', edgecolor=DIM)

total_vanna   = float(vanna_vals.sum())
vanna_label   = 'Vol spike \u2192 dealer BUYING' if total_vanna > 0 else 'Vol spike \u2192 dealer SELLING'
vanna_color   = GREEN if total_vanna > 0 else RED
ax5.annotate(
    f'Net Vanna: {total_vanna:+.0f}\n{vanna_label}',
    xy=(0.02, 0.97), xycoords='axes fraction',
    fontsize=8, color=vanna_color, va='top',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='#0a1020', edgecolor=DIM, alpha=0.8)
)

# ─── RIGHT: Charm ─────────────────────────────────────────
ax6 = axes3[1]
charm_vals = vanna_by_strike['dollar_charm'].values

ax6.bar(x_v, charm_vals,
        color=[GREEN if v >= 0 else RED for v in charm_vals],
        alpha=0.8, width=0.7, linewidth=0)
ax6.axhline(0, color=DIM, linewidth=0.8)
ax6.axvline(
    x=int(np.searchsorted(strikes_v, spot)),
    color=GOLD, linewidth=1.2, linestyle=':', alpha=0.7,
    label=f'Spot ${spot:,.0f}'
)
ax6.set_facecolor(BG)
ax6.set_title(
    'CHARM EXPOSURE BY STRIKE\n(Time decay \u2192 delta unwind into expiry)',
    color=SILVER, fontsize=9
)
ax6.set_xticks(x_v[::step_v])
ax6.set_xticklabels([f'${s:,.0f}' for s in strikes_v[::step_v]], rotation=45, fontsize=8)
ax6.set_ylabel('Dollar Charm', color=GOLD, fontsize=9)
ax6.grid(axis='y', alpha=0.3)
ax6.legend(fontsize=8, facecolor='#0a1020', edgecolor=DIM)

total_charm  = float(charm_vals.sum())
charm_label  = 'Time \u2192 dealer BUY into expiry' if total_charm > 0 else 'Time \u2192 dealer SELL into expiry'
charm_color  = GREEN if total_charm > 0 else RED
ax6.annotate(
    f'Net Charm: {total_charm:+.0f}\n{charm_label}',
    xy=(0.02, 0.97), xycoords='axes fraction',
    fontsize=8, color=charm_color, va='top',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='#0a1020', edgecolor=DIM, alpha=0.8)
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
# ── Inline display ─────────────────────────────────────
import io as _io, IPython.display as _ipyd
_buf = _io.BytesIO()
plt.savefig(_buf, dpi=150, bbox_inches='tight', facecolor=BG)
plt.savefig('vanna_charm.png', dpi=150, bbox_inches='tight', facecolor=BG)
_buf.seek(0)
_ipyd.display(_ipyd.Image(_buf.read()))
plt.close()
print("[PLOT] Vanna & Charm saved.")


In [ ]:
# ============================================================
# CELL 9 — LIQUIDITY MECHANICS: KEY LEVEL IDENTIFICATION
# ============================================================

def compute_max_pain(chain_df: pd.DataFrame, expiry: str) -> float:
    exp_chain = chain_df[chain_df['expiration'] == pd.to_datetime(expiry)]
    if exp_chain.empty:
        return np.nan
    strikes   = sorted(exp_chain['strike_gc'].unique())
    pain_vals = []
    for test_strike in strikes:
        calls      = exp_chain[exp_chain['side'] == 'call']
        puts       = exp_chain[exp_chain['side'] == 'put']
        call_pain  = ((test_strike - calls['strike_gc']).clip(lower=0) *
                      calls['openInterest'].fillna(0) * CONTRACT_SIZE).sum()
        put_pain   = ((puts['strike_gc'] - test_strike).clip(lower=0) *
                      puts['openInterest'].fillna(0) * CONTRACT_SIZE).sum()
        pain_vals.append(call_pain + put_pain)
    return strikes[int(np.argmin(pain_vals))]


def find_gex_flip(by_strike_df: pd.DataFrame) -> float:
    df          = by_strike_df.sort_values('strike_gc').copy()
    df['cum_gex'] = df['gex'].cumsum()
    sign_changes  = df[df['cum_gex'].shift(1) * df['cum_gex'] < 0]
    if sign_changes.empty:
        return float(df.iloc[len(df) // 2]['strike_gc'])
    return float(sign_changes.iloc[0]['strike_gc'])


near_exp = chain['expiration'].min().strftime('%Y-%m-%d')
max_pain = compute_max_pain(chain, near_exp)
gex_flip = find_gex_flip(by_strike)

# ─── OI walls (rebuild filter inline) ─────────────────────
call_oi_near = (
    chain.loc[(chain['side'] == 'call') &
              (chain['strike_gc'] >= spot * 0.85) &
              (chain['strike_gc'] <= spot * 1.15)]
    .groupby('strike_gc')['openInterest'].sum()
    .sort_values(ascending=False)
)
put_oi_near = (
    chain.loc[(chain['side'] == 'put') &
              (chain['strike_gc'] >= spot * 0.85) &
              (chain['strike_gc'] <= spot * 1.15)]
    .groupby('strike_gc')['openInterest'].sum()
    .sort_values(ascending=False)
)

call_wall = float(call_oi_near.index[0]) if len(call_oi_near) else np.nan
put_wall  = float(put_oi_near.index[0])  if len(put_oi_near)  else np.nan

# ─── Summary ───────────────────────────────────────────────
print("=" * 60)
print("  GOLD OPTIONS LIQUIDITY MECHANICS SUMMARY")
print(f"  GC Reference Price:  ${GC_FUTURES_PRICE:>10,.1f}")
print("=" * 60)
print(f"  Near-Term Expiry:    {near_exp}")
print(f"  Max Pain Strike:     ${max_pain:>10,.1f}")
print(f"  GEX Flip Level:      ${gex_flip:>10,.1f}")
print(f"  Call OI Wall:        ${call_wall:>10,.1f}")
print(f"  Put OI Wall:         ${put_wall:>10,.1f}")
print(f"  Net Dealer GEX:      ${total_gex/1e6:>10.1f}M")
print(f"  Dealer Regime:       {gex_sign}")
print("=" * 60)

if GC_FUTURES_PRICE > gex_flip:
    print(f"  [!] Price ABOVE GEX flip -> dealers LONG gamma -> expect mean reversion")
else:
    print(f"  [!] Price BELOW GEX flip -> dealers SHORT gamma -> expect trending moves")

if abs(GC_FUTURES_PRICE - max_pain) < 50:
    print(f"  [!] Price within $50 of Max Pain -> expiry gravity ACTIVE")
else:
    drift    = "ABOVE" if GC_FUTURES_PRICE > max_pain else "BELOW"
    dist_str = f"${abs(GC_FUTURES_PRICE - max_pain):.0f} {drift} max pain"
    print(f"  [!] Price {dist_str} -> gravitational pull toward ${max_pain:,.0f}")

print("=" * 60)


In [ ]:
# ============================================================
# CELL 10 — DASHBOARD PLOT 4: LIQUIDITY MAP
# ============================================================

# ─── Safety guards: recompute if Cell 9 didn't run cleanly ─
spot = GC_FUTURES_PRICE

try:
    _ = max_pain
except NameError:
    print("[WARN] max_pain not defined — run Cell 9 first. Using spot as fallback.")
    max_pain = spot

try:
    _ = gex_flip
except NameError:
    print("[WARN] gex_flip not defined — run Cell 9 first. Using spot as fallback.")
    gex_flip = spot

try:
    _ = call_wall
except NameError:
    call_wall = np.nan

try:
    _ = put_wall
except NameError:
    put_wall = np.nan

try:
    _ = total_gex
    _ = gex_sign
except NameError:
    total_gex = 0.0
    gex_sign  = "UNKNOWN — run Cell 4 first"

# ─── Rebuild OI filter inline (safe index alignment) ────────
calls_by_s = (
    chain.loc[(chain['side'] == 'call') &
              (chain['strike_gc'] >= spot * 0.85) &
              (chain['strike_gc'] <= spot * 1.15)]
    .groupby('strike_gc')['openInterest'].sum()
)
puts_by_s = (
    chain.loc[(chain['side'] == 'put') &
              (chain['strike_gc'] >= spot * 0.85) &
              (chain['strike_gc'] <= spot * 1.15)]
    .groupby('strike_gc')['openInterest'].sum()
)

all_strikes = sorted(set(calls_by_s.index) | set(puts_by_s.index))
c_vals      = np.array([calls_by_s.get(s, 0) for s in all_strikes])
p_vals      = np.array([puts_by_s.get(s, 0)  for s in all_strikes])

fig4, ax7 = plt.subplots(figsize=(16, 8), facecolor=BG)
ax7.set_facecolor(BG)

x_all  = np.arange(len(all_strikes))
step_a = max(1, len(all_strikes) // 18)

ax7.bar(x_all,  c_vals / 1000, color=GREEN, alpha=0.7, width=0.6, label='Call OI')
ax7.bar(x_all, -p_vals / 1000, color=RED,   alpha=0.7, width=0.6, label='Put OI')
ax7.axhline(0, color='white', linewidth=0.5, alpha=0.3)

# ─── Spot line ──────────────────────────────────────────────
spot_idx = int(np.searchsorted(all_strikes, spot))
ax7.axvline(spot_idx, color=GOLD, linewidth=2, linestyle='--',
            alpha=0.9, label=f'Spot ${spot:,.0f}')

# ─── Max pain ───────────────────────────────────────────────
if not np.isnan(max_pain):
    mp_idx = int(np.searchsorted(all_strikes, max_pain))
    ax7.axvline(mp_idx, color='#ff88ff', linewidth=1.5, linestyle=':',
                alpha=0.8, label=f'Max Pain ${max_pain:,.0f}')

# ─── GEX flip ───────────────────────────────────────────────
if not np.isnan(gex_flip):
    gf_idx = int(np.searchsorted(all_strikes, gex_flip))
    ax7.axvline(gf_idx, color='#88ccff', linewidth=1.5, linestyle='-.',
                alpha=0.8, label=f'GEX Flip ${gex_flip:,.0f}')

# ─── Call & Put walls ───────────────────────────────────────
if not np.isnan(call_wall) and len(c_vals) > 0:
    cw_idx = int(np.searchsorted(all_strikes, call_wall))
    ax7.annotate(
        'CALL WALL',
        xy=(cw_idx, max(c_vals / 1000) * 0.85),
        fontsize=8, color=GREEN, ha='center',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='#0a1020', edgecolor=GREEN, alpha=0.9)
    )

if not np.isnan(put_wall) and len(p_vals) > 0:
    pw_idx = int(np.searchsorted(all_strikes, put_wall))
    ax7.annotate(
        'PUT WALL',
        xy=(pw_idx, -max(p_vals / 1000) * 0.85),
        fontsize=8, color=RED, ha='center',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='#0a1020', edgecolor=RED, alpha=0.9)
    )

# ─── Axes formatting ────────────────────────────────────────
ax7.set_xticks(x_all[::step_a])
ax7.set_xticklabels(
    [f'${s:,.0f}' for s in np.array(all_strikes)[::step_a]],
    rotation=45, fontsize=8
)
ax7.set_ylabel('Open Interest (000s)  |  Calls \u2191  Puts \u2193', color=GOLD, fontsize=9)

call_wall_str = f'${call_wall:,.0f}' if not np.isnan(call_wall) else 'N/A'
put_wall_str  = f'${put_wall:,.0f}'  if not np.isnan(put_wall)  else 'N/A'
mp_str        = f'${max_pain:,.0f}'  if not np.isnan(max_pain)  else 'N/A'
gf_str        = f'${gex_flip:,.0f}' if not np.isnan(gex_flip)  else 'N/A'

ax7.set_title(
    f'GOLD OPTIONS LIQUIDITY MAP  |  Call/Put OI + Key Dealer Levels\n'
    f'Max Pain {mp_str}  |  GEX Flip {gf_str}  |  Call Wall {call_wall_str}  |  Put Wall {put_wall_str}',
    color=SILVER, fontsize=10, pad=10
)
ax7.legend(facecolor='#0a1020', edgecolor=DIM, fontsize=9, loc='upper right')
ax7.grid(axis='y', alpha=0.3)
ax7.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{abs(x):.0f}K'))

plt.tight_layout()
# ── Inline display ─────────────────────────────────────
import io as _io, IPython.display as _ipyd
_buf = _io.BytesIO()
plt.savefig(_buf, dpi=150, bbox_inches='tight', facecolor=BG)
plt.savefig('liquidity_map.png', dpi=150, bbox_inches='tight', facecolor=BG)
_buf.seek(0)
_ipyd.display(_ipyd.Image(_buf.read()))
plt.close()
print("[PLOT] Liquidity Map saved.")


In [ ]:
# ============================================================
# CELL 11 — DAILY TRADE BRIEF (ACTIONABLE SUMMARY)
# ============================================================

# ─── Safety guards ──────────────────────────────────────────
_safe_gex_flip   = gex_flip   if 'gex_flip'   in dir() and not np.isnan(gex_flip)   else spot
_safe_max_pain   = max_pain   if 'max_pain'   in dir() and not np.isnan(max_pain)   else spot
_safe_call_wall  = call_wall  if 'call_wall'  in dir() and not np.isnan(call_wall)  else spot
_safe_put_wall   = put_wall   if 'put_wall'   in dir() and not np.isnan(put_wall)   else spot
_safe_total_gex  = total_gex  if 'total_gex'  in dir() else 0.0
_safe_gex_sign   = gex_sign   if 'gex_sign'   in dir() else 'UNKNOWN'
_safe_vanna      = total_vanna if 'total_vanna' in dir() else 0.0
_safe_charm      = total_charm if 'total_charm' in dir() else 0.0

# ─── Build second-order labels (no inline conditionals in f-string) ─
vanna_label = 'Vol spike -> dealer BUY'  if _safe_vanna > 0 else 'Vol spike -> dealer SELL'
charm_label = 'Time -> dealer BUY'       if _safe_charm > 0 else 'Time -> dealer SELL'

print("\n" + "\u2588" * 65)
print("  GOLD OPTIONS MARKET STRUCTURE — DAILY BRIEF")
print(f"  Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')} Athens")
print("\u2588" * 65)

print(f"  {'─'*51}")
print(f"  PRICE CONTEXT")
print(f"  {'─'*51}")
print(f"  GC Futures Reference:  ${GC_FUTURES_PRICE:>10,.1f}")
print(f"  Proxy Ticker Used:     {OPTIONS_TICKER}")

print(f"\n  {'─'*51}")
print(f"  DEALER GAMMA REGIME")
print(f"  {'─'*51}")
print(f"  Net GEX (+-15% spot): ${_safe_total_gex/1e6:>10.1f}M")
print(f"  GEX Flip Level:       ${_safe_gex_flip:>10,.1f}")
print(f"  Regime:               {_safe_gex_sign}")

print(f"\n  {'─'*51}")
print(f"  POSITIONING LEVELS")
print(f"  {'─'*51}")
print(f"  Max Pain (near exp):  ${_safe_max_pain:>10,.1f}")
print(f"  Call OI Wall:         ${_safe_call_wall:>10,.1f}   <- resistance / dealer short gamma")
print(f"  Put OI Wall:          ${_safe_put_wall:>10,.1f}   <- support / dealer long gamma")

print(f"\n  {'─'*51}")
print(f"  SECOND-ORDER FLOWS")
print(f"  {'─'*51}")
print(f"  Net Vanna:            {_safe_vanna:>+10.0f}   ({vanna_label})")
print(f"  Net Charm:            {_safe_charm:>+10.0f}   ({charm_label})")

print(f"\n  {'─'*51}")
print(f"  INTRADAY BIAS")
print(f"  {'─'*51}")

if _safe_total_gex > 0:
    print(f"  Long gamma environment. Price likely to REVERT to ${_safe_max_pain:,.0f}.")
    print(f"  Fade rallies above ${_safe_call_wall:,.0f}, fade drops below ${_safe_put_wall:,.0f}.")
    print(f"  Vol supply likely to cap breakouts.")
else:
    print(f"  Short gamma environment. Expect TRENDING moves with vol acceleration.")
    print(f"  GEX flip at ${_safe_gex_flip:,.0f} is your gamma wall pivot.")
    print(f"  Momentum strategies preferred. Breakouts likely to extend.")

pain_dist  = abs(GC_FUTURES_PRICE - _safe_max_pain)
pain_drift = "ABOVE" if GC_FUTURES_PRICE > _safe_max_pain else "BELOW"

if pain_dist < 50:
    print(f"\n  [!] Price within $50 of Max Pain -> expiry gravity ACTIVE")
else:
    print(f"\n  [!] Price ${pain_dist:.0f} {pain_drift} max pain -> gravitational pull toward ${_safe_max_pain:,.0f}")

print("\n" + "\u2588" * 65)
print("[DONE] Brief complete. Update GC_FUTURES_PRICE in Cell 1 each session.")
print(f"       Current configured price: ${GC_FUTURES_PRICE:,.1f}")
print(f"       Live GC approx (May 9):  $4,725  -- verify in Quantower before trading.")
print("\u2588" * 65 + "\n")


Vol skew - TRUE GEX, Vanna, Charm, full Greeks summary
¶

In [ ]:
# =============================================================================
# TRUE GEX  |  Gamma Exposure by Strike + Zero-Gamma Flip Level
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import norm
from matplotlib import rcParams
import yfinance as yf
import warnings
warnings.filterwarnings('ignore'); unmute_convergence()

rcParams.update({
    'figure.facecolor':'#0d0d0d','axes.facecolor':'#111111','axes.edgecolor':'#2a2a2a',
    'axes.labelcolor':'#cccccc','axes.grid':True,'grid.color':'#1e1e1e',
    'grid.linewidth':0.5,'xtick.color':'#888888','ytick.color':'#888888',
    'text.color':'#cccccc','legend.facecolor':'#1a1a1a','legend.edgecolor':'#333333',
    'font.family':'monospace','font.size':9,
})
GOLD='#C9A84C'; GREEN='#2ecc71'; RED='#e74c3c'; BLUE='#3498db'
GREY='#444444'; ORANGE='#e67e22'; CYAN='#1abc9c'; PURPLE='#9b59b6'

# ── Black-Scholes Greeks ──────────────────────────────────────────────────────

def bs_greeks(S, K, T, r, sigma, flag):
    if T <= 0 or sigma <= 0:
        return 0.0, 0.0, 0.0
    try:
        d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        pdf_d1 = norm.pdf(d1)
        gamma  = pdf_d1 / (S * sigma * np.sqrt(T))
        delta  = norm.cdf(d1) if flag == 'c' else norm.cdf(d1) - 1
        vanna  = -d2 * pdf_d1 / sigma
        return delta, gamma, vanna
    except:
        return 0.0, 0.0, 0.0

# ── Fetch prices ──────────────────────────────────────────────────────────────

print("=" * 60)
print("  TRUE GEX  |  Gamma Exposure + Zero-Flip Level")
print("=" * 60)

print("\n  Fetching price data ...")
gld_tk   = yf.Ticker('GLD')
gld_spot = float(gld_tk.history(period='5d')['Close'].iloc[-1])  # ← use 5d, not 1d

# Pull actual gold futures price — multi-ticker fallback
gold_spot = None

# Try 1: GC=F with longer period (1d often fails on weekends/holidays)
for period in ['5d', '1mo']:
    try:
        gc_tk = yf.Ticker('GC=F')
        hist  = gc_tk.history(period=period)
        if not hist.empty:
            val = safe_at(hist['Close'], -1)
            if val > 3000:
                gold_spot = val
                print(f"  ✓ Gold futures (GC=F)  : ${gold_spot:,.2f}")
                break
    except Exception:
        pass

# Try 2: GLD * known ratio as calibrated estimate
if gold_spot is None:
    try:
        # Use recent GLD/Gold ratio as a reliable fallback
        gold_spot = gld_spot * LIVE_RATIO   # was hardcoded 10.89
        print(f"  ⚠ Fallback ratio used  : ${gold_spot:,.2f}")
    except Exception as e:
        gold_spot = 4700.0
        print(f"  ⚠ Hard fallback        : ${gold_spot:,.2f}")

gld_to_gold = gold_spot / gld_spot
print(f"  GLD spot               : ${gld_spot:.2f}")
print(f"  Gold futures           : ${gold_spot:,.2f}")
print(f"  GLD→Gold ratio         : {gld_to_gold:.4f}x")

# Risk-free rate
try:
    irx = yf.Ticker('^IRX')
    r   = float(irx.history(period='5d')['Close'].iloc[-1]) / 100
except:
    r = 0.053
print(f"  Risk-free rate         : {r*100:.2f}%")

# GVZ implied vol
try:
    gvz_tk = yf.Ticker('^GVZ')
    gvz    = float(gvz_tk.history(period='5d')['Close'].iloc[-1]) / 100
except:
    gvz = 0.265
print(f"  GVZ (implied vol)      : {gvz*100:.1f}%")
current_gvz = gvz * 100

# ── Fetch options chain ───────────────────────────────────────────────────────

print("\n  Fetching GLD options chain ...")
exps   = gld_tk.options
today  = pd.Timestamp.today()
rows   = []
loaded = 0

for exp in exps[:8]:
    exp_dt = pd.Timestamp(exp)
    T      = max((exp_dt - today).days / 365, 1/365)
    try:
        chain = gld_tk.option_chain(exp)
        calls = chain.calls.copy()
        puts  = chain.puts.copy()
    except:
        continue

    for _, row in calls.iterrows():
        K   = float(row['strike'])
        oi  = float(row['openInterest']) if row['openInterest'] > 0 else 0
        vol = float(row['impliedVolatility']) if row['impliedVolatility'] > 0.01 else gvz
        if oi == 0: continue
        delta, gamma, vanna = bs_greeks(gld_spot, K, T, r, vol, 'c')
        rows.append({'strike': K, 'type': 'call', 'oi': oi,
                     'delta': delta, 'gamma': gamma, 'vanna': vanna,
                     'T': T, 'exp': exp})

    for _, row in puts.iterrows():
        K   = float(row['strike'])
        oi  = float(row['openInterest']) if row['openInterest'] > 0 else 0
        vol = float(row['impliedVolatility']) if row['impliedVolatility'] > 0.01 else gvz
        if oi == 0: continue
        delta, gamma, vanna = bs_greeks(gld_spot, K, T, r, vol, 'p')
        rows.append({'strike': K, 'type': 'put', 'oi': oi,
                     'delta': delta, 'gamma': gamma, 'vanna': vanna,
                     'T': T, 'exp': exp})
    loaded += 1

print(f"  Loaded {loaded} expirations  |  {len(rows)} option rows")
df_opts = pd.DataFrame(rows)

CONTRACT = 100
lo_strike = gld_spot * 0.75
hi_strike = gld_spot * 1.25

# ── GEX ───────────────────────────────────────────────────────────────────────
df_opts['gex_raw'] = df_opts.apply(
    lambda r: r['oi'] * r['gamma'] * CONTRACT * (gld_spot**2 / 100)
              * (1 if r['type'] == 'call' else -1), axis=1)

# ── VEX ───────────────────────────────────────────────────────────────────────
df_opts['vex_raw'] = df_opts.apply(
    lambda r: r['oi'] * r['vanna'] * CONTRACT
              * (1 if r['type'] == 'call' else -1), axis=1)

# ── DEX ───────────────────────────────────────────────────────────────────────
df_opts['dex_raw'] = df_opts['oi'] * df_opts['delta'] * CONTRACT * \
    df_opts['type'].map({'call': 1, 'put': -1})

# ── Aggregate by strike ───────────────────────────────────────────────────────
gex_by_strike = (df_opts[df_opts['strike'].between(lo_strike, hi_strike)]
                 .groupby('strike')['gex_raw'].sum().sort_index())
vex_by_strike = (df_opts[df_opts['strike'].between(lo_strike, hi_strike)]
                 .groupby('strike')['vex_raw'].sum().sort_index())
dex_by_strike = (df_opts[df_opts['strike'].between(lo_strike, hi_strike)]
                 .groupby('strike')['dex_raw'].sum().sort_index())

# ── Zero-gamma flip ───────────────────────────────────────────────────────────
strikes_arr = gex_by_strike.index.values
gex_arr     = gex_by_strike.values
cum_gex     = np.cumsum(gex_arr)
flip_level  = None

for i in range(1, len(cum_gex)):
    if cum_gex[i-1] * cum_gex[i] < 0:
        w          = abs(cum_gex[i-1]) / (abs(cum_gex[i-1]) + abs(cum_gex[i]))
        flip_level = strikes_arr[i-1] + w * (strikes_arr[i] - strikes_arr[i-1])
        break

top_gex_pos = gex_by_strike[gex_by_strike > 0].nlargest(5)
top_gex_neg = gex_by_strike[gex_by_strike < 0].nsmallest(5)
total_dex   = dex_by_strike.sum()

# ── Print report ──────────────────────────────────────────────────────────────
print(f"\n  ─── GEX Report ──────────────────────────────────────────")
print(f"  GLD Spot          : ${gld_spot:.2f}")
print(f"  Gold Futures      : ${gold_spot:,.2f}")
print(f"  GLD→Gold Ratio    : {gld_to_gold:.4f}x")
print(f"  Total Net GEX     : {gex_by_strike.sum():+,.0f}")
if flip_level:
    flip_gold = flip_level * gld_to_gold
    print(f"  Zero-Gamma Flip   : ${flip_level:.2f} GLD  (~${flip_gold:,.0f} Gold)")
    if gld_spot > flip_level:
        print(f"  Regime            : ABOVE zero-gamma → dealers LONG gamma → moves SUPPRESSED")
    else:
        print(f"  Regime            : BELOW zero-gamma → dealers SHORT gamma → moves AMPLIFIED")
else:
    print(f"  Zero-Gamma Flip   : Could not interpolate")

print(f"\n  Top CALL GEX Walls (resistance):")
for strike, val in top_gex_pos.items():
    print(f"    GLD ${strike:.0f}  (~Gold ${strike*gld_to_gold:,.0f})   GEX: {val:+,.0f}")

print(f"\n  Top PUT GEX Walls (support):")
for strike, val in top_gex_neg.items():
    print(f"    GLD ${strike:.0f}  (~Gold ${strike*gld_to_gold:,.0f})   GEX: {val:+,.0f}")

print(f"\n  Net Delta Exposure : {total_dex:+,.0f}")
if total_dex > 0:
    print(f"  Delta Bias         : BULLISH — dealers net long delta")
else:
    print(f"  Delta Bias         : BEARISH — dealers net short delta")

# ── Chart ─────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor('#0d0d0d')
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.5, wspace=0.35)
fig.suptitle(f'TRUE GEX + VANNA EXPOSURE  //  GLD ${gld_spot:.2f}  '
             f'(Gold ${gold_spot:,.0f})  ratio {gld_to_gold:.2f}x',
             fontsize=12, color=GOLD, fontfamily='monospace', y=0.98)

# P1: GEX
ax1 = fig.add_subplot(gs[0, :])
cols = [GREEN if v > 0 else RED for v in gex_by_strike.values]
ax1.bar(gex_by_strike.index, gex_by_strike.values / 1e6,
        color=cols, width=0.8, alpha=0.85)
ax1.axhline(0, color=GREY, lw=0.8)
ax1.axvline(gld_spot, color=GOLD, lw=1.2, ls='--',
            label=f'GLD Spot ${gld_spot:.2f} (Gold ${gold_spot:,.0f})')
if flip_level:
    ax1.axvline(flip_level, color=CYAN, lw=1.5, ls=':',
                label=f'Zero-Flip GLD ${flip_level:.1f} (~Gold ${flip_level*gld_to_gold:,.0f})')
for strike in top_gex_pos.index[:3]:
    ax1.axvline(strike, color=GREEN, lw=0.7, ls=':', alpha=0.6)
    ax1.text(strike + 0.2, ax1.get_ylim()[1] * 0.5 if ax1.get_ylim()[1] != 0 else 1,
             f'Gold ${strike*gld_to_gold:,.0f}',
             color=GREEN, fontsize=6.5, rotation=90,
             va='bottom', fontfamily='monospace')
for strike in top_gex_neg.index[:3]:
    ax1.axvline(strike, color=RED, lw=0.7, ls=':', alpha=0.6)
    ax1.text(strike + 0.2, ax1.get_ylim()[0] * 0.5 if ax1.get_ylim()[0] != 0 else -1,
             f'Gold ${strike*gld_to_gold:,.0f}',
             color=RED, fontsize=6.5, rotation=90,
             va='top', fontfamily='monospace')
ax1.set_title('Gamma Exposure (GEX) by Strike  |  '
              'Green=dealer long gamma (suppresses)  Red=dealer short gamma (amplifies)  '
              'Cyan=Zero-Flip',
              fontsize=9, color='#aaa', pad=4)
ax1.set_xlabel('GLD Strike  (multiply by ratio for Gold price)', fontsize=8)
ax1.set_ylabel('GEX ($M)', fontsize=8)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1f}M'))
ax1.legend(fontsize=8)
ax1.set_facecolor('#111111')
for sp in ax1.spines.values(): sp.set_color('#2a2a2a')

# P2: Vanna
ax2 = fig.add_subplot(gs[1, 0])
vcols = [BLUE if v > 0 else ORANGE for v in vex_by_strike.values]
ax2.bar(vex_by_strike.index, vex_by_strike.values / 1e3,
        color=vcols, width=0.8, alpha=0.85)
ax2.axhline(0, color=GREY, lw=0.8)
ax2.axvline(gld_spot, color=GOLD, lw=1.2, ls='--',
            label=f'Spot ${gld_spot:.2f}')
if flip_level:
    ax2.axvline(flip_level, color=CYAN, lw=1.2, ls=':',
                label=f'Flip ${flip_level:.1f}')
ax2.set_title('Vanna (VEX)  |  Blue=VIX drop→BUY  Orange=VIX drop→SELL',
              fontsize=9, color='#aaa', pad=4)
ax2.set_xlabel('GLD Strike', fontsize=8)
ax2.set_ylabel('VEX (K)', fontsize=8)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}K'))
ax2.legend(fontsize=7)
ax2.set_facecolor('#111111')
for sp in ax2.spines.values(): sp.set_color('#2a2a2a')
ax2.tick_params(colors='#888', labelsize=7)

# P3: Delta
ax3 = fig.add_subplot(gs[1, 1])
dcols = [GREEN if v > 0 else RED for v in dex_by_strike.values]
ax3.bar(dex_by_strike.index, dex_by_strike.values / 1e3,
        color=dcols, width=0.8, alpha=0.85)
ax3.axhline(0, color=GREY, lw=0.8)
ax3.axvline(gld_spot, color=GOLD, lw=1.2, ls='--',
            label=f'Spot ${gld_spot:.2f}')
if flip_level:
    ax3.axvline(flip_level, color=CYAN, lw=1.2, ls=':',
                label=f'Flip ${flip_level:.1f}')
ax3.set_title(f'Net Delta (DEX)  |  Total: {total_dex/1e3:+.0f}K  '
              f'→ {"BULLISH dealer bias" if total_dex > 0 else "BEARISH dealer bias"}',
              fontsize=9,
              color=GREEN if total_dex > 0 else RED, pad=4)
ax3.set_xlabel('GLD Strike', fontsize=8)
ax3.set_ylabel('DEX (K shares)', fontsize=8)
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}K'))
ax3.legend(fontsize=7)
ax3.set_facecolor('#111111')
for sp in ax3.spines.values(): sp.set_color('#2a2a2a')
ax3.tick_params(colors='#888', labelsize=7)

# ── Inline display ─────────────────────────────────────
import io as _io, IPython.display as _ipyd
_buf = _io.BytesIO()
plt.savefig(_buf, dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.savefig('gex_vanna_dashboard.png', dpi=150,
            bbox_inches='tight', facecolor='#0d0d0d')
_buf.seek(0)
_ipyd.display(_ipyd.Image(_buf.read()))
plt.close()
print("\n✓ Saved: gex_vanna_dashboard.png")


In [ ]:
# =============================================================================
# VANNA-VIX ALIGNMENT  |  Price impact of VIX moves at each strike level
# =============================================================================

print("=" * 60)
print("  VANNA-VIX ALIGNMENT  |  Dealer Flow on Vol Moves")
print("=" * 60)

vix_scenarios = [-5, -3, -1, +1, +3, +5]
total_vex     = vex_by_strike.sum()

print(f"\n  Current GVZ   : {current_gvz:.1f}%")
print(f"  GLD Spot      : ${gld_spot:.2f}")
print(f"  Gold Futures  : ${gold_spot:,.2f}")
print(f"  GLD→Gold Ratio: {gld_to_gold:.4f}x")
print(f"  Total Net VEX : {total_vex/1e3:+.1f}K")
if total_vex > 0:
    print(f"  Vanna Bias    : POSITIVE → VIX DROP forces dealers to BUY → bullish flow")
else:
    print(f"  Vanna Bias    : NEGATIVE → VIX DROP forces dealers to SELL → bearish flow")

print(f"\n  ─── Dealer Flow Scenarios (Δ VIX) ──────────────────────")
print(f"  {'ΔVIX':>8}  {'New GVZ':>10}  {'Dealer Action':>35}  {'Flow':>12}")
print(f"  {'─'*72}")
for dv in vix_scenarios:
    new_gvz   = max(current_gvz + dv, 1)
    flow      = total_vex * (-dv / current_gvz) * 0.01
    direction = 'DEALERS BUY  →  bullish' if flow > 0 else 'DEALERS SELL →  bearish'
    print(f"  {dv:>+8.0f}  {new_gvz:>9.1f}%  {direction:>35}  {flow/1e3:>+10.1f}K delta")

print(f"\n  ─── Top Vanna Levels (most sensitive to VIX moves) ─────")
print(f"  {'GLD Strike':>12}  {'Gold ~':>10}  {'Net VEX':>10}  "
      f"{'VIX drop':>14}  {'VIX spike':>14}  {'Dist':>8}")
print(f"  {'─'*76}")

top_vex_idx = vex_by_strike.abs().nlargest(10).index
for strike in sorted(top_vex_idx):
    vex_val  = vex_by_strike.get(strike, 0)
    on_drop  = '▲ BUY'  if vex_val > 0 else '▼ SELL'
    on_spike = '▼ SELL' if vex_val > 0 else '▲ BUY'
    dist     = ((strike - gld_spot) / gld_spot) * 100
    gold_eq  = strike * gld_to_gold
    print(f"  ${strike:>10.0f}  ~${gold_eq:>8,.0f}  {vex_val/1e3:>+8.1f}K  "
          f"{on_drop:>14}  {on_spike:>14}  {dist:>+6.1f}%")

print(f"\n  ─── Combined GEX + Vanna Level Map (Institutional Gravity) ─")
print(f"  Levels where BOTH gamma AND vanna are significant:\n")

combined = (gex_by_strike.abs() / (gex_by_strike.abs().max() + 1e-10) +
            vex_by_strike.abs() / (vex_by_strike.abs().max() + 1e-10))
top_combined = combined.nlargest(8).sort_index()

for strike, score in top_combined.items():
    gex_val  = gex_by_strike.get(strike, 0)
    vex_val  = vex_by_strike.get(strike, 0)
    dist     = ((strike - gld_spot) / gld_spot) * 100
    gold_eq  = strike * gld_to_gold
    gex_role = 'RESISTANCE' if gex_val > 0 else 'SUPPORT'
    vex_role = 'VIX-drop BUY' if vex_val > 0 else 'VIX-spike BUY'
    print(f"  GLD ${strike:.0f}  (~Gold ${gold_eq:,.0f})  |  dist {dist:+.1f}%  |  "
          f"GEX: {gex_role}  |  Vanna: {vex_role}  |  score {score:.2f}")

# ── Chart ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(18, 12), sharex=True)
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle(f'VANNA-VIX ALIGNMENT  //  GLD ${gld_spot:.2f}  '
             f'(Gold ${gold_spot:,.0f})  GVZ {current_gvz:.1f}%',
             fontsize=12, color=GOLD, fontfamily='monospace', y=0.98)

ax1, ax2 = axes

# Panel 1: GEX bars + Vanna line overlay
c1 = [GREEN if v > 0 else RED for v in gex_by_strike.values]
ax1.bar(gex_by_strike.index, gex_by_strike.values / 1e6,
        color=c1, width=0.8, alpha=0.7, label='GEX ($M)')
ax1.axhline(0, color=GREY, lw=0.8)
ax1.axvline(gld_spot, color=GOLD, lw=1.5, ls='--',
            label=f'GLD ${gld_spot:.2f} (Gold ${gold_spot:,.0f})')
if flip_level:
    ax1.axvline(flip_level, color=CYAN, lw=1.5, ls=':',
                label=f'Zero-Flip GLD ${flip_level:.1f} (Gold ${flip_level*gld_to_gold:,.0f})')
ax1b = ax1.twinx()
ax1b.plot(vex_by_strike.index, vex_by_strike.values / 1e3,
          color=BLUE, lw=1.5, alpha=0.8, label='Vanna (K)')
ax1b.axhline(0, color=BLUE, lw=0.4, ls=':')
ax1b.set_ylabel('Vanna Exposure (K)', color=BLUE, fontsize=8)
ax1b.tick_params(colors=BLUE, labelsize=7)
ax1.set_title('GEX (bars) + Vanna (line)  |  '
              'Aligned levels = highest institutional gravity',
              fontsize=9, color='#aaa', pad=4)
ax1.set_ylabel('GEX ($M)', fontsize=8)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1f}M'))
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1b.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper left')
ax1.set_facecolor('#111111')
for sp in ax1.spines.values(): sp.set_color('#2a2a2a')
ax1.tick_params(colors='#888', labelsize=7)

# Panel 2: Combined gravity score
gravity  = combined.reindex(gex_by_strike.index, fill_value=0)
gcols    = [PURPLE if v > gravity.quantile(0.75) else '#2a2a3a'
            for v in gravity.values]
ax2.bar(gravity.index, gravity.values, color=gcols, width=0.8, alpha=0.85)
ax2.axvline(gld_spot, color=GOLD, lw=1.5, ls='--',
            label=f'GLD ${gld_spot:.2f} (Gold ${gold_spot:,.0f})')
if flip_level:
    ax2.axvline(flip_level, color=CYAN, lw=1.5, ls=':',
                label=f'Zero-Flip (Gold ${flip_level*gld_to_gold:,.0f})')
for strike, score in top_combined.items():
    if score > combined.quantile(0.85):
        gold_eq = strike * gld_to_gold
        ax2.annotate(f'Gold\n${gold_eq:,.0f}',
                     xy=(strike, score),
                     xytext=(0, 10), textcoords='offset points',
                     ha='center', fontsize=6.5, color=PURPLE,
                     fontfamily='monospace',
                     arrowprops=dict(arrowstyle='->', color=PURPLE, lw=0.8))
ax2.set_title('Combined Institutional Gravity Score (GEX + Vanna)  |  '
              'Purple = highest-conviction entry/target zones',
              fontsize=9, color='#aaa', pad=4)
ax2.set_xlabel('GLD Strike Price  (see Gold equivalents in annotations)',
               fontsize=8)
ax2.set_ylabel('Gravity Score', fontsize=8)
ax2.legend(fontsize=8)
ax2.set_facecolor('#111111')
for sp in ax2.spines.values(): sp.set_color('#2a2a2a')
ax2.tick_params(colors='#888', labelsize=7)

plt.tight_layout(rect=[0, 0, 1, 0.96])
# ── Inline display ─────────────────────────────────────
import io as _io, IPython.display as _ipyd
_buf = _io.BytesIO()
plt.savefig(_buf, dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.savefig('vanna_vix_alignment.png', dpi=150,
            bbox_inches='tight', facecolor='#0d0d0d')
_buf.seek(0)
_ipyd.display(_ipyd.Image(_buf.read()))
plt.close()
print("\n✓ Saved: vanna_vix_alignment.png")


In [ ]:
# =============================================================================
# CHARM DECAY  |  Weekend Theta → Monday Forced Flows
# =============================================================================

print("=" * 60)
print("  CHARM DECAY  |  Weekend Theta → Monday Forced Flows")
print("=" * 60)

def bs_charm(S, K, T, r, sigma, flag):
    if T <= 0 or sigma <= 0:
        return 0.0
    try:
        d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        pdf_d1 = norm.pdf(d1)
        charm  = -pdf_d1 * (2 * r * T - d2 * sigma * np.sqrt(T)) / \
                 (2 * T * sigma * np.sqrt(T))
        return charm
    except:
        return 0.0

DAYS_PASSED = 3 / 365   # weekend = 3 calendar days

df_opts['charm'] = df_opts.apply(
    lambda row: bs_charm(gld_spot, row['strike'], row['T'],
                         r, gvz, row['type']), axis=1)

df_opts['charm_flow'] = df_opts.apply(
    lambda row: row['oi'] * row['charm'] * CONTRACT * DAYS_PASSED
                * (1 if row['type'] == 'call' else -1), axis=1)

charm_by_strike = (
    df_opts[df_opts['strike'].between(lo_strike, hi_strike)]
    .groupby('strike')['charm_flow'].sum()
    .sort_index()
)
total_charm_flow = charm_by_strike.sum()

print(f"\n  GLD Spot       : ${gld_spot:.2f}")
print(f"  Gold Futures   : ${gold_spot:,.2f}")
print(f"  GLD→Gold Ratio : {gld_to_gold:.4f}x")
print(f"  Weekend decay  : 3 calendar days")
print(f"\n  Total Charm Flow : {total_charm_flow/1e3:+.2f}K delta")
if total_charm_flow > 0:
    print(f"  Monday Bias      : BULLISH — time decay forces dealers to net BUY delta")
else:
    print(f"  Monday Bias      : BEARISH — time decay forces dealers to net SELL delta")

print(f"\n  ─── Top Charm Strike Levels (biggest Monday forced flows) ──")
print(f"  {'GLD Strike':>12}  {'Gold ~':>10}  {'Charm Flow':>12}  "
      f"{'Direction':>20}  {'Dist':>8}")
print(f"  {'─'*70}")

top_charm_idx = charm_by_strike.abs().nlargest(10).index
for strike in sorted(top_charm_idx):
    cf      = charm_by_strike.get(strike, 0)
    dist    = ((strike - gld_spot) / gld_spot) * 100
    gold_eq = strike * gld_to_gold
    dirn    = '▲ dealers BUY' if cf > 0 else '▼ dealers SELL'
    print(f"  ${strike:>10.0f}  ~${gold_eq:>8,.0f}  {cf/1e3:>+10.2f}K  "
          f"{dirn:>20}  {dist:>+6.1f}%")

# ── Chart ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(18, 10), sharex=True)
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle(f'CHARM DECAY  //  Monday Open Forced Flows  |  '
             f'GLD ${gld_spot:.2f}  (Gold ${gold_spot:,.0f})  3-day weekend',
             fontsize=12, color=GOLD, fontfamily='monospace', y=0.98)

ax1, ax2 = axes

ccols = [GREEN if v > 0 else RED for v in charm_by_strike.values]
ax1.bar(charm_by_strike.index, charm_by_strike.values / 1e3,
        color=ccols, width=0.8, alpha=0.85)
ax1.axhline(0, color=GREY, lw=0.8)
ax1.axvline(gld_spot, color=GOLD, lw=1.5, ls='--',
            label=f'GLD ${gld_spot:.2f} (Gold ${gold_spot:,.0f})')
if flip_level:
    ax1.axvline(flip_level, color=CYAN, lw=1.2, ls=':',
                label=f'Zero-Flip (Gold ${flip_level*gld_to_gold:,.0f})')

# Annotate top charm strikes with gold price
for strike in charm_by_strike.abs().nlargest(5).index:
    cf      = charm_by_strike.get(strike, 0)
    gold_eq = strike * gld_to_gold
    ax1.text(strike, cf / 1e3 + (0.02 if cf > 0 else -0.04),
             f'${gold_eq:,.0f}',
             ha='center', fontsize=6.5,
             color=GREEN if cf > 0 else RED,
             fontfamily='monospace')

ax1.set_title('Charm Flow by Strike  |  '
              'Green=dealers forced BUY Monday open  Red=dealers forced SELL  '
              'Labels=Gold equivalent price',
              fontsize=9, color='#aaa', pad=4)
ax1.set_ylabel('Charm Flow (K delta)', fontsize=8)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1f}K'))
ax1.legend(fontsize=8)
ax1.set_facecolor('#111111')
for sp in ax1.spines.values(): sp.set_color('#2a2a2a')
ax1.tick_params(colors='#888', labelsize=7)

cum_charm = charm_by_strike.cumsum()
ax2.plot(cum_charm.index, cum_charm.values / 1e3,
         color=ORANGE, lw=1.5, label='Cumulative charm flow')
ax2.fill_between(cum_charm.index, 0, cum_charm.values / 1e3,
                 where=cum_charm.values > 0, color=GREEN, alpha=0.15)
ax2.fill_between(cum_charm.index, 0, cum_charm.values / 1e3,
                 where=cum_charm.values < 0, color=RED, alpha=0.15)
ax2.axhline(0, color=GREY, lw=0.8)
ax2.axvline(gld_spot, color=GOLD, lw=1.5, ls='--',
            label=f'GLD ${gld_spot:.2f} (Gold ${gold_spot:,.0f})')
if flip_level:
    ax2.axvline(flip_level, color=CYAN, lw=1.2, ls=':',
                label=f'Zero-Flip (Gold ${flip_level*gld_to_gold:,.0f})')
ax2.set_title(f'Cumulative Charm Flow  |  Total: {total_charm_flow/1e3:+.2f}K  →  '
              f'{"NET BUY bias Monday" if total_charm_flow > 0 else "NET SELL bias Monday"}',
              fontsize=9,
              color=GREEN if total_charm_flow > 0 else RED, pad=4)
ax2.set_xlabel('GLD Strike  (Gold equivalents annotated on top panel)', fontsize=8)
ax2.set_ylabel('Cumulative Flow (K delta)', fontsize=8)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1f}K'))
ax2.legend(fontsize=8)
ax2.set_facecolor('#111111')
for sp in ax2.spines.values(): sp.set_color('#2a2a2a')
ax2.tick_params(colors='#888', labelsize=7)

plt.tight_layout(rect=[0, 0, 1, 0.96])
# ── Inline display ─────────────────────────────────────
import io as _io, IPython.display as _ipyd
_buf = _io.BytesIO()
plt.savefig(_buf, dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.savefig('charm_decay_monday.png', dpi=150,
            bbox_inches='tight', facecolor='#0d0d0d')
_buf.seek(0)
_ipyd.display(_ipyd.Image(_buf.read()))
plt.close()
print("\n✓ Saved: charm_decay_monday.png")


In [ ]:
# =============================================================================
# FULL OPTIONS GREEKS SUMMARY DASHBOARD
# GEX + Vanna + Charm + Delta + P/C — single unified view
# =============================================================================

print("=" * 60)
print("  FULL OPTIONS GREEKS SUMMARY DASHBOARD")
print("=" * 60)

# ── P/C ratio ─────────────────────────────────────────────────────────────────
calls_df      = df_opts[df_opts['type'] == 'call']
puts_df       = df_opts[df_opts['type'] == 'put']
total_call_oi = calls_df['oi'].sum()
total_put_oi  = puts_df['oi'].sum()
pc_ratio      = total_put_oi / (total_call_oi + 1e-10)

# ── Combined gravity score ────────────────────────────────────────────────────
all_strikes = sorted(set(gex_by_strike.index) |
                     set(vex_by_strike.index) |
                     set(charm_by_strike.index))

level_df = pd.DataFrame(index=all_strikes)
level_df['gex']   = gex_by_strike.reindex(all_strikes,   fill_value=0)
level_df['vex']   = vex_by_strike.reindex(all_strikes,   fill_value=0)
level_df['charm'] = charm_by_strike.reindex(all_strikes, fill_value=0)
level_df['dex']   = dex_by_strike.reindex(all_strikes,   fill_value=0)

for col in ['gex', 'vex', 'charm', 'dex']:
    mx = level_df[col].abs().max()
    level_df[f'{col}_norm'] = level_df[col].abs() / mx if mx > 0 else 0

level_df['total_gravity'] = (
    level_df['gex_norm']   * 0.35 +
    level_df['vex_norm']   * 0.30 +
    level_df['charm_norm'] * 0.20 +
    level_df['dex_norm']   * 0.15
)

near       = level_df[level_df.index.to_series().between(
                 gld_spot * 0.80, gld_spot * 1.20)]
top_levels = near.nlargest(10, 'total_gravity')

# ── Bias scorecard ────────────────────────────────────────────────────────────
net_gex   = gex_by_strike.sum()
net_vex   = vex_by_strike.sum()
net_charm = charm_by_strike.sum()
net_dex   = total_dex

scores = {
    'P/C Ratio' : 1 if pc_ratio < 0.7 else (-1 if pc_ratio > 1.1 else 0),
    'Net GEX'   : 1 if net_gex > 0   else -1,
    'Net Vanna' : 1 if net_vex > 0   else -1,
    'Charm Flow': 1 if net_charm > 0  else -1,
    'Net Delta' : 1 if net_dex > 0   else -1,
}
bull_count = sum(1 for v in scores.values() if v ==  1)
bear_count = sum(1 for v in scores.values() if v == -1)

if   bull_count >= 4: options_bias, bias_col = 'STRONGLY BULLISH', GREEN
elif bull_count == 3: options_bias, bias_col = 'BULLISH LEAN',     GREEN
elif bear_count >= 4: options_bias, bias_col = 'STRONGLY BEARISH', RED
elif bear_count == 3: options_bias, bias_col = 'BEARISH LEAN',     RED
else:                 options_bias, bias_col = 'NEUTRAL / MIXED',  GREY

# ── Print summary ─────────────────────────────────────────────────────────────
print(f"\n  ══════════════════════════════════════════════════════════")
print(f"  OPTIONS GREEKS BIAS SUMMARY")
print(f"  GLD ${gld_spot:.2f}  |  Gold ${gold_spot:,.2f}  |  Ratio {gld_to_gold:.4f}x")
print(f"  ══════════════════════════════════════════════════════════")
print(f"\n  {'Metric':<22}  {'Value':>14}  {'Signal':>24}")
print(f"  {'─'*64}")
print(f"  {'P/C OI Ratio':<22}  {pc_ratio:>13.3f}  "
      f"{'✦ Bullish skew' if pc_ratio < 0.7 else ('⚠ Bearish skew' if pc_ratio > 1.1 else '– Neutral'):>24}")
print(f"  {'Net GEX':<22}  {net_gex/1e6:>+12.1f}M  "
      f"{'✦ Dealers long gamma' if net_gex > 0 else '⚠ Dealers short gamma':>24}")
if flip_level:
    flip_gold = flip_level * gld_to_gold
    regime    = 'ABOVE=suppressed' if gld_spot > flip_level else 'BELOW=amplified'
    print(f"  {'Zero-Gamma Flip':<22}  GLD${flip_level:>7.1f}  "
          f"Gold ${flip_gold:>7,.0f}  {regime}")
print(f"  {'Net Vanna (VEX)':<22}  {net_vex/1e3:>+12.1f}K  "
      f"{'✦ VIX drop=BUY flow' if net_vex > 0 else '⚠ VIX drop=SELL flow':>24}")
print(f"  {'Charm (Mon flow)':<22}  {net_charm/1e3:>+12.2f}K  "
      f"{'✦ Net buy Monday' if net_charm > 0 else '⚠ Net sell Monday':>24}")
print(f"  {'Net Delta (DEX)':<22}  {net_dex/1e3:>+12.1f}K  "
      f"{'✦ Dealer long bias' if net_dex > 0 else '⚠ Dealer short bias':>24}")
print(f"  {'GVZ':<22}  {current_gvz:>13.1f}%  "
      f"{'⚠ Elevated IV' if current_gvz > 30 else ('✦ Low IV' if current_gvz < 18 else '– Normal IV'):>24}")
print(f"\n  Options Score : {bull_count} bullish  |  {bear_count} bearish")
print(f"  Options Bias  : {options_bias}")

print(f"\n  ─── Top 10 Institutional Gravity Levels ────────────────────")
print(f"  {'GLD Strike':>12}  {'Gold ~':>10}  {'Dist':>7}  "
      f"{'GEX Role':>12}  {'Vanna':>14}  {'Charm':>10}  {'Score':>7}")
print(f"  {'─'*82}")
for strike, row in top_levels.iterrows():
    dist     = ((strike - gld_spot) / gld_spot) * 100
    gold_eq  = strike * gld_to_gold
    gex_role = 'RESISTANCE' if row['gex'] > 0 else 'SUPPORT'
    van_role = 'BUY flow'   if row['vex'] > 0 else 'SELL flow'
    chm_role = '▲ buy'      if row['charm'] > 0 else '▼ sell'
    print(f"  ${strike:>10.0f}  ~${gold_eq:>8,.0f}  {dist:>+5.1f}%  "
          f"{gex_role:>12}  {van_role:>14}  {chm_role:>10}  "
          f"{row['total_gravity']:>7.3f}")

# ── Chart ─────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 16))
fig.patch.set_facecolor('#0d0d0d')
gs = gridspec.GridSpec(3, 2, figure=fig,
                       hspace=0.55, wspace=0.35,
                       left=0.06, right=0.97,
                       top=0.93, bottom=0.05)
fig.suptitle(f'FULL OPTIONS GREEKS DASHBOARD  //  '
             f'GLD ${gld_spot:.2f}  Gold ${gold_spot:,.0f}  '
             f'Ratio {gld_to_gold:.2f}x  |  Bias: {options_bias}',
             fontsize=12, color=bias_col,
             fontfamily='monospace', y=0.97)

def style(ax):
    ax.set_facecolor('#111111')
    ax.tick_params(colors='#888', labelsize=7)
    for sp in ax.spines.values(): sp.set_color('#2a2a2a')
    ax.grid(True, color='#1a1a1a', lw=0.4)

def add_vlines(ax):
    ax.axvline(gld_spot, color=GOLD, lw=1.2, ls='--',
               label=f'GLD ${gld_spot:.2f} (Gold ${gold_spot:,.0f})')
    if flip_level:
        ax.axvline(flip_level, color=CYAN, lw=1.2, ls=':',
                   label=f'Flip GLD ${flip_level:.0f} (Gold ${flip_level*gld_to_gold:,.0f})')

# P1: GEX (full width)
ax1 = fig.add_subplot(gs[0, :])
style(ax1)
gcols = [GREEN if v > 0 else RED for v in gex_by_strike.values]
ax1.bar(gex_by_strike.index, gex_by_strike.values / 1e6,
        color=gcols, width=0.8, alpha=0.85, label='GEX')
ax1.axhline(0, color=GREY, lw=0.8)
add_vlines(ax1)
for strike in top_levels.index[:5]:
    gold_eq = strike * gld_to_gold
    ax1.axvline(strike, color=PURPLE, lw=0.8, ls=':', alpha=0.7)
    ax1.text(strike + 0.2, 0, f'Gold\n${gold_eq:,.0f}',
             color=PURPLE, fontsize=6, rotation=0,
             va='bottom', fontfamily='monospace')
ax1.set_title('GEX by Strike  |  Purple = top gravity levels  '
              'Cyan = Zero-Gamma Flip  |  All labels in Gold ($) equivalent',
              fontsize=9, color='#aaa', pad=4)
ax1.set_ylabel('GEX ($M)', fontsize=8)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1f}M'))
ax1.legend(fontsize=8, loc='upper left')

# P2: Vanna
ax2 = fig.add_subplot(gs[1, 0])
style(ax2)
vcols = [BLUE if v > 0 else ORANGE for v in vex_by_strike.values]
ax2.bar(vex_by_strike.index, vex_by_strike.values / 1e3,
        color=vcols, width=0.8, alpha=0.85)
ax2.axhline(0, color=GREY, lw=0.8)
add_vlines(ax2)
ax2.set_title('Vanna (VEX)  |  Blue=VIX drop→BUY  Orange=VIX drop→SELL',
              fontsize=9, color='#aaa', pad=4)
ax2.set_ylabel('VEX (K)', fontsize=8)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}K'))
ax2.legend(fontsize=7)

# P3: Charm
ax3 = fig.add_subplot(gs[1, 1])
style(ax3)
chcols = [GREEN if v > 0 else RED for v in charm_by_strike.values]
ax3.bar(charm_by_strike.index, charm_by_strike.values / 1e3,
        color=chcols, width=0.8, alpha=0.85)
ax3.axhline(0, color=GREY, lw=0.8)
add_vlines(ax3)
for strike in charm_by_strike.abs().nlargest(4).index:
    cf      = charm_by_strike.get(strike, 0)
    gold_eq = strike * gld_to_gold
    ax3.text(strike, cf / 1e3 + (0.015 if cf > 0 else -0.03),
             f'${gold_eq:,.0f}',
             ha='center', fontsize=6,
             color=GREEN if cf > 0 else RED,
             fontfamily='monospace')
ax3.set_title(f'Charm (Mon open flow)  |  Total: {net_charm/1e3:+.2f}K  '
              f'→ {"NET BUY" if net_charm > 0 else "NET SELL"}',
              fontsize=9, color=GREEN if net_charm > 0 else RED, pad=4)
ax3.set_ylabel('Charm Flow (K)', fontsize=8)
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1f}K'))
ax3.legend(fontsize=7)

# P4: Delta
ax4 = fig.add_subplot(gs[2, 0])
style(ax4)
dcols = [GREEN if v > 0 else RED for v in dex_by_strike.values]
ax4.bar(dex_by_strike.index, dex_by_strike.values / 1e3,
        color=dcols, width=0.8, alpha=0.85)
ax4.axhline(0, color=GREY, lw=0.8)
add_vlines(ax4)
ax4.set_title(f'Net Delta (DEX)  |  Total: {net_dex/1e3:+.0f}K  '
              f'→ {"Dealer long bias" if net_dex > 0 else "Dealer short bias"}',
              fontsize=9, color=GREEN if net_dex > 0 else RED, pad=4)
ax4.set_ylabel('DEX (K shares)', fontsize=8)
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}K'))
ax4.legend(fontsize=7)

# P5: Scorecard text panel
ax5 = fig.add_subplot(gs[2, 1])
ax5.axis('off')
ax5.set_facecolor('#0a0a14')

ax5.text(0.05, 0.97, 'OPTIONS GREEKS SCORECARD',
         transform=ax5.transAxes, fontsize=9,
         color=GOLD, va='top', fontfamily='monospace', fontweight='bold')

flip_gold  = flip_level * gld_to_gold if flip_level else 0
flip_str   = f'GLD ${flip_level:.1f}  Gold ${flip_gold:,.0f}' if flip_level else 'N/A'
regime_str = ('ABOVE → suppressed' if (flip_level and gld_spot > flip_level)
              else 'BELOW → amplified')

scorecard_lines = [
    (f"P/C Ratio  : {pc_ratio:.3f}",
     GREEN if pc_ratio < 0.7 else (RED if pc_ratio > 1.1 else GREY)),
    (f"Net GEX    : {net_gex/1e6:+.1f}M",
     GREEN if net_gex > 0 else RED),
    (f"Flip Level : {flip_str}", CYAN),
    (f"Regime     : {regime_str}",
     GREEN if (flip_level and gld_spot > flip_level) else ORANGE),
    (f"Net Vanna  : {net_vex/1e3:+.1f}K",
     GREEN if net_vex > 0 else RED),
    (f"Charm Mon  : {net_charm/1e3:+.2f}K",
     GREEN if net_charm > 0 else RED),
    (f"Net Delta  : {net_dex/1e3:+.0f}K",
     GREEN if net_dex > 0 else RED),
    (f"GVZ        : {current_gvz:.1f}%",
     RED if current_gvz > 30 else (GREEN if current_gvz < 18 else GREY)),
    ('', GREY),
    (f"Score : {bull_count} bull  /  {bear_count} bear", GOLD),
    (f"BIAS  : {options_bias}", bias_col),
]

for i, (txt, col) in enumerate(scorecard_lines):
    ax5.text(0.05, 0.82 - i * 0.072, txt,
             transform=ax5.transAxes, fontsize=8.5,
             color=col, va='top', fontfamily='monospace')

# Top gravity levels in Gold price
top_gold_levels = [f'${s*gld_to_gold:,.0f}' for s in list(top_levels.index[:5])]
ax5.text(0.05, 0.06,
         'TOP ZONES (Gold):',
         transform=ax5.transAxes, fontsize=7.5,
         color=PURPLE, va='bottom', fontfamily='monospace')
ax5.text(0.05, 0.02,
         '  |  '.join(top_gold_levels),
         transform=ax5.transAxes, fontsize=7.5,
         color=PURPLE, va='bottom', fontfamily='monospace')

# ── Inline display ─────────────────────────────────────
import io as _io, IPython.display as _ipyd
_buf = _io.BytesIO()
plt.savefig(_buf, dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.savefig('full_greeks_dashboard.png', dpi=150,
            bbox_inches='tight', facecolor='#0d0d0d')
_buf.seek(0)
_ipyd.display(_ipyd.Image(_buf.read()))
plt.close()

print("\n✓ Saved: full_greeks_dashboard.png")
print("\n" + "=" * 60)
print("  QUICK REFERENCE  |  How to use before each session")
print("=" * 60)
print(f"""
  Prices used:
    GLD spot   : ${gld_spot:.2f}
    Gold price : ${gold_spot:,.2f}
    Ratio      : {gld_to_gold:.4f}x

  STEP 1 — REGIME (Zero-Gamma Flip at Gold ${flip_gold:,.0f})
  ├─ Gold ABOVE ${flip_gold:,.0f}  →  suppressed, fade extremes
  └─ Gold BELOW ${flip_gold:,.0f}  →  amplified, ride breakouts

  STEP 2 — BIAS ({options_bias})
  ├─ 4-5 green  →  structurally bullish options positioning
  └─ 4-5 red    →  structurally bearish options positioning

  STEP 3 — VIX PRE-MARKET
  ├─ VIX drop  +  net vanna {net_vex/1e3:+.1f}K  →  {'dealers BUY' if net_vex > 0 else 'dealers SELL'}
  └─ VIX spike +  net vanna {net_vex/1e3:+.1f}K  →  {'dealers SELL' if net_vex > 0 else 'dealers BUY'}

  STEP 4 — MONDAY CHARM (net {net_charm/1e3:+.2f}K)
  └─  {'Weekend decay = NET BUY pressure at open' if net_charm > 0 else 'Weekend decay = NET SELL pressure at open'}

  STEP 5 — KEY GOLD LEVELS
  └─  {chr(10).join(['      ' + l for l in top_gold_levels])}
""")


Tier 4 - Technical Structure
¶
Where price is, and what traps exist

In [ ]:
# =============================================================================
# WEEKLY LEVELS SCANNER  //  VWAP · Prior Week H/L/C · ATR bands
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import warnings
warnings.filterwarnings('ignore'); unmute_convergence()

BG    = '#080810'
GOLD  = '#C9A84C'
GOLD2 = '#FFD700'
RED2  = '#FF4444'
BLUE2 = '#4FC3F7'
GREEN = '#00C853'

print("Fetching intraday data for VWAP and levels...")

# Daily data for level calculation
gc = yf.download('GC=F', period='3mo', interval='1d',
                  auto_adjust=True, progress=False)

if len(gc) == 0:
    print("No data returned")
else:
    gc.columns = [c[0] if isinstance(c, tuple) else c for c in gc.columns]

    # --- Prior week H/L/C
    gc.index = pd.to_datetime(gc.index)
    gc['week'] = gc.index.isocalendar().week
    gc['year'] = gc.index.isocalendar().year

    weekly = gc.groupby(['year','week']).agg(
        High=('High','max'),
        Low=('Low','min'),
        Close=('Close','last'),
        Open=('Open','first'),
        Volume=('Volume','sum')
    ).reset_index()

    if len(weekly) >= 2:
        pw = weekly.iloc[-2]   # prior week
        cw = weekly.iloc[-1]   # current week (partial if running mid-week)

        pw_high  = float(pw['High'])
        pw_low   = float(pw['Low'])
        pw_close = float(pw['Close'])
        pw_range = pw_high - pw_low

        # Pivot points (classic)
        pivot  = (pw_high + pw_low + pw_close) / 3
        r1     = 2 * pivot - pw_low
        r2     = pivot + pw_range
        r3     = pw_high + 2 * (pivot - pw_low)
        s1     = 2 * pivot - pw_high
        s2     = pivot - pw_range
        s3     = pw_low - 2 * (pw_high - pivot)

        # ATR
        high_s = gc['High']
        low_s  = gc['Low']
        close_s= gc['Close']
        tr     = pd.concat([
            high_s - low_s,
            (high_s - close_s.shift(1)).abs(),
            (low_s  - close_s.shift(1)).abs()
        ], axis=1).max(axis=1)
        atr14  = float(tr.rolling(14).mean().iloc[-1])

        # Monthly VWAP (approximate — rolling 21 day)
        typical_price = (gc['High'] + gc['Low'] + gc['Close']) / 3
        tp_vol  = (typical_price * gc['Volume']).rolling(21).sum()
        vol_sum = gc['Volume'].rolling(21).sum()
        mvwap   = (tp_vol / vol_sum)

        current_price = safe_at(gc['Close'], -1)
        mvwap_latest  = safe_at(mvwap, -1)

        print(f"\n{'='*55}")
        print(f"  GOLD WEEKLY LEVELS  |  Current: ${current_price:,.0f}")
        print(f"{'='*55}")
        print(f"  Prior Week High  : ${pw_high:,.0f}")
        print(f"  Prior Week Low   : ${pw_low:,.0f}")
        print(f"  Prior Week Close : ${pw_close:,.0f}")
        print(f"  ATR(14)          : ${atr14:,.0f}")
        print(f"  Monthly VWAP     : ${mvwap_latest:,.0f}")
        print(f"{'='*55}")
        print(f"  PIVOT LEVELS:")
        print(f"  R3 : ${r3:,.0f}   R2 : ${r2:,.0f}   R1 : ${r1:,.0f}")
        print(f"  PP : ${pivot:,.0f}")
        print(f"  S1 : ${s1:,.0f}   S2 : ${s2:,.0f}   S3 : ${s3:,.0f}")
        print(f"{'='*55}")
        print(f"  ATR BANDS from close:")
        print(f"  +1 ATR: ${current_price + atr14:,.0f}")
        print(f"  -1 ATR: ${current_price - atr14:,.0f}")
        print(f"  +2 ATR: ${current_price + 2*atr14:,.0f}")
        print(f"  -2 ATR: ${current_price - 2*atr14:,.0f}")
        print(f"{'='*55}")

        # --- Plot
        fig, axes = plt.subplots(2, 1, figsize=(16, 11), facecolor=BG,
                                  gridspec_kw={'height_ratios':[3,1]})
        fig.patch.set_facecolor(BG)
        fig.suptitle(f'GOLD WEEKLY LEVELS  //  Pivots · VWAP · ATR Bands  |  ${current_price:,.0f}',
                     fontsize=12, color=GOLD, fontweight='bold', fontfamily='monospace')

        ax = axes[0]
        ax.set_facecolor('#0a0a14')

        # Price
        ax.plot(gc.index, gc['Close'], color=GOLD2, lw=1.2, label='Gold Close', zorder=5)
        ax.plot(gc.index, mvwap,       color=BLUE2,  lw=1.0, ls='--', label='Monthly VWAP', alpha=0.8)

        # Level lines
        levels = [
            (r3, RED2,   f'R3 ${r3:,.0f}',   '--'),
            (r2, RED2,   f'R2 ${r2:,.0f}',   '--'),
            (r1, RED2,   f'R1 ${r1:,.0f}',   '-'),
            (pivot, GOLD, f'PP ${pivot:,.0f}', '-'),
            (s1, GREEN,  f'S1 ${s1:,.0f}',   '-'),
            (s2, GREEN,  f'S2 ${s2:,.0f}',   '--'),
            (s3, GREEN,  f'S3 ${s3:,.0f}',   '--'),
        ]
        for lvl, col, lbl, ls in levels:
            ax.axhline(lvl, color=col, lw=0.7, ls=ls, alpha=0.7)
            ax.text(gc.index[-1], lvl, f'  {lbl}',
                    color=col, fontsize=7, fontfamily='monospace', va='center')

        # ATR bands
        ax.axhline(current_price + atr14,   color='#555', lw=0.5, ls=':')
        ax.axhline(current_price - atr14,   color='#555', lw=0.5, ls=':')
        ax.axhline(current_price + 2*atr14, color='#444', lw=0.5, ls=':')
        ax.axhline(current_price - 2*atr14, color='#444', lw=0.5, ls=':')

        ax.set_title('Price  |  Pivot Levels  |  Monthly VWAP',
                     color=GOLD, fontsize=9, fontfamily='monospace', pad=3)
        ax.legend(fontsize=7, facecolor='#0d0d1a', edgecolor='#333', labelcolor='white')
        ax.tick_params(colors='#555', labelsize=7)
        for sp in ax.spines.values(): sp.set_color('#222')
        ax.grid(True, color='#111', lw=0.3)

        # Volume
        ax2 = axes[1]
        ax2.set_facecolor('#0a0a14')
        colors = [GOLD2 if c >= o else RED2
                  for c, o in zip(gc['Close'], gc['Open'])]
        ax2.bar(gc.index, gc['Volume'], color=colors, alpha=0.6, width=0.8)
        ax2.set_title('Volume', color=GOLD, fontsize=9, fontfamily='monospace', pad=3)
        ax2.tick_params(colors='#555', labelsize=7)
        for sp in ax2.spines.values(): sp.set_color('#222')
        ax2.grid(True, color='#111', lw=0.3)

        plt.tight_layout()
        import io as _io, IPython.display as _ipyd
        _buf = _io.BytesIO()
        plt.savefig(_buf, dpi=150, bbox_inches="tight")
        _buf.seek(0)
        _ipyd.display(_ipyd.Image(_buf.read()))
        plt.close()


In [ ]:
# ============================================================
# TPO / MARKET PROFILE CHART  |  Gold Futures GC=F
# Professional Quant Desk  —  Elena Nael
# ============================================================

import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore'); unmute_convergence()

# ─── Style ────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  '#020818',
    'axes.facecolor':    '#020818',
    'axes.edgecolor':    '#1a2540',
    'axes.labelcolor':   '#c8d0e0',
    'text.color':        '#c8d0e0',
    'xtick.color':       '#7a8aaa',
    'ytick.color':       '#7a8aaa',
    'grid.color':        '#0d1530',
    'grid.linewidth':    0.4,
    'font.family':       'monospace',
    'font.size':         8,
})

GOLD   = '#e8c840'
GREEN  = '#2ecc8a'
RED    = '#e85540'
BLUE   = '#4488ff'
PURPLE = '#aa66ff'
DIM    = '#1a2540'
BG     = '#020818'
WHITE  = '#c8d0e0'

# ─── Config ───────────────────────────────────────────────
TICKER          = 'GC=F'
TICK_SIZE       = 0.50        # $0.50 per row  (standard GC TPO granularity)
LETTERS         = 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'
TPO_PERIOD_MIN  = 30          # 30-min TPO periods (standard CME session)
DAYS_BACK       = 5           # last 5 sessions (enough for context + IB)

# ============================================================
# 1. FETCH DATA
# ============================================================
print("=" * 60)
print("  GOLD TPO / MARKET PROFILE CHART  |  GC=F")
print("=" * 60)
print(f"\n[1/5] Fetching {DAYS_BACK} days of 30-min OHLCV from yfinance ...")

raw = yf.download(
    TICKER,
    period   = f'{DAYS_BACK + 2}d',   # small buffer for weekends
    interval = '30m',
    auto_adjust = True,
    progress = False
)

if hasattr(raw, 'columns') and isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

raw = raw[['Open','High','Low','Close','Volume']].dropna()
raw.index = pd.to_datetime(raw.index)

# Force UTC-aware → UTC-naive for clean date handling
if raw.index.tz is not None:
    raw.index = raw.index.tz_convert('UTC').tz_localize(None)

# ─── Keep only the last N complete trading days ───────────
dates_available = sorted(raw.index.normalize().unique())
if len(dates_available) < 1:
    raise RuntimeError("No data returned — check internet / market hours.")

# Drop today if market is still open (incomplete session)
now_utc = pd.Timestamp.utcnow().tz_localize(None)
last_date = dates_available[-1]
if (now_utc - last_date).total_seconds() < 86400:
    # Session may be incomplete — keep it but flag
    print(f"  [i] Today ({last_date.date()}) may be partial session — included.")

use_dates = sorted(dates_available)[-DAYS_BACK:]
df = raw[raw.index.normalize().isin(use_dates)].copy()

print(f"  ✓ {len(df)} bars loaded  "
      f"[{df.index[0].strftime('%Y-%m-%d %H:%M')} → {df.index[-1].strftime('%Y-%m-%d %H:%M')}]")
print(f"  ✓ Sessions: {[d.strftime('%a %b %d') for d in use_dates]}")

# ============================================================
# 2. BUILD TPO PROFILE PER DAY
# ============================================================
print("\n[2/5] Building TPO profiles ...")

def build_tpo_profile(session_df, tick_size=0.50):
    """
    For each 30-min bar, assign a TPO letter and mark every
    price level the bar traded through.
    Returns:
        profile   : dict {price_level: [letters]}
        periods   : list of (letter, period_high, period_low, open, close)
        poc       : price of highest TPO count
        val_high  : 70% of TPO volume upper bound
        val_low   : 70% of TPO volume lower bound
        ibh       : initial balance high (first 2 periods = 1hr)
        ibl       : initial balance low
    """
    if session_df.empty:
        return None

    session_df = session_df.sort_index().reset_index()
    profile    = {}
    periods    = []

    for i, row in session_df.iterrows():
        letter  = LETTERS[i % len(LETTERS)]
        lo_snap = np.floor(row['Low']  / tick_size) * tick_size
        hi_snap = np.ceil (row['High'] / tick_size) * tick_size
        levels  = np.arange(lo_snap, hi_snap + tick_size * 0.5, tick_size)
        levels  = np.round(levels, 2)

        for lvl in levels:
            if lvl not in profile:
                profile[lvl] = []
            profile[lvl].append(letter)

        periods.append({
            'letter': letter,
            'high':   row['High'],
            'low':    row['Low'],
            'open':   row['Open'],
            'close':  row['Close'],
            'vol':    row['Volume'],
            'time':   row['index'] if 'index' in row else session_df['index'].iloc[i],
        })

    if not profile:
        return None

    # POC — price with most TPO letters
    poc = max(profile, key=lambda x: len(profile[x]))

    # Value Area — 70% of total TPOs centred on POC
    total_tpos = sum(len(v) for v in profile.values())
    target     = total_tpos * 0.70
    sorted_px  = sorted(profile.keys())
    poc_idx    = sorted_px.index(poc)
    va_count   = len(profile[poc])
    va_levels  = [poc]
    lo_ptr     = poc_idx - 1
    hi_ptr     = poc_idx + 1

    while va_count < target:
        add_hi = len(profile[sorted_px[hi_ptr]]) if hi_ptr < len(sorted_px) else 0
        add_lo = len(profile[sorted_px[lo_ptr]]) if lo_ptr >= 0             else 0
        if add_hi >= add_lo and hi_ptr < len(sorted_px):
            va_levels.append(sorted_px[hi_ptr])
            va_count += add_hi
            hi_ptr   += 1
        elif lo_ptr >= 0:
            va_levels.append(sorted_px[lo_ptr])
            va_count += add_lo
            lo_ptr   -= 1
        else:
            break

    val_high = max(va_levels)
    val_low  = min(va_levels)

    # Initial Balance — first 2 × 30-min bars (1 hour)
    ib_bars = session_df.head(2)
    ibh = ib_bars['High'].max()
    ibl = ib_bars['Low'].min()

    return {
        'profile':  profile,
        'periods':  periods,
        'poc':      poc,
        'val_high': val_high,
        'val_low':  val_low,
        'ibh':      ibh,
        'ibl':      ibl,
        'day_high': session_df['High'].max(),
        'day_low':  session_df['Low'].min(),
        'day_open': session_df['Open'].iloc[0],
        'day_close':session_df['Close'].iloc[-1],
        'date':     session_df['index'].iloc[0].normalize()
              if hasattr(session_df['index'].iloc[0], 'normalize')
              else pd.Timestamp(session_df['index'].iloc[0]).normalize(),
    }

# Build per-session
sessions = {}
for d in use_dates:
    s_df = df[df.index.normalize() == d].copy()
    s_df = s_df.reset_index()   # move datetime to column
    s_df.rename(columns={'index': 'index', s_df.columns[0]: 'index'}, inplace=True)
    s_df.columns.values[0] = 'index'
    result = build_tpo_profile(s_df, TICK_SIZE)
    if result:
        sessions[d] = result
        print(f"  ✓ {d.strftime('%a %b %d')}  |  "
              f"Range ${result['day_low']:.1f}–${result['day_high']:.1f}  |  "
              f"POC ${result['poc']:.1f}  |  "
              f"VA ${result['val_low']:.1f}–${result['val_high']:.1f}  |  "
              f"IB ${result['ibl']:.1f}–${result['ibh']:.1f}")

if not sessions:
    raise RuntimeError("No sessions built — check data fetch.")

# ============================================================
# 3. COMPOSITE PROFILE (all days merged)
# ============================================================
print("\n[3/5] Building composite profile ...")

composite = {}
for sess in sessions.values():
    for px, letters in sess['profile'].items():
        if px not in composite:
            composite[px] = []
        composite[px].extend(letters)

comp_poc = max(composite, key=lambda x: len(composite[x]))
total_comp = sum(len(v) for v in composite.values())
target_comp = total_comp * 0.70
sorted_cpx  = sorted(composite.keys())
cpoc_idx    = sorted_cpx.index(comp_poc)
cva_count   = len(composite[comp_poc])
cva_levels  = [comp_poc]
clo, chi    = cpoc_idx - 1, cpoc_idx + 1
while cva_count < target_comp:
    ah = len(composite[sorted_cpx[chi]]) if chi < len(sorted_cpx) else 0
    al = len(composite[sorted_cpx[clo]]) if clo >= 0             else 0
    if ah >= al and chi < len(sorted_cpx):
        cva_levels.append(sorted_cpx[chi]); cva_count += ah; chi += 1
    elif clo >= 0:
        cva_levels.append(sorted_cpx[clo]); cva_count += al; clo -= 1
    else:
        break
comp_val_high = max(cva_levels)
comp_val_low  = min(cva_levels)

print(f"  ✓ Composite POC: ${comp_poc:.1f}  "
      f"VA: ${comp_val_low:.1f}–${comp_val_high:.1f}")

# ============================================================
# 4. PLOT
# ============================================================
print("\n[4/5] Rendering chart ...")

n_sess = len(sessions)
fig = plt.figure(figsize=(6 * n_sess + 4, 14), facecolor=BG)

# GridSpec: daily TPO panels + composite panel
gs = GridSpec(
    1, n_sess + 1,
    figure=fig,
    width_ratios=[3] * n_sess + [2],
    wspace=0.04,
    left=0.04, right=0.97, top=0.93, bottom=0.05
)

# ─── Shared y-axis range (all days + composite) ───────────
all_prices = []
for sess in sessions.values():
    all_prices.extend(sess['profile'].keys())
all_prices.extend(composite.keys())
global_lo = min(all_prices) - TICK_SIZE * 4
global_hi = max(all_prices) + TICK_SIZE * 4

# Snap to clean $5 ticks for y-axis labels
ytick_step = 10.0   # $10 labels
yticks = np.arange(
    np.floor(global_lo / ytick_step) * ytick_step,
    np.ceil (global_hi / ytick_step) * ytick_step + ytick_step,
    ytick_step
)

# ─── Color map: letter index → hue cycle ──────────────────
PERIOD_COLORS = [
    '#e8c840','#2ecc8a','#4488ff','#e85540','#aa66ff',
    '#ff8844','#44ddff','#ff44aa','#88ff44','#ffaa00',
    '#00ffcc','#ff6666','#aaaaff','#ffcc66','#66ffaa',
]

def get_period_color(letter):
    idx = LETTERS.index(letter) if letter in LETTERS else 0
    return PERIOD_COLORS[idx % len(PERIOD_COLORS)]

axes_daily = []

for col_idx, (date, sess) in enumerate(sorted(sessions.items())):
    ax = fig.add_subplot(gs[0, col_idx])
    axes_daily.append(ax)

    profile  = sess['profile']
    poc      = sess['poc']
    val_high = sess['val_high']
    val_low  = sess['val_low']
    ibh      = sess['ibh']
    ibl      = sess['ibl']

    # Sorted price levels
    sorted_levels = sorted(profile.keys())
    max_count     = max(len(v) for v in profile.values())

    # ── Value Area background ──────────────────────────────
    ax.axhspan(val_low, val_high, color='#1a3a1a', alpha=0.35, zorder=0)

    # ── IB background ─────────────────────────────────────
    ax.axhspan(ibl, ibh, color='#1a1a3a', alpha=0.20, zorder=0)

    # ── Draw TPO letters ──────────────────────────────────
    char_width  = 1.0
    for px_lvl in sorted_levels:
        letters_at = profile[px_lvl]
        for char_idx, letter in enumerate(letters_at):
            col = get_period_color(letter)
            ax.text(
                char_idx * char_width + char_width * 0.5,
                px_lvl,
                letter,
                color=col,
                fontsize=6.5,
                va='center', ha='center',
                fontweight='bold',
                fontfamily='monospace',
            )

    # ── POC line ──────────────────────────────────────────
    ax.axhline(poc, color=GOLD, linewidth=1.2, linestyle='-', alpha=0.9,
               zorder=5, label=f'POC ${poc:.1f}')

    # ── VAH / VAL lines ───────────────────────────────────
    ax.axhline(val_high, color=GREEN, linewidth=0.8, linestyle='--', alpha=0.8,
               zorder=5, label=f'VAH ${val_high:.1f}')
    ax.axhline(val_low,  color=RED,   linewidth=0.8, linestyle='--', alpha=0.8,
               zorder=5, label=f'VAL ${val_low:.1f}')

    # ── IB lines ──────────────────────────────────────────
    ax.axhline(ibh, color=BLUE, linewidth=0.7, linestyle=':', alpha=0.7,
               zorder=5, label=f'IBH ${ibh:.1f}')
    ax.axhline(ibl, color=BLUE, linewidth=0.7, linestyle=':', alpha=0.7,
               zorder=5, label=f'IBL ${ibl:.1f}')

    # ── Open / Close markers ──────────────────────────────
    ax.axhline(sess['day_open'],  color=WHITE,  linewidth=0.5,
               linestyle=(0,(2,4)), alpha=0.5, zorder=4)
    ax.axhline(sess['day_close'], color=PURPLE, linewidth=0.6,
               linestyle=(0,(2,4)), alpha=0.7, zorder=4)

    # ── Axis formatting ───────────────────────────────────
    ax.set_ylim(global_lo, global_hi)
    ax.set_xlim(-0.5, max_count + 0.5)
    ax.set_yticks(yticks)
    ax.set_facecolor(BG)
    ax.grid(axis='y', alpha=0.15, linewidth=0.4)
    ax.tick_params(axis='x', which='both', bottom=False, labelbottom=False)

    if col_idx == 0:
        ax.set_yticklabels([f'${y:,.0f}' for y in yticks], fontsize=7, color='#7a8aaa')
    else:
        ax.set_yticklabels([])
        ax.tick_params(axis='y', length=0)

    # ── Day title ─────────────────────────────────────────
    day_range = sess['day_high'] - sess['day_low']
    ax.set_title(
        f"{date.strftime('%a  %b %d')}\n"
        f"Range ${sess['day_low']:,.0f}–${sess['day_high']:,.0f}  Δ${day_range:.0f}",
        color=GOLD, fontsize=8, pad=6, fontfamily='monospace'
    )

    # ── Legend (right side of each panel) ─────────────────
    legend_items = [
        mpatches.Patch(color=GOLD,  label=f'POC  ${poc:.1f}'),
        mpatches.Patch(color=GREEN, label=f'VAH  ${val_high:.1f}'),
        mpatches.Patch(color=RED,   label=f'VAL  ${val_low:.1f}'),
        mpatches.Patch(color=BLUE,  label=f'IBH  ${ibh:.1f}'),
        mpatches.Patch(color=BLUE,  label=f'IBL  ${ibl:.1f}'),
        mpatches.Patch(color=WHITE, label=f'Open ${sess["day_open"]:.1f}'),
        mpatches.Patch(color=PURPLE,label=f'Close${sess["day_close"]:.1f}'),
    ]
    ax.legend(
        handles=legend_items,
        loc='upper right', fontsize=6,
        facecolor='#050d1a', edgecolor='#1a2540',
        framealpha=0.85, handlelength=1.0,
        borderpad=0.5, labelspacing=0.3,
    )

    # ── Naked POC annotation (if POC not in VA of current session) ───
    # A naked POC is one price hasn't returned to — show it on all OTHER days
    ax.annotate(
        '◄ POC', xy=(max_count + 0.2, poc),
        fontsize=6, color=GOLD, va='center',
        fontfamily='monospace'
    )

# ============================================================
# COMPOSITE PROFILE PANEL
# ============================================================
ax_comp = fig.add_subplot(gs[0, n_sess])

sorted_comp = sorted(composite.keys())
max_comp    = max(len(v) for v in composite.values())

# Value Area background
ax_comp.axhspan(comp_val_low, comp_val_high, color='#1a3a1a', alpha=0.35, zorder=0)

# Horizontal histogram bars (TPO count = bar width)
for px_lvl in sorted_comp:
    count = len(composite[px_lvl])
    # Color intensity by count
    intensity = count / max_comp
    bar_color = plt.cm.YlOrRd(0.2 + intensity * 0.7)
    ax_comp.barh(
        px_lvl, count,
        height=TICK_SIZE * 0.85,
        color=bar_color,
        alpha=0.85,
        linewidth=0,
        zorder=3,
    )

# Composite POC
ax_comp.axhline(comp_poc,      color=GOLD,  linewidth=1.5, linestyle='-',  zorder=6)
ax_comp.axhline(comp_val_high, color=GREEN, linewidth=0.8, linestyle='--', zorder=6)
ax_comp.axhline(comp_val_low,  color=RED,   linewidth=0.8, linestyle='--', zorder=6)

# Previous session POC lines (naked POCs)
prev_pocs = []
sorted_sess = sorted(sessions.items())
for i, (d, s) in enumerate(sorted_sess[:-1]):  # all but last
    is_naked = not (s['val_low'] <= s['poc'] <= s['val_high']
                    for _, ss in sorted_sess[i+1:])
    ax_comp.axhline(
        s['poc'], color='#aa66ff', linewidth=0.6,
        linestyle=':', alpha=0.6, zorder=4
    )
    ax_comp.annotate(
        f"  {d.strftime('%a')} POC",
        xy=(max_comp * 0.05, s['poc']),
        fontsize=5.5, color='#aa66ff', va='center'
    )

ax_comp.set_ylim(global_lo, global_hi)
ax_comp.set_xlim(0, max_comp * 1.15)
ax_comp.set_yticks(yticks)
ax_comp.set_yticklabels([])
ax_comp.tick_params(axis='y', length=0)
ax_comp.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
ax_comp.set_facecolor(BG)
ax_comp.grid(axis='y', alpha=0.12, linewidth=0.4)
ax_comp.set_title(
    f"Composite\n{n_sess}d Profile",
    color=GOLD, fontsize=8, pad=6, fontfamily='monospace'
)

# Composite legend
comp_items = [
    mpatches.Patch(color=GOLD,   label=f'CPOC ${comp_poc:.1f}'),
    mpatches.Patch(color=GREEN,  label=f'CVAH ${comp_val_high:.1f}'),
    mpatches.Patch(color=RED,    label=f'CVAL ${comp_val_low:.1f}'),
    mpatches.Patch(color=PURPLE, label='Prior POCs'),
]
ax_comp.legend(
    handles=comp_items, loc='upper right', fontsize=6,
    facecolor='#050d1a', edgecolor='#1a2540',
    framealpha=0.85, handlelength=1.0,
    borderpad=0.5, labelspacing=0.3,
)

# ============================================================
# FIGURE TITLE
# ============================================================
current_price = df['Close'].iloc[-1]
fig.suptitle(
    f"GOLD FUTURES  GC=F  |  TPO MARKET PROFILE  |  {TICK_SIZE} tick  |  30-min periods\n"
    f"Last price ${current_price:,.2f}  ·  {df.index[-1].strftime('%Y-%m-%d %H:%M UTC')}",
    fontsize=10, color=GOLD, fontweight='bold', y=0.98,
    fontfamily='monospace'
)

# ─── Shared right-side annotation: key levels summary ─────
summary_lines = [
    f"COMPOSITE LEVELS",
    f"──────────────────",
    f"CPOC  ${comp_poc:,.1f}",
    f"CVAH  ${comp_val_high:,.1f}",
    f"CVAL  ${comp_val_low:,.1f}",
    f"",
    f"DAILY LEVELS",
    f"──────────────────",
]
for d, s in sorted(sessions.items()):
    summary_lines.append(
        f"{d.strftime('%a')}  POC ${s['poc']:.1f}  "
        f"VA ${s['val_low']:.0f}–${s['val_high']:.0f}"
    )

fig.text(
    0.994, 0.94,
    '\n'.join(summary_lines),
    fontsize=6.5, color='#8aa0cc',
    va='top', ha='right',
    fontfamily='monospace',
    transform=fig.transFigure,
)

# ============================================================
# 5. OUTPUT
# ============================================================
print("\n[5/5] Chart ready.\n")
# ── Inline display ─────────────────────────────────────
import io as _io, IPython.display as _ipyd
_buf = _io.BytesIO()
plt.savefig(_buf, dpi=150, bbox_inches='tight', facecolor=BG)
plt.savefig('gold_tpo_chart.png', dpi=150, bbox_inches='tight', facecolor=BG)
_buf.seek(0)
_ipyd.display(_ipyd.Image(_buf.read()))
plt.close()
print("  Saved → gold_tpo_chart.png")

# ─── Text summary ─────────────────────────────────────────
print("\n" + "=" * 60)
print("  TPO KEY LEVELS SUMMARY  |  Gold Futures GC=F")
print("=" * 60)
for d, s in sorted(sessions.items()):
    print(f"\n  {d.strftime('%A %b %d')}")
    print(f"  ─────────────────────────────────────")
    print(f"  Range  : ${s['day_low']:,.1f} – ${s['day_high']:,.1f}  "
          f"(Δ${s['day_high']-s['day_low']:.1f})")
    print(f"  Open   : ${s['day_open']:,.1f}  |  Close: ${s['day_close']:,.1f}")
    print(f"  POC    : ${s['poc']:,.1f}  ← highest volume node")
    print(f"  VAH    : ${s['val_high']:,.1f}  (value area high)")
    print(f"  VAL    : ${s['val_low']:,.1f}  (value area low)")
    print(f"  IBH    : ${s['ibh']:,.1f}  (initial balance high)")
    print(f"  IBL    : ${s['ibl']:,.1f}  (initial balance low)")
    ib_ext = "EXTENDED ABOVE IB" if s['day_high'] > s['ibh'] else \
             "EXTENDED BELOW IB" if s['day_low']  < s['ibl'] else \
             "NO EXTENSION — balanced day"
    print(f"  IB ext : {ib_ext}")

print(f"\n  COMPOSITE ({len(sessions)} days)")
print(f"  ─────────────────────────────────────")
print(f"  CPOC   : ${comp_poc:,.1f}")
print(f"  CVAH   : ${comp_val_high:,.1f}")
print(f"  CVAL   : ${comp_val_low:,.1f}")
print(f"\n  Current price: ${current_price:,.2f}")
location = ("ABOVE CVAH — extended, watch for rejection" if current_price > comp_val_high else
            "BELOW CVAL — extended, watch for support"  if current_price < comp_val_low  else
            "INSIDE VALUE AREA — balanced, two-sided")
print(f"  Location: {location}")
print("=" * 60)


In [ ]:
# =============================================================================
# GOLD TECHNICALS SCANNER  |  MA Stack · RSI · MACD · ATR · Key Levels
# =============================================================================

import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import rcParams
import warnings; warnings.filterwarnings('ignore'); unmute_convergence()

rcParams.update({
    'figure.facecolor':'#0d0d0d','axes.facecolor':'#111111','axes.edgecolor':'#2a2a2a',
    'axes.labelcolor':'#cccccc','axes.grid':True,'grid.color':'#1e1e1e',
    'grid.linewidth':0.5,'xtick.color':'#888888','ytick.color':'#888888',
    'text.color':'#cccccc','legend.facecolor':'#1a1a1a','legend.edgecolor':'#333333',
    'font.family':'monospace','font.size':9,
})
GOLD='#C9A84C'; GREEN='#2ecc71'; RED='#e74c3c'; BLUE='#3498db'; PURPLE='#9b59b6'; GREY='#444444'

print("Fetching OHLCV data ...")
raw  = yf.download('GC=F', period='3y', auto_adjust=True, progress=False)
px   = raw['Close'].squeeze()
hi   = raw['High'].squeeze()
lo   = raw['Low'].squeeze()

# ── Indicators ────────────────────────────────────────────────────────────────
# Moving averages
ma20  = px.rolling(20).mean()
ma50  = px.rolling(50).mean()
ma100 = px.rolling(100).mean()
ma200 = px.rolling(200).mean()

# RSI (14)
def rsi(s, n=14):
    d = s.diff(); g = d.clip(lower=0).rolling(n).mean(); l = (-d.clip(upper=0)).rolling(n).mean()
    return 100 - 100 / (1 + g / (l + 1e-10))
rsi14 = rsi(px)

# MACD (12/26/9)
ema12 = px.ewm(span=12, adjust=False).mean()
ema26 = px.ewm(span=26, adjust=False).mean()
macd  = ema12 - ema26
signal_line = macd.ewm(span=9, adjust=False).mean()
hist  = macd - signal_line

# ATR (14) — Average True Range
tr  = pd.concat([hi - lo, (hi - px.shift()).abs(), (lo - px.shift()).abs()], axis=1).max(axis=1)
atr = tr.rolling(14).mean()

# Bollinger Bands (20, 2σ)
bb_mid  = ma20
bb_std  = px.rolling(20).std()
bb_up   = bb_mid + 2 * bb_std
bb_lo   = bb_mid - 2 * bb_std
bb_pct  = (px - bb_lo) / (bb_up - bb_lo + 1e-10)   # 0=lower band, 1=upper

# Support/Resistance: rolling 20-day highs/lows
resist = hi.rolling(20).max()
supprt = lo.rolling(20).min()

# MA trend score
latest = {
    'px': px.iloc[-1], 'ma20': ma20.iloc[-1], 'ma50': ma50.iloc[-1],
    'ma100': ma100.iloc[-1], 'ma200': ma200.iloc[-1],
    'rsi': rsi14.iloc[-1], 'macd': macd.iloc[-1], 'macd_sig': signal_line.iloc[-1],
    'hist': hist.iloc[-1], 'atr': atr.iloc[-1], 'bb_pct': bb_pct.iloc[-1],
}
above = sum([latest['px'] > latest[k] for k in ['ma20','ma50','ma100','ma200']])
trend_score = above  # 0-4

print(f"\n  ─── Technical Summary  [{px.index[-1].date()}] ───────────────")
print(f"  Price          : ${latest['px']:,.2f}")
print(f"  MA Trend Score : {trend_score}/4  ({'Full bull stack' if trend_score==4 else 'Full bear stack' if trend_score==0 else 'Mixed'})")
print(f"  RSI (14)       : {latest['rsi']:.1f}  {'(overbought)' if latest['rsi']>70 else '(oversold)' if latest['rsi']<30 else '(neutral)'}")
print(f"  MACD vs Signal : {latest['macd']:+.2f} vs {latest['macd_sig']:+.2f}  {'▲ bullish cross' if latest['hist']>0 else '▼ bearish cross'}")
print(f"  ATR (14)       : ${latest['atr']:.2f}  ({latest['atr']/latest['px']*100:.2f}% of price)")
print(f"  BB %B          : {latest['bb_pct']*100:.1f}%  {'(near upper band)' if latest['bb_pct']>0.8 else '(near lower band)' if latest['bb_pct']<0.2 else '(mid-band)'}")

# ── Dashboard ─────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 16))
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle('GOLD TECHNICALS SCANNER  //  MA Stack · Bollinger · RSI · MACD · ATR',
             fontsize=12, color=GOLD, fontfamily='monospace', y=0.98)

gs  = gridspec.GridSpec(5, 1, figure=fig, hspace=0.5,
                         left=0.07, right=0.97, top=0.95, bottom=0.04,
                         height_ratios=[3, 1, 1, 1, 1])
start = px.index[-504]

# P1: Price + MAs + Bollinger
ax1 = fig.add_subplot(gs[0])
ax1.fill_between(px.loc[start:].index, bb_lo.loc[start:], bb_up.loc[start:],
                 color=GREY, alpha=0.15, label='BB (20,2σ)')
ax1.plot(px.loc[start:],   color=GOLD,   lw=1.3, label=f'Gold ${latest["px"]:,.0f}')
ax1.plot(ma20.loc[start:], color='white', lw=0.7, ls='--', label='MA20')
ax1.plot(ma50.loc[start:], color=BLUE,   lw=0.9, label='MA50')
ax1.plot(ma100.loc[start:],color='#e67e22', lw=0.9, label='MA100')
ax1.plot(ma200.loc[start:],color=RED,    lw=1.0, label='MA200')
ax1.set_title(f'XAU/USD  |  Trend Score {trend_score}/4  |  ATR=${latest["atr"]:.1f}',
              fontsize=9, color='#aaa', pad=3)
ax1.legend(fontsize=7, loc='upper left', ncol=6)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))

# Shade regime
ax1.axhline(latest['ma200'], color=RED, lw=0.5, ls=':', alpha=0.5)

# P2: RSI
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax2.plot(rsi14.loc[start:], color=PURPLE, lw=0.9, label='RSI 14')
ax2.axhline(70, color=RED,   lw=0.7, ls='--')
ax2.axhline(30, color=GREEN, lw=0.7, ls='--')
ax2.axhline(50, color=GREY,  lw=0.5, ls=':')
ax2.fill_between(rsi14.loc[start:].index, 70, rsi14.loc[start:].clip(70), color=RED,   alpha=0.2)
ax2.fill_between(rsi14.loc[start:].index, rsi14.loc[start:].clip(upper=30), 30, color=GREEN, alpha=0.2)
ax2.set_title(f'RSI (14) = {latest["rsi"]:.1f}', fontsize=9, color='#aaa', pad=3)
ax2.set_ylim(10, 90)
ax2.legend(fontsize=7)

# P3: MACD
ax3 = fig.add_subplot(gs[2], sharex=ax1)
ax3.plot(macd.loc[start:],        color=BLUE,  lw=0.9, label='MACD')
ax3.plot(signal_line.loc[start:], color=RED,   lw=0.9, label='Signal')
ax3.bar(hist.loc[start:].index, hist.loc[start:],
        color=[GREEN if v > 0 else RED for v in hist.loc[start:]], width=1, alpha=0.6, label='Histogram')
ax3.axhline(0, color=GREY, lw=0.7)
ax3.set_title('MACD (12,26,9)', fontsize=9, color='#aaa', pad=3)
ax3.legend(fontsize=7)

# P4: ATR (normalised %)
atr_pct = atr / px * 100
ax4 = fig.add_subplot(gs[3], sharex=ax1)
ax4.plot(atr_pct.loc[start:], color='#e67e22', lw=0.9, label='ATR %')
ax4.fill_between(atr_pct.loc[start:].index, atr_pct.loc[start:].rolling(63).mean().loc[start:],
                 atr_pct.loc[start:], alpha=0.15, color='#e67e22')
ax4.set_title(f'ATR% (14-day)  |  Current = {atr_pct.iloc[-1]:.2f}%  |  Higher = wider expected range',
              fontsize=9, color='#aaa', pad=3)
ax4.legend(fontsize=7)

# P5: BB %B
ax5 = fig.add_subplot(gs[4], sharex=ax1)
ax5.plot(bb_pct.loc[start:] * 100, color=GOLD, lw=0.9, label='BB %B')
ax5.axhline(100, color=RED,   lw=0.6, ls='--')
ax5.axhline(0,   color=GREEN, lw=0.6, ls='--')
ax5.axhline(50,  color=GREY,  lw=0.5, ls=':')
ax5.set_title('Bollinger %B  |  100=upper band, 0=lower band', fontsize=9, color='#aaa', pad=3)
ax5.legend(fontsize=7)
ax5.tick_params(axis='x', rotation=30)

# ── Inline display ─────────────────────────────────────
import io as _io, IPython.display as _ipyd
_buf = _io.BytesIO()
plt.savefig(_buf, dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.savefig('gold_technicals.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
_buf.seek(0)
_ipyd.display(_ipyd.Image(_buf.read()))
plt.close()
print("✓ Saved: gold_technicals.png")


Wild Cards
¶
BTC correlation, oil, SGE premium - signals funds always monitor

In [ ]:
# =============================================================================
# WILD CARD SCANNER  |  BTC Correlation · WTI Oil · SGE Premium Proxy
# =============================================================================

import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import rcParams
import warnings; warnings.filterwarnings('ignore'); unmute_convergence()

rcParams.update({
    'figure.facecolor':'#0d0d0d','axes.facecolor':'#111111','axes.edgecolor':'#2a2a2a',
    'axes.labelcolor':'#cccccc','axes.grid':True,'grid.color':'#1e1e1e',
    'grid.linewidth':0.5,'xtick.color':'#888888','ytick.color':'#888888',
    'text.color':'#cccccc','legend.facecolor':'#1a1a1a','legend.edgecolor':'#333333',
    'font.family':'monospace','font.size':9,
})
GOLD='#C9A84C'; GREEN='#2ecc71'; RED='#e74c3c'; BLUE='#3498db'; GREY='#444444'
ORANGE='#e67e22'; BTC_COL='#f7931a'

print("Fetching wild card data ...")

raw = yf.download(['GC=F','BTC-USD','CL=F','GLD'], period='2y',
                   auto_adjust=True, progress=False)['Close']
raw.columns = ['BTC','Oil_WTI','Gold','GLD']
raw = raw.ffill().dropna()

rets = np.log(raw / raw.shift(1)).dropna()

# Rolling correlations (21d, 63d)
corr_btc_21  = rets['Gold'].rolling(21).corr(rets['BTC'])
corr_btc_63  = rets['Gold'].rolling(63).corr(rets['BTC'])
corr_oil_21  = rets['Gold'].rolling(21).corr(rets['Oil_WTI'])
corr_oil_63  = rets['Gold'].rolling(63).corr(rets['Oil_WTI'])

# BTC momentum (risk-on proxy)
btc_mom_1w   = raw['BTC'].pct_change(5)  * 100
btc_mom_1m   = raw['BTC'].pct_change(21) * 100
gold_mom_1m  = raw['Gold'].pct_change(21) * 100

# Oil momentum (inflation expectation proxy)
oil_mom_1m   = raw['Oil_WTI'].pct_change(21) * 100
oil_ma50     = raw['Oil_WTI'].rolling(50).mean()

# SGE premium proxy: approximate as LBMA (GC=F) vs GLD-adjusted
# Real SGE data requires Bloomberg; proxy = rolling z-score of Gold/GLD ratio
gld_ratio    = raw['Gold'] / (raw['GLD'] * LIVE_RATIO)   # was * 10
sge_proxy_z  = (gld_ratio - gld_ratio.rolling(63).mean()) / (gld_ratio.rolling(63).std() + 1e-10)

latest = raw.iloc[-1]
print(f"\n  ─── Wild Card Snapshot ──────────────────────────────")
print(f"  Gold          : ${latest['Gold']:,.2f}  (1m {gold_mom_1m.iloc[-1]:+.1f}%)")
print(f"  BTC           : ${latest['BTC']:,.0f}  (1w {btc_mom_1w.iloc[-1]:+.1f}%, 1m {btc_mom_1m.iloc[-1]:+.1f}%)")
print(f"  WTI Oil       : ${latest['Oil_WTI']:.2f}  (1m {oil_mom_1m.iloc[-1]:+.1f}%)")
print(f"\n  Gold/BTC Corr (21d) : {corr_btc_21.iloc[-1]:+.3f}  {'(risk-on aligned)' if corr_btc_21.iloc[-1]>0.2 else '(diverging)' if corr_btc_21.iloc[-1]<-0.1 else '(uncorrelated)'}")
print(f"  Gold/Oil Corr (21d) : {corr_oil_21.iloc[-1]:+.3f}  {'(inflation bid active)' if corr_oil_21.iloc[-1]>0.2 else ''}")
print(f"  SGE Proxy z-score   : {sge_proxy_z.iloc[-1]:+.2f}  {'(premium = physical demand surge)' if sge_proxy_z.iloc[-1]>1 else '(discount = weak China)' if sge_proxy_z.iloc[-1]<-1 else '(neutral)'}")

# ── Dashboard ─────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle('WILD CARD SCANNER  //  BTC · WTI Oil · SGE Premium Proxy  |  Gold Context',
             fontsize=12, color=GOLD, fontfamily='monospace', y=0.98)

gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.5, wspace=0.3,
                        left=0.07, right=0.97, top=0.95, bottom=0.05)
start = raw.index[-504]

# P1: Gold vs BTC normalised
ax1 = fig.add_subplot(gs[0, :])
gold_n = raw['Gold'].loc[start:] / raw['Gold'].loc[start:].iloc[0]
btc_n  = raw['BTC'].loc[start:] / raw['BTC'].loc[start:].iloc[0]
ax1.plot(gold_n, color=GOLD,    lw=1.2, label='Gold (rebased)')
ax1.plot(btc_n,  color=BTC_COL, lw=0.9, label='BTC (rebased)', alpha=0.8)
ax1.set_title('Gold vs BTC  |  Rebased to 1.0', fontsize=9, color='#aaa', pad=3)
ax1.legend(fontsize=8)

# P2: Rolling BTC/Gold correlation
ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(corr_btc_21.loc[start:], color=BTC_COL, lw=0.9, label='21d corr')
ax2.plot(corr_btc_63.loc[start:], color=GOLD,    lw=1.1, label='63d corr')
ax2.axhline(0, color=GREY, lw=0.8, ls='--')
ax2.axhline(0.3,  color=GREEN, lw=0.5, ls=':')
ax2.axhline(-0.3, color=RED,   lw=0.5, ls=':')
ax2.fill_between(corr_btc_21.loc[start:].index, 0.3, corr_btc_21.loc[start:].clip(0.3),
                 color=GREEN, alpha=0.15, label='Risk-on mode')
ax2.set_title('Gold / BTC Rolling Correlation', fontsize=9, color='#aaa', pad=3)
ax2.legend(fontsize=7)
ax2.set_ylim(-1, 1)

# P3: WTI Oil + momentum
ax3 = fig.add_subplot(gs[1, 1])
ax3.plot(raw['Oil_WTI'].loc[start:], color=ORANGE, lw=1.0, label='WTI Crude')
ax3.plot(oil_ma50.loc[start:], color=GREY, lw=0.7, ls='--', label='MA50')
ax3.fill_between(raw['Oil_WTI'].loc[start:].index,
                 oil_ma50.loc[start:], raw['Oil_WTI'].loc[start:],
                 where=raw['Oil_WTI'].loc[start:] > oil_ma50.loc[start:], color=GREEN, alpha=0.1)
ax3.fill_between(raw['Oil_WTI'].loc[start:].index,
                 oil_ma50.loc[start:], raw['Oil_WTI'].loc[start:],
                 where=raw['Oil_WTI'].loc[start:] < oil_ma50.loc[start:], color=RED, alpha=0.1)
ax3.set_title('WTI Crude  |  Inflation expectations proxy', fontsize=9, color='#aaa', pad=3)
ax3.legend(fontsize=7)
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:.0f}'))

# P4: Gold/Oil rolling correlation
ax4 = fig.add_subplot(gs[2, 0])
ax4.plot(corr_oil_21.loc[start:], color=ORANGE, lw=0.9, label='21d')
ax4.plot(corr_oil_63.loc[start:], color=GOLD,   lw=1.1, label='63d')
ax4.axhline(0, color=GREY, lw=0.8, ls='--')
ax4.set_title('Gold / WTI Correlation  |  >0.3 = inflation theme dominant', fontsize=9, color='#aaa', pad=3)
ax4.legend(fontsize=7)
ax4.set_ylim(-1, 1)

# P5: SGE proxy
ax5 = fig.add_subplot(gs[2, 1])
ax5.plot(sge_proxy_z.loc[start:], color=RED, lw=0.9, label='SGE proxy z-score')
ax5.axhline(0,  color=GREY,  lw=0.8, ls='--')
ax5.axhline(1,  color=GREEN, lw=0.6, ls=':')
ax5.axhline(-1, color=RED,   lw=0.6, ls=':')
ax5.fill_between(sge_proxy_z.loc[start:].index, 1, sge_proxy_z.loc[start:].clip(1),
                 color=GREEN, alpha=0.15, label='China premium')
ax5.fill_between(sge_proxy_z.loc[start:].index, sge_proxy_z.loc[start:].clip(upper=-1), -1,
                 color=RED, alpha=0.15, label='China discount')
ax5.set_title('SGE Premium Proxy (z-score)  |  >1 = physical demand surge', fontsize=9, color='#aaa', pad=3)
ax5.legend(fontsize=7)

# ── Inline display ─────────────────────────────────────
import io as _io, IPython.display as _ipyd
_buf = _io.BytesIO()
plt.savefig(_buf, dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.savefig('gold_wildcards.png', dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
_buf.seek(0)
_ipyd.display(_ipyd.Image(_buf.read()))
plt.close()
print("✓ Saved: gold_wildcards.png")


Additional Research
¶
Beyond the weekly checklist - deeper quant models, kept for reference

In [ ]:
# =============================================================================
# GOLD BAYESIAN REGIME MODEL  |  Quantitative Research
# Author: Internal Quant Strategy Desk
# Version: 2.1.0
# Description: Bayesian Hidden Markov Model for Gold regime detection
#              with dynamic posterior updating and risk-adjusted signals
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import rcParams
import warnings
warnings.filterwarnings('ignore'); unmute_convergence()

# --- Optional: install if needed ---
# !pip install yfinance pymc scipy statsmodels --quiet

import yfinance as yf
from scipy import stats
from scipy.special import logsumexp
from statsmodels.tsa.stattools import adfuller
from statsmodels.regression.rolling import RollingOLS
import statsmodels.api as sm

# ── Plot Aesthetics ───────────────────────────────────────────────────────────
rcParams.update({
    'figure.facecolor':    '#0d0d0d',
    'axes.facecolor':      '#111111',
    'axes.edgecolor':      '#2a2a2a',
    'axes.labelcolor':     '#cccccc',
    'axes.grid':           True,
    'grid.color':          '#1e1e1e',
    'grid.linewidth':      0.6,
    'xtick.color':         '#888888',
    'ytick.color':         '#888888',
    'text.color':          '#cccccc',
    'legend.facecolor':    '#1a1a1a',
    'legend.edgecolor':    '#333333',
    'font.family':         'monospace',
    'font.size':           9,
})

GOLD   = '#C9A84C'
GREEN  = '#2ecc71'
RED    = '#e74c3c'
BLUE   = '#3498db'
PURPLE = '#9b59b6'
GREY   = '#555555'

# =============================================================================
# 1. DATA INGESTION
# =============================================================================

print("=" * 65)
print("  GOLD BAYESIAN REGIME MODEL  |  Quant Research")
print("=" * 65)
print("\n[1/6] Fetching market data ...")

tickers = {
    'GC=F':  'Gold',
    'DX-Y.NYB': 'DXY',
    'TLT':   'TLT',       # Long-duration rates proxy
    '^VIX':  'VIX',
    'GDX':   'GDX',       # Gold miners
}

raw = yf.download(
    list(tickers.keys()),
    start='2010-01-01',
    auto_adjust=True,
    progress=False
)['Close'].rename(columns=tickers)

raw.dropna(how='all', inplace=True)
raw.ffill(inplace=True)
raw.dropna(inplace=True)

# Log returns
rets = np.log(raw / raw.shift(1)).dropna()

gold_ret  = rets['Gold']
gold_px   = raw['Gold']

print(f"    ✓ Loaded {len(gold_px)} trading days  [{gold_px.index[0].date()} → {gold_px.index[-1].date()}]")

# =============================================================================
# 2. FEATURE ENGINEERING
# =============================================================================

print("\n[2/6] Engineering quantitative features ...")

df = pd.DataFrame(index=rets.index)

# Core gold signal
df['ret']          = gold_ret
df['ret_5d']       = gold_ret.rolling(5).sum()
df['ret_21d']      = gold_ret.rolling(21).sum()

# Realised volatility (annualised)
df['rvol_10d']     = gold_ret.rolling(10).std()  * np.sqrt(252)
df['rvol_63d']     = gold_ret.rolling(63).std()  * np.sqrt(252)
df['vol_regime']   = df['rvol_10d'] / df['rvol_63d']   # vol-of-vol ratio

# Momentum & trend
df['mom_1m']       = gold_px.pct_change(21)
df['mom_3m']       = gold_px.pct_change(63)
df['mom_6m']       = gold_px.pct_change(126)
df['sma50']        = gold_px.rolling(50).mean()
df['sma200']       = gold_px.rolling(200).mean()
df['trend_score']  = (gold_px - df['sma200']) / df['sma200']   # % above/below 200d

# RSI
def compute_rsi(series, period=14):
    delta = series.diff()
    gain  = delta.clip(lower=0).rolling(period).mean()
    loss  = (-delta.clip(upper=0)).rolling(period).mean()
    rs    = gain / (loss + 1e-10)
    return 100 - 100 / (1 + rs)

df['rsi']          = compute_rsi(gold_px)
df['rsi_z']        = (df['rsi'] - df['rsi'].rolling(252).mean()) / df['rsi'].rolling(252).std()

# Macro factors
df['dxy_ret']      = rets['DXY']
df['tlt_ret']      = rets['TLT']
df['vix']          = raw['VIX'].reindex(df.index).ffill()
df['vix_z']        = (df['vix'] - df['vix'].rolling(252).mean()) / df['vix'].rolling(252).std()

# Rolling correlation: Gold vs DXY (63-day)
df['corr_dxy']     = gold_ret.rolling(63).corr(rets['DXY'])

# Z-score of gold price (1yr lookback)
df['px_zscore']    = (gold_px - gold_px.rolling(252).mean()) / gold_px.rolling(252).std()
df['px_zscore']    = df['px_zscore'].reindex(df.index)

df.dropna(inplace=True)

print(f"    ✓ {df.shape[1]} features constructed over {len(df)} observations")

# =============================================================================
# 3. BAYESIAN HIDDEN MARKOV MODEL  (3-state: Bear / Neutral / Bull)
# =============================================================================

print("\n[3/6] Fitting Bayesian HMM (Expectation-Maximisation) ...")

class BayesianHMM:
    """
    3-state Gaussian HMM with Bayesian prior regularisation.
    States: 0 = Bear  |  1 = Neutral  |  2 = Bull
    Inference via Baum-Welch (EM) with Dirichlet priors on transitions.
    """

    def __init__(self, n_states=3, n_iter=200, tol=1e-6,
                 dirichlet_prior=1.5, random_state=42):
        self.K          = n_states
        self.n_iter     = n_iter
        self.tol        = tol
        self.alpha      = dirichlet_prior   # Dirichlet concentration
        self.rs         = random_state
        np.random.seed(random_state)

    def _init_params(self, X):
        N, D = X.shape
        # Initialise means via k-means++ style seeding
        idx       = np.random.choice(N)
        means     = [X[idx]]
        for _ in range(self.K - 1):
            dists = np.array([min(np.sum((x - m)**2) for m in means) for x in X])
            probs = dists / dists.sum()
            idx   = np.random.choice(N, p=probs)
            means.append(X[idx])
        self.mu_  = np.array(means)
        self.cov_ = np.array([np.eye(D) * X.var(axis=0).mean() for _ in range(self.K)])
        # Transition matrix — slight diagonal preference
        A = np.ones((self.K, self.K)) * self.alpha
        np.fill_diagonal(A, self.alpha * 5)
        self.A_   = A / A.sum(axis=1, keepdims=True)
        self.pi_  = np.ones(self.K) / self.K

    def _emission_logprob(self, X):
        """Log-likelihood of each obs under each state Gaussian."""
        N, D = X.shape
        log_p = np.zeros((N, self.K))
        for k in range(self.K):
            try:
                mvn = stats.multivariate_normal(mean=self.mu_[k], cov=self.cov_[k],
                                                allow_singular=True)
                log_p[:, k] = mvn.logpdf(X)
            except Exception:
                log_p[:, k] = -1e10
        return log_p

    def _forward(self, log_emit):
        N = log_emit.shape[0]
        log_alpha = np.zeros((N, self.K))
        log_alpha[0] = np.log(self.pi_ + 1e-300) + log_emit[0]
        log_A = np.log(self.A_ + 1e-300)
        for t in range(1, N):
            for k in range(self.K):
                log_alpha[t, k] = logsumexp(log_alpha[t-1] + log_A[:, k]) + log_emit[t, k]
        return log_alpha

    def _backward(self, log_emit):
        N = log_emit.shape[0]
        log_beta = np.zeros((N, self.K))
        log_A = np.log(self.A_ + 1e-300)
        for t in range(N-2, -1, -1):
            for k in range(self.K):
                log_beta[t, k] = logsumexp(log_A[k] + log_emit[t+1] + log_beta[t+1])
        return log_beta

    def _e_step(self, X):
        log_emit  = self._emission_logprob(X)
        log_alpha = self._forward(log_emit)
        log_beta  = self._backward(log_emit)
        log_gamma = log_alpha + log_beta
        log_gamma -= logsumexp(log_gamma, axis=1, keepdims=True)
        gamma = np.exp(log_gamma)

        N = X.shape[0]
        log_A  = np.log(self.A_ + 1e-300)
        xi_sum = np.zeros((self.K, self.K))
        for t in range(N-1):
            xi_t = (log_alpha[t][:, None]
                    + log_A
                    + log_emit[t+1][None, :]
                    + log_beta[t+1][None, :])
            xi_t -= logsumexp(xi_t.ravel())
            xi_sum += np.exp(xi_t)

        ll = logsumexp(log_alpha[-1])
        return gamma, xi_sum, ll

    def _m_step(self, X, gamma, xi_sum):
        N, D = X.shape
        self.pi_ = gamma[0] / gamma[0].sum()
        # Bayesian update of transition matrix (Dirichlet posterior)
        A_num = xi_sum + (self.alpha - 1)
        A_num = np.maximum(A_num, 1e-10)
        self.A_ = A_num / A_num.sum(axis=1, keepdims=True)
        # Update Gaussian parameters
        w = gamma.sum(axis=0) + 1e-10
        self.mu_ = (gamma.T @ X) / w[:, None]
        for k in range(self.K):
            diff = X - self.mu_[k]
            self.cov_[k] = (gamma[:, k][:, None] * diff).T @ diff / w[k]
            self.cov_[k] += np.eye(D) * 1e-4   # regularisation

    def fit(self, X):
        self._init_params(X)
        prev_ll = -np.inf
        for i in range(self.n_iter):
            gamma, xi_sum, ll = self._e_step(X)
            self._m_step(X, gamma, xi_sum)
            if abs(ll - prev_ll) < self.tol:
                print(f"    ✓ Converged at iteration {i+1}  |  log-likelihood = {ll:.2f}")
                break
            prev_ll = ll
        self.gamma_ = gamma
        self.ll_    = ll
        return self

    def predict_proba(self, X):
        log_emit  = self._emission_logprob(X)
        log_alpha = self._forward(log_emit)
        log_beta  = self._backward(log_emit)
        log_gamma = log_alpha + log_beta
        log_gamma -= logsumexp(log_gamma, axis=1, keepdims=True)
        return np.exp(log_gamma)

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)


# Feature matrix for HMM
hmm_features = ['ret', 'rvol_10d', 'mom_1m', 'mom_3m',
                 'trend_score', 'rsi_z', 'vix_z', 'corr_dxy']

X_raw = df[hmm_features].values

# Standardise
X_mean = X_raw.mean(axis=0)
X_std  = X_raw.std(axis=0) + 1e-10
X      = (X_raw - X_mean) / X_std

hmm = BayesianHMM(n_states=3, n_iter=300, dirichlet_prior=2.0)
hmm.fit(X)

proba  = hmm.predict_proba(X)
states = hmm.predict(X)

# --- Label states by mean return (Bear < Neutral < Bull) ---
state_rets = {k: df['ret'].values[states == k].mean() for k in range(3)}
rank       = sorted(state_rets, key=state_rets.get)   # ascending
label_map  = {rank[0]: 'Bear', rank[1]: 'Neutral', rank[2]: 'Bull'}
label_arr  = np.array([label_map[s] for s in states])

prob_bear    = proba[:, rank[0]]
prob_neutral = proba[:, rank[1]]
prob_bull    = proba[:, rank[2]]

df['state']      = label_arr
df['p_bear']     = prob_bear
df['p_neutral']  = prob_neutral
df['p_bull']     = prob_bull

print(f"\n    State distribution:")
for lbl in ['Bull', 'Neutral', 'Bear']:
    pct = (label_arr == lbl).mean() * 100
    avg = df.loc[label_arr == lbl, 'ret'].mean() * 252 * 100
    print(f"      {lbl:>8s}  {pct:5.1f}%  |  Ann. return ~ {avg:+.1f}%")

# =============================================================================
# 4. BAYESIAN SIGNAL GENERATION
# =============================================================================

print("\n[4/6] Computing Bayesian composite signal ...")

# Posterior probability-weighted signal (−1 to +1)
df['raw_signal'] = df['p_bull'] - df['p_bear']

# Overlay macro filter: suppress bull signal when DXY momentum is strongly positive
df['dxy_ma10']   = raw['DXY'].reindex(df.index).ffill().rolling(10).mean()
df['dxy_ma40']   = raw['DXY'].reindex(df.index).ffill().rolling(40).mean()
df['dxy_trend']  = np.sign(df['dxy_ma10'] - df['dxy_ma40'])

df['signal']     = df['raw_signal'] * (1 - 0.3 * df['dxy_trend'].clip(0))   # penalise bull USD

# Clip to [-1, +1]
df['signal']     = df['signal'].clip(-1, 1)

# Volatility-scaled position (target 15% annualised vol)
target_vol       = 0.15
df['pos_raw']    = df['signal'] * (target_vol / (df['rvol_63d'].clip(0.05, 0.8)))
df['position']   = df['pos_raw'].clip(-1, 1).shift(1)   # execute next day

# =============================================================================
# 5. BACKTEST ENGINE
# =============================================================================

print("\n[5/6] Running backtest ...")

TC_BPS = 3   # one-way transaction cost in bps

df['strat_ret']  = df['position'] * df['ret']
df['tc']         = TC_BPS / 10000 * df['position'].diff().abs()
df['strat_ret']  = df['strat_ret'] - df['tc']
df['bh_ret']     = df['ret']

# Cumulative performance
df['strat_cum']  = df['strat_ret'].cumsum().apply(np.exp)
df['bh_cum']     = df['bh_ret'].cumsum().apply(np.exp)

def perf_stats(ret_series, label=''):
    ann_ret  = ret_series.mean() * 252
    ann_vol  = ret_series.std()  * np.sqrt(252)
    sharpe   = ann_ret / (ann_vol + 1e-10)
    cum      = (1 + ret_series).cumprod()
    rolling_max = cum.cummax()
    dd       = (cum - rolling_max) / rolling_max
    max_dd   = dd.min()
    calmar   = ann_ret / (abs(max_dd) + 1e-10)
    hit_rate = (ret_series > 0).mean()
    return {
        'Ann. Return':   f"{ann_ret*100:+.2f}%",
        'Ann. Vol':      f"{ann_vol*100:.2f}%",
        'Sharpe':        f"{sharpe:.3f}",
        'Max Drawdown':  f"{max_dd*100:.2f}%",
        'Calmar':        f"{calmar:.3f}",
        'Hit Rate':      f"{hit_rate*100:.1f}%",
    }

strat_stats = perf_stats(df['strat_ret'], 'Strategy')
bh_stats    = perf_stats(df['bh_ret'],   'Buy & Hold')

print(f"\n    {'Metric':<18}  {'Bayesian HMM':>14}  {'Buy & Hold':>12}")
print(f"    {'-'*48}")
for k in strat_stats:
    print(f"    {k:<18}  {strat_stats[k]:>14}  {bh_stats[k]:>12}")

# =============================================================================
# 6. VISUALISATION  (7-panel dashboard)
# =============================================================================

print("\n[6/6] Rendering dashboard ...")

fig = plt.figure(figsize=(18, 22))
fig.patch.set_facecolor('#0d0d0d')
gs  = gridspec.GridSpec(7, 2, figure=fig, hspace=0.55, wspace=0.3,
                         left=0.06, right=0.97, top=0.95, bottom=0.04)

# ── Header ────────────────────────────────────────────────────────────────────
fig.text(0.06, 0.974, 'GOLD  //  BAYESIAN REGIME MODEL',
         fontsize=14, fontweight='bold', color=GOLD, fontfamily='monospace')
fig.text(0.06, 0.962,
         f"HMM-3 State  |  Vol-Scaled Positioning  |  {df.index[0].date()} – {df.index[-1].date()}",
         fontsize=8, color=GREY, fontfamily='monospace')

# --- Panel 1: Gold price + regime shading ---
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(gold_px.reindex(df.index), color=GOLD, lw=1.1, zorder=5, label='XAU/USD')
ax1.set_title('Gold Price  |  Regime Overlay', fontsize=9, color='#aaaaaa', pad=4)

for i in range(len(df) - 1):
    c = GREEN if df['state'].iloc[i] == 'Bull' else (RED if df['state'].iloc[i] == 'Bear' else '#333333')
    ax1.axvspan(df.index[i], df.index[i+1], alpha=0.12, color=c, linewidth=0)

ax1.set_ylabel('USD / troy oz', fontsize=8)
ax1.legend(loc='upper left', fontsize=8)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# --- Panel 2: Posterior probabilities ---
ax2 = fig.add_subplot(gs[1, :])
ax2.stackplot(df.index,
              df['p_bull'], df['p_neutral'], df['p_bear'],
              colors=[GREEN, GREY, RED], alpha=0.72,
              labels=['P(Bull)', 'P(Neutral)', 'P(Bear)'])
ax2.set_title('Posterior State Probabilities', fontsize=9, color='#aaaaaa', pad=4)
ax2.set_ylabel('Probability', fontsize=8)
ax2.set_ylim(0, 1)
ax2.legend(loc='upper left', fontsize=7, ncol=3)

# --- Panel 3: Composite signal ---
ax3 = fig.add_subplot(gs[2, :])
signal_smooth = df['signal'].rolling(5).mean()
ax3.plot(df.index, signal_smooth, color=BLUE, lw=0.8, label='Signal (5d smooth)')
ax3.fill_between(df.index, 0, signal_smooth,
                 where=signal_smooth > 0, color=GREEN, alpha=0.25)
ax3.fill_between(df.index, 0, signal_smooth,
                 where=signal_smooth < 0, color=RED,   alpha=0.25)
ax3.axhline(0, color=GREY, lw=0.8, ls='--')
ax3.set_title('Bayesian Composite Signal  [−1 Bear → +1 Bull]', fontsize=9, color='#aaaaaa', pad=4)
ax3.set_ylabel('Signal', fontsize=8)
ax3.set_ylim(-1.15, 1.15)
ax3.legend(loc='upper left', fontsize=7)

# --- Panel 4: Vol-scaled position ---
ax4 = fig.add_subplot(gs[3, :])
ax4.plot(df.index, df['position'], color=PURPLE, lw=0.7, label='Position (vol-scaled)')
ax4.fill_between(df.index, 0, df['position'],
                 where=df['position'] > 0, color=GREEN,  alpha=0.2)
ax4.fill_between(df.index, 0, df['position'],
                 where=df['position'] < 0, color=RED,    alpha=0.2)
ax4.axhline(0,  color=GREY, lw=0.7, ls='--')
ax4.axhline(1,  color=GREY, lw=0.4, ls=':')
ax4.axhline(-1, color=GREY, lw=0.4, ls=':')
ax4.set_title('Vol-Scaled Position  [target σ = 15% p.a.]', fontsize=9, color='#aaaaaa', pad=4)
ax4.set_ylabel('Position', fontsize=8)
ax4.legend(loc='upper left', fontsize=7)

# --- Panel 5: Equity curves ---
ax5 = fig.add_subplot(gs[4, :])
ax5.plot(df.index, df['strat_cum'], color=GREEN, lw=1.3, label='Bayesian HMM Strategy')
ax5.plot(df.index, df['bh_cum'],   color=GOLD,  lw=1.0, ls='--', label='Buy & Hold Gold')
ax5.set_title('Cumulative Performance  (log-returns, net of 3bps TC)', fontsize=9, color='#aaaaaa', pad=4)
ax5.set_ylabel('Growth of $1', fontsize=8)
ax5.legend(loc='upper left', fontsize=8)
ax5.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:.2f}'))

# --- Panel 6L: Return distribution by regime ---
ax6 = fig.add_subplot(gs[5, 0])
for lbl, col in [('Bull', GREEN), ('Neutral', GREY), ('Bear', RED)]:
    sub = df.loc[df['state'] == lbl, 'ret'] * 100
    sub.hist(ax=ax6, bins=60, alpha=0.5, color=col, label=lbl, density=True)
    mu, sd = sub.mean(), sub.std()
    xs = np.linspace(sub.min(), sub.max(), 200)
    ax6.plot(xs, stats.norm.pdf(xs, mu, sd), color=col, lw=1.2)
ax6.set_title('Daily Return Distribution by Regime', fontsize=9, color='#aaaaaa', pad=4)
ax6.set_xlabel('Daily Return (%)', fontsize=8)
ax6.legend(fontsize=7)

# --- Panel 6R: Rolling Sharpe ---
ax7 = fig.add_subplot(gs[5, 1])
roll_sh = (df['strat_ret'].rolling(252).mean() /
           (df['strat_ret'].rolling(252).std() + 1e-10)) * np.sqrt(252)
ax7.plot(df.index, roll_sh, color=BLUE, lw=0.9)
ax7.axhline(0, color=GREY, lw=0.7, ls='--')
ax7.axhline(1, color=GREEN, lw=0.4, ls=':')
ax7.fill_between(df.index, 0, roll_sh, where=roll_sh > 0, color=GREEN, alpha=0.15)
ax7.fill_between(df.index, 0, roll_sh, where=roll_sh < 0, color=RED,   alpha=0.15)
ax7.set_title('Rolling 1Y Sharpe Ratio', fontsize=9, color='#aaaaaa', pad=4)
ax7.set_ylabel('Sharpe', fontsize=8)

# --- Panel 7: Stats table ---
ax8 = fig.add_subplot(gs[6, :])
ax8.axis('off')

table_data = [[k, strat_stats[k], bh_stats[k]] for k in strat_stats]
col_labels  = ['Metric', 'Bayesian HMM', 'Buy & Hold']

tbl = ax8.table(
    cellText=table_data, colLabels=col_labels,
    loc='center', cellLoc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.8)

for (row, col), cell in tbl.get_celld().items():
    cell.set_facecolor('#1a1a1a' if row % 2 == 0 else '#111111')
    cell.set_edgecolor('#2a2a2a')
    cell.set_text_props(color='#cccccc', fontfamily='monospace')
    if row == 0:
        cell.set_facecolor('#0a0a0a')
        cell.set_text_props(color=GOLD, fontweight='bold')

import io as _io, IPython.display as _ipyd
_buf = _io.BytesIO()
plt.savefig(_buf, dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
plt.savefig('gold_bayesian_hmm.png', dpi=150, bbox_inches='tight',
            facecolor='#0d0d0d')
_buf.seek(0)
_ipyd.display(_ipyd.Image(_buf.read()))
plt.close()

print("\n" + "=" * 65)
print("  Run complete.  Chart saved → gold_bayesian_hmm.png")
print("=" * 65)


In [ ]:
# =============================================================================
# DAILY PRE-MARKET CHECK  |  Run this every morning before open
# Takes ~10 seconds. Only rerun full dashboard if flagged below.
# =============================================================================

import yfinance as yf
import pandas as pd
from datetime import datetime

def fetch(ticker, period='5d'):
    try:
        t = yf.Ticker(ticker)
        h = t.history(period=period)
        if not h.empty:
            return safe_at(h['Close'], -1)
    except:
        pass
    return None

print("=" * 62)
print(f"  DAILY PRE-MARKET CHECK  |  {datetime.now().strftime('%A %d %b %Y  %H:%M')}")
print("=" * 62)

# ── Prices ────────────────────────────────────────────────────────────────────
gld       = fetch('GLD')
gold      = fetch('GC=F') or fetch('GC=F', '1mo')
vix       = fetch('^VIX')
gvz       = fetch('^GVZ')
irx       = fetch('^IRX')

# Baseline values from your Sunday analysis — update these each Sunday
BASELINE_GOLD = 4730.70
BASELINE_GLD  = 433.77
BASELINE_VIX  = 17.19    # Friday close VIX
BASELINE_GVZ  = 26.5

ratio = gold / gld if gold and gld else 10.906

print(f"\n  {'Instrument':<22}  {'Now':>10}  {'Sunday base':>12}  {'Δ':>10}")
print(f"  {'─' * 58}")

def row(label, now, base, fmt='.2f'):
    if now is None:
        print(f"  {label:<22}  {'N/A':>10}  {base:>12{fmt}}  {'?':>10}")
        return
    delta = now - base
    arrow = '▲' if delta > 0 else '▼' if delta < 0 else '–'
    now_str  = format(now,        fmt)
    base_str = format(base,       fmt)
    diff_str = format(abs(delta), fmt)
    print(f"  {label:<22}  {now_str:>10}  {base_str:>12}  {arrow}{diff_str:>8}")

row('Gold futures (GC=F)', gold,  BASELINE_GOLD, '.2f')
row('GLD spot',            gld,   BASELINE_GLD,  '.2f')
row('VIX',                 vix,   BASELINE_VIX,  '.2f')
row('GVZ (gold vol)',      gvz,   BASELINE_GVZ,  '.1f')
if irx:
    print(f"  {'Risk-free (^IRX)':<22}  {irx:.2f}%")

# ── Regime check ──────────────────────────────────────────────────────────────
FLIP_GOLD = 3562   # your zero-gamma flip level in gold — update each Sunday

print(f"\n  ── Regime ───────────────────────────────────────────────")
if gold:
    dist_from_flip = gold - FLIP_GOLD
    if gold > FLIP_GOLD:
        print(f"  Gold ${gold:,.0f} is ${dist_from_flip:,.0f} ABOVE flip (${FLIP_GOLD:,})")
        print(f"  → Suppressed regime intact — fade extremes, don't chase")
    else:
        print(f"  Gold ${gold:,.0f} is ${abs(dist_from_flip):,.0f} BELOW flip (${FLIP_GOLD:,})")
        print(f"  ⚠ REGIME FLIPPED → amplified moves — ride breakouts")

# ── VIX signal ────────────────────────────────────────────────────────────────
NET_VANNA = 12727.0   # from your Sunday run — update each Sunday

print(f"\n  ── Vanna/VIX signal (net vanna {NET_VANNA:+,.0f}K) ─────────────")
if vix and BASELINE_VIX:
    vix_delta = vix - BASELINE_VIX
    dealer_flow = NET_VANNA * (-vix_delta / BASELINE_GVZ) * 0.01
    if vix_delta < -1:
        print(f"  VIX DOWN {abs(vix_delta):.1f} pts → dealers forced to BUY ~{dealer_flow:+,.0f}K delta")
        print(f"  → Bullish tailwind at open")
    elif vix_delta > 1:
        print(f"  VIX UP {vix_delta:.1f} pts → dealers forced to SELL ~{dealer_flow:+,.0f}K delta")
        print(f"  ⚠ Bearish headwind at open")
    else:
        print(f"  VIX roughly flat ({vix_delta:+.1f}) → dealer flow neutral")

# ── Gold move check ───────────────────────────────────────────────────────────
print(f"\n  ── Gold move vs Sunday ──────────────────────────────────")
if gold:
    move = gold - BASELINE_GOLD
    move_pct = (move / BASELINE_GOLD) * 100
    print(f"  Gold moved {move:+,.0f} ({move_pct:+.1f}%) since Sunday close")
    if abs(move) > 200:
        print(f"  ⚠ LARGE MOVE — rerun full dashboard, levels may be stale")
    elif abs(move) > 100:
        print(f"  △ Moderate move — levels still valid but watch boundaries")
    else:
        print(f"  ✓ Small move — Sunday levels still valid")

# ── Key levels reminder ───────────────────────────────────────────────────────
print(f"""
  ── Key levels (from Sunday) ─────────────────────────────
  RESISTANCE  $4,900  (GLD $450)  ← take-profit long / fade zone
              $4,792  (GLD $440)  ← first ceiling
              $4,737  (GLD $435)  ← immediate cap above spot

  SUPPORT     $4,628  (GLD $425)  ← primary long entry  ★
              $4,519  (GLD $415)  ← mid support
              $4,465  (GLD $410)  ← deep support / add zone

  FLIP LEVEL  $3,562              ← regime changes below this
""")

# ── Rerun trigger ─────────────────────────────────────────────────────────────
print("  ── Do I need to rerun the full dashboard? ───────────────")
rerun = False
reasons = []

if gold and abs(gold - BASELINE_GOLD) > 200:
    rerun = True
    reasons.append(f"Gold moved ${abs(gold - BASELINE_GOLD):,.0f} (>$200 threshold)")
if gvz and gvz > 30:
    rerun = True
    reasons.append(f"GVZ at {gvz:.1f}% — above 30% suppressed-regime threshold")
if gvz and abs(gvz - BASELINE_GVZ) > 4:
    rerun = True
    reasons.append(f"GVZ shifted {abs(gvz - BASELINE_GVZ):.1f}pts — vol regime may have changed")

# Check if today is Friday (expiration day warning)
if datetime.now().weekday() == 4:
    rerun = True
    reasons.append("It's Friday — options expiry today will shift open interest")

if rerun:
    print(f"  ⚠ YES — rerun full dashboard. Reasons:")
    for r in reasons:
        print(f"    · {r}")
else:
    print(f"  ✓ NO — Sunday analysis still valid. Use existing levels.")

print("\n" + "=" * 62)


In [ ]:
# =============================================================================
# LIVE VIX MONITOR  |  Run anytime before/during session to check dealer bias
# =============================================================================

import yfinance as yf
import pandas as pd
from datetime import datetime

# ── Your Sunday baseline values — update each Sunday ─────────────────────────
BASELINE_VIX  = 17.19
BASELINE_GVZ  = 26.5
NET_VANNA     = 12727.0   # from cell 47 output

# ── Your key levels — update each Sunday ─────────────────────────────────────
LEVELS = {
    'RESISTANCE': [
        (4900, 450, 'take-profit / fade zone'),
        (4792, 440, 'first ceiling'),
        (4737, 435, 'immediate cap above spot'),
    ],
    'SUPPORT': [
        (4628, 425, 'PRIMARY entry  ★  score 1.51'),
        (4519, 415, 'mid support'),
        (4465, 410, 'deep support — spike scenario only'),
    ]
}

# ── Fetch ─────────────────────────────────────────────────────────────────────
def fetch(ticker):
    try:
        t = yf.Ticker(ticker)
        h = t.history(period='5d', interval='1m')
        if not h.empty:
            return safe_at(h['Close'], -1)
    except:
        pass
    return None

vix = fetch('^VIX')
gvz = fetch('^GVZ')

now = datetime.now().strftime('%A %d %b %Y  %H:%M:%S')

# ── Signal logic ──────────────────────────────────────────────────────────────
def get_signal(vix, baseline, net_vanna, gvz_base):
    delta     = vix - baseline
    flow      = net_vanna * (-delta / gvz_base) * 0.01
    pct       = (delta / baseline) * 100

    if delta < -2:
        regime  = 'BULLISH TAILWIND'
        action  = 'Dealers forced to BUY'
        entry   = 'Enter long near $4,628  ★'
        stop    = 'Stop below $4,465'
        target  = 'Target $4,792 → $4,900'
        color   = '✅'
    elif delta < -1:
        regime  = 'MILD BULLISH LEAN'
        action  = 'Dealers buying modestly'
        entry   = 'Wait for dip to $4,628 before entering'
        stop    = 'Stop below $4,465'
        target  = 'Target $4,792'
        color   = '🟩'
    elif delta > 2.5:
        regime  = '⚠  VIX SPIKE — PLAYBOOK SHIFTS'
        action  = 'Dealers forced to SELL'
        entry   = 'DO NOT enter long yet — watch $4,465'
        stop    = 'Wait for VIX to stabilize first'
        target  = 'Re-evaluate once VIX flattens'
        color   = '🔴'
    elif delta > 1:
        regime  = 'CAUTION — LET DIP DEVELOP'
        action  = 'Dealers selling, headwind at open'
        entry   = 'Wait for $4,519–$4,628 and stabilization'
        stop    = 'Stop below $4,465'
        target  = 'Target $4,737 once stable'
        color   = '🟡'
    else:
        regime  = 'NEUTRAL — STRUCTURAL BULL BIAS'
        action  = 'Dealer flow near zero'
        entry   = 'Use $4,628 as normal entry zone'
        stop    = 'Stop below $4,465'
        target  = 'Target $4,792 → $4,900'
        color   = '⬜'

    return delta, flow, pct, regime, action, entry, stop, target, color

# ── Print ─────────────────────────────────────────────────────────────────────
print("=" * 62)
print(f"  LIVE VIX CHECK  |  {now}")
print("=" * 62)

if vix is None:
    print("\n  ⚠ Could not fetch VIX — check connection or market hours")
else:
    delta, flow, pct, regime, action, entry, stop, target, color = \
        get_signal(vix, BASELINE_VIX, NET_VANNA, BASELINE_GVZ)

    arrow = '▲' if delta > 0 else '▼' if delta < 0 else '–'

    print(f"\n  VIX now       : {vix:.2f}")
    print(f"  Baseline      : {BASELINE_VIX:.2f}  (Friday close)")
    print(f"  Change        : {arrow} {abs(delta):.2f} pts  ({pct:+.1f}%)")
    if gvz:
        print(f"  GVZ now       : {gvz:.1f}%  (gold implied vol)")
    print(f"  Dealer flow   : {flow:+,.0f}K delta  ({'BUY' if flow > 0 else 'SELL'} pressure)")

    print(f"\n  {color}  {regime}")
    print(f"\n  Action        : {action}")
    print(f"  Entry         : {entry}")
    print(f"  Stop          : {stop}")
    print(f"  Target        : {target}")

# ── Levels ────────────────────────────────────────────────────────────────────
print(f"\n  {'─' * 58}")
print(f"  {'RESISTANCE LEVELS':}")
for gold, gld, note in LEVELS['RESISTANCE']:
    print(f"    Gold ${gold:,}  (GLD ${gld})  —  {note}")

print(f"\n  {'SUPPORT LEVELS':}")
for gold, gld, note in LEVELS['SUPPORT']:
    marker = '  ★' if 'PRIMARY' in note else ''
    print(f"    Gold ${gold:,}  (GLD ${gld})  —  {note}{marker}")

# ── VIX scenario table ────────────────────────────────────────────────────────
print(f"\n  {'─' * 58}")
print(f"  VIX scenario reference  (net vanna {NET_VANNA:+,.0f}K)")
print(f"  {'ΔVIX':>8}  {'Dealer flow':>14}  {'Action':>10}")
print(f"  {'─' * 38}")
for dv in [-5, -3, -1, +1, +3, +5]:
    f = NET_VANNA * (-dv / BASELINE_GVZ) * 0.01
    direction = 'BUY ' if f > 0 else 'SELL'
    print(f"  {dv:>+8.0f}  {f:>+12,.0f}K  dealers {direction}")

print(f"\n{'=' * 62}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import warnings
warnings.filterwarnings('ignore'); unmute_convergence()

# =============================================================================
# REAL YIELDS DASHBOARD
# TIP ETF (inflation expectations) vs TNX (10Y nominal) → real yield proxy
# =============================================================================

BG     = '#080810'
GOLD   = '#C9A84C'
GOLD2  = '#FFD700'
RED2   = '#FF4444'
BLUE2  = '#4FC3F7'
GREEN  = '#00C853'

print("Fetching real yield data...")

tickers = {
    'GC=F'  : 'Gold',
    'TIP'   : 'TIP (inflation expectations)',
    '^TNX'  : '10Y Nominal Yield',
    'UUP'   : 'DXY Proxy',
    '^IRX'  : '3M T-Bill',
}

raw = {}
for t, label in tickers.items():
    try:
        s = yf.download(t, period='2y', auto_adjust=True, progress=False)['Close'].squeeze()
        if len(s) > 0:
            raw[label] = s
            print(f"  ✓ {label}")
        else:
            print(f"  ✗ {label} — no data")
    except Exception as e:
        print(f"  ✗ {label} — {e}")

df = pd.DataFrame(raw).ffill().dropna()

# --- Real yield proxy: TNX minus TIP rolling return annualised
tip_ret       = np.log(df['TIP (inflation expectations)'] / df['TIP (inflation expectations)'].shift(1))
tip_ann       = tip_ret.rolling(63).mean() * 252 * 100   # annualised TIP return as inflation proxy
nominal_yield = df['10Y Nominal Yield']
real_yield    = nominal_yield - tip_ann                   # simplified real yield proxy

# --- TIP/Gold correlation
gold_ret = np.log(df['Gold'] / df['Gold'].shift(1))
tip_corr = gold_ret.rolling(63).corr(tip_ret)

# --- Yield curve
curve = df['10Y Nominal Yield'] - df['3M T-Bill'] / 100  # TNX is already in %, IRX needs /100? check
# IRX is in annualised %, TNX is in %, both comparable directly
curve2 = df['10Y Nominal Yield'] - df['3M T-Bill']

print(f"\n  Latest readings:")
print(f"  10Y Nominal Yield : {nominal_yield.iloc[-1]:.2f}%")
print(f"  TIP ann. return   : {tip_ann.iloc[-1]:.2f}%")
print(f"  Real Yield Proxy  : {real_yield.iloc[-1]:.2f}%")
print(f"  Gold/TIP 63d corr : {tip_corr.iloc[-1]:.3f}")

# =============================================================================
# PLOT
# =============================================================================
fig, axes = plt.subplots(4, 1, figsize=(16, 14), facecolor=BG)
fig.patch.set_facecolor(BG)
fig.suptitle('REAL YIELDS DASHBOARD  //  Gold · TIP · TNX · DXY',
             fontsize=13, color=GOLD, fontweight='bold', fontfamily='monospace', y=0.98)

dates = df.index

# Panel 1 — Gold price
ax = axes[0]
ax.set_facecolor('#0a0a14')
ax.plot(dates, df['Gold'], color=GOLD2, lw=1.2)
ax.set_title('Gold Price (GC=F)', color=GOLD, fontsize=9, fontfamily='monospace', pad=3)
ax.tick_params(colors='#555', labelsize=7)
for sp in ax.spines.values(): sp.set_color('#222')
ax.grid(True, color='#111', lw=0.4)
ax.yaxis.label.set_color('#555')

# Panel 2 — Nominal vs Real yield
ax = axes[1]
ax.set_facecolor('#0a0a14')
ax.plot(dates, nominal_yield, color=RED2,   lw=1.0, label='10Y Nominal')
ax.plot(dates, real_yield,    color=BLUE2,  lw=1.0, label='Real Yield Proxy')
ax.axhline(0, color='#444', lw=0.6, ls='--')
ax.fill_between(dates, real_yield, 0,
                where=real_yield < 0, color=GREEN, alpha=0.15, label='Negative real yield (bullish gold)')
ax.fill_between(dates, real_yield, 0,
                where=real_yield > 0, color=RED2,  alpha=0.10, label='Positive real yield (headwind)')
latest_real = real_yield.iloc[-1]
color_real  = GREEN if latest_real < 0 else RED2
ax.set_title(f'Real Yield Proxy  |  latest = {latest_real:.2f}%  '
             f'{"▼ BULLISH for gold" if latest_real < 0 else "▲ HEADWIND for gold"}',
             color=color_real, fontsize=9, fontfamily='monospace', pad=3)
ax.legend(fontsize=7, facecolor='#0d0d1a', edgecolor='#333', labelcolor='white')
ax.tick_params(colors='#555', labelsize=7)
for sp in ax.spines.values(): sp.set_color('#222')
ax.grid(True, color='#111', lw=0.4)

# Panel 3 — Gold / TIP correlation
ax = axes[2]
ax.set_facecolor('#0a0a14')
ax.plot(dates, tip_corr, color=GOLD2, lw=1.0)
ax.axhline(0,    color='#444', lw=0.6, ls='--')
ax.axhline(0.3,  color=GREEN,  lw=0.5, ls=':', alpha=0.7)
ax.axhline(-0.3, color=RED2,   lw=0.5, ls=':', alpha=0.7)
ax.fill_between(dates, tip_corr, 0,
                where=tip_corr > 0, color=GREEN, alpha=0.15)
ax.fill_between(dates, tip_corr, 0,
                where=tip_corr < 0, color=RED2,  alpha=0.15)
latest_corr = tip_corr.iloc[-1]
ax.set_title(f'Gold / TIP 63d Rolling Correlation  |  latest = {latest_corr:.3f}  '
             f'(+ve = inflation trade on / -ve = safe haven only)',
             color=GOLD, fontsize=9, fontfamily='monospace', pad=3)
ax.set_ylim(-1, 1)
ax.tick_params(colors='#555', labelsize=7)
for sp in ax.spines.values(): sp.set_color('#222')
ax.grid(True, color='#111', lw=0.4)

# Panel 4 — DXY proxy
ax = axes[3]
ax.set_facecolor('#0a0a14')
dxy_s = df['DXY Proxy']
ax.plot(dates, dxy_s, color=BLUE2, lw=1.0)
ax.fill_between(dates, dxy_s, dxy_s.mean(),
                where=dxy_s < dxy_s.mean(), color=GREEN, alpha=0.15, label='Below mean (gold bullish)')
ax.fill_between(dates, dxy_s, dxy_s.mean(),
                where=dxy_s > dxy_s.mean(), color=RED2,  alpha=0.10, label='Above mean (gold headwind)')
ax.axhline(dxy_s.mean(), color='#444', lw=0.6, ls='--')
ax.set_title(f'DXY Proxy (UUP)  |  latest = {dxy_s.iloc[-1]:.2f}  mean = {dxy_s.mean():.2f}',
             color=BLUE2, fontsize=9, fontfamily='monospace', pad=3)
ax.legend(fontsize=7, facecolor='#0d0d1a', edgecolor='#333', labelcolor='white')
ax.tick_params(colors='#555', labelsize=7)
for sp in ax.spines.values(): sp.set_color('#222')
ax.grid(True, color='#111', lw=0.4)

plt.tight_layout()
import io as _io, IPython.display as _ipyd
_buf = _io.BytesIO()
plt.savefig(_buf, dpi=150, bbox_inches="tight")
_buf.seek(0)
_ipyd.display(_ipyd.Image(_buf.read()))
plt.close()

print("\n" + "="*60)
print(f"  REAL YIELD SUMMARY")
print(f"  Nominal 10Y     : {nominal_yield.iloc[-1]:.2f}%")
print(f"  Real Yield Proxy: {real_yield.iloc[-1]:.2f}%  {'← BULLISH' if real_yield.iloc[-1] < 0 else '← HEADWIND'}")
print(f"  Gold/TIP corr   : {tip_corr.iloc[-1]:.3f}  {'← inflation trade' if tip_corr.iloc[-1] > 0.3 else '← safe haven only' if tip_corr.iloc[-1] < -0.3 else '← mixed'}")
print(f"  DXY (UUP)       : {dxy_s.iloc[-1]:.2f}  {'← bullish' if dxy_s.iloc[-1] < dxy_s.mean() else '← headwind'}")
print("="*60)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import yfinance as yf
import warnings
warnings.filterwarnings('ignore'); unmute_convergence()

# =============================================================================
# GOLD VOLUME HEATMAP  //  Quant Flow Attribution Dashboard
# =============================================================================

BG    = '#080810'
GOLD  = '#C9A84C'
GOLD2 = '#FFD700'
RED2  = '#FF4444'
BLUE2 = '#4FC3F7'
GREEN = '#00C853'
PURP  = '#B388FF'
ORNG  = '#FF9800'

print("="*60)
print("  GOLD FLOW ATTRIBUTION HEATMAP  |  QRD")
print("="*60)

# =============================================================================
# 1. MARKET DATA  (robust fetch — handles all yfinance version quirks)
# =============================================================================
print("\n[1/4] Fetching market data...")

import yfinance as yf
import pandas as pd
import numpy as np

tickers = {
    'GC=F'   : 'Gold',
    'GLD'    : 'GLD ETF',
    'IAU'    : 'IAU ETF',
    'UUP'    : 'DXY',
    'TIP'    : 'TIP',
    '^VIX'   : 'VIX',
    'GDX'    : 'Gold Miners',
    'SPY'    : 'SPY',
    'BTC-USD': 'BTC',
}

vol_tickers = {
    'GLD': 'GLD_vol',
    'IAU': 'IAU_vol',
    'GDX': 'GDX_vol',
}

def fetch_single(ticker, col='Close', period='1y'):
    """
    Bulletproof single-ticker fetch.
    Handles MultiIndex, tuple columns, scalar squeeze.
    """
    try:
        raw_df = yf.download(
            ticker,
            period=period,
            auto_adjust=True,
            progress=False,
            group_by='ticker'   # force flat columns
        )

        # --- flatten whatever column structure yfinance returned
        if hasattr(raw_df, 'columns') and isinstance(raw_df.columns, pd.MultiIndex):
            # e.g. ('Close', 'GC=F') or ('GC=F', 'Close')
            flat = []
            for c in raw_df.columns:
                parts = [str(p) for p in c if str(p) not in ('', ticker, ticker.replace('=',''))]
                flat.append(parts[0] if parts else str(c[0]))
            raw_df.columns = flat

        # now columns should be strings like 'Close','Open','Volume',...
        if col not in raw_df.columns:
            # try case-insensitive match
            match = [c for c in raw_df.columns if c.lower() == col.lower()]
            if not match:
                print(f"    ✗ {ticker}: column '{col}' not found — available: {list(raw_df.columns)}")
                return None
            col = match[0]

        s = raw_df[col].copy()

        # squeeze in case it's still a DataFrame
        if isinstance(s, pd.DataFrame):
            s = s.iloc[:, 0]

        s = s.ffill().dropna()

        if len(s) < 10:
            print(f"    ✗ {ticker}: too few rows ({len(s)})")
            return None

        return s

    except Exception as e:
        print(f"    ✗ {ticker}: {e}")
        return None

# --- fetch prices
raw = {}
for ticker, label in tickers.items():
    s = fetch_single(ticker, col='Close', period='1y')
    if s is not None:
        raw[label] = s
        print(f"  ✓ {label}  ({len(s)} rows)")
    else:
        print(f"  ✗ {label}")

# --- fetch volumes separately
vol_raw = {}
for ticker, label in vol_tickers.items():
    s = fetch_single(ticker, col='Volume', period='1y')
    if s is not None:
        vol_raw[label] = s

# --- build DataFrames WITHOUT align (align was destroying data)
prices = pd.DataFrame(raw).ffill().dropna()
vols   = pd.DataFrame(vol_raw).ffill()

# --- align index manually — keep only dates in prices
vols = vols.reindex(prices.index).ffill().fillna(0)

print(f"\n  prices shape : {prices.shape}")
print(f"  prices cols  : {list(prices.columns)}")
print(f"  vols cols    : {list(vols.columns)}")

# --- guard
required = ['Gold','GLD ETF','IAU ETF','DXY','TIP',
            'VIX','Gold Miners','SPY','BTC']
missing  = [c for c in required if c not in prices.columns]
if missing:
    print(f"\n  ⚠ Still missing: {missing}")
    print("  Check your internet connection or yfinance version")
    raise RuntimeError(f"Cannot continue — missing: {missing}")
else:
    print(f"\n  ✓ All columns present  |  "
          f"{prices.index[0].date()} → {prices.index[-1].date()}")

# =============================================================================
# 2. FLOW PROXIES
# =============================================================================
print("\n[2/4] Computing flow proxies...")

gold_ret = np.log(prices['Gold']        / prices['Gold'].shift(1))
gld_ret  = np.log(prices['GLD ETF']     / prices['GLD ETF'].shift(1))
dxy_ret  = np.log(prices['DXY']         / prices['DXY'].shift(1))
tip_ret  = np.log(prices['TIP']         / prices['TIP'].shift(1))
vix_chg  = prices['VIX'].diff()
gdx_ret  = np.log(prices['Gold Miners'] / prices['Gold Miners'].shift(1))
spy_ret  = np.log(prices['SPY']         / prices['SPY'].shift(1))
btc_ret  = np.log(prices['BTC']         / prices['BTC'].shift(1))

# Rolling 21d correlations with gold
W = 21
corr_dxy = gold_ret.rolling(W).corr(dxy_ret)
corr_tip = gold_ret.rolling(W).corr(tip_ret)
corr_vix = gold_ret.rolling(W).corr(vix_chg)
corr_gdx = gold_ret.rolling(W).corr(gdx_ret)
corr_spy = gold_ret.rolling(W).corr(spy_ret)
corr_btc = gold_ret.rolling(W).corr(btc_ret)

# ETF dollar volume
gld_dv = (prices['GLD ETF']     * vols['GLD_vol']).fillna(0)
iau_dv = (prices['IAU ETF']     * vols['IAU_vol']).fillna(0)
gdx_dv = (prices['Gold Miners'] * vols['GDX_vol']).fillna(0)

total_etf_flow = gld_dv + iau_dv
etf_share      = total_etf_flow / (total_etf_flow + gdx_dv + 1)

# Realised vol
rvol_5  = gold_ret.rolling(5).std()  * np.sqrt(252)
rvol_21 = gold_ret.rolling(21).std() * np.sqrt(252)

# Signed volume / buy-sell pressure
signed_vol    = gold_ret * gld_dv.reindex(gold_ret.index).fillna(0)
buy_pressure  = signed_vol.clip(lower=0).rolling(21).mean()
sell_pressure = signed_vol.clip(upper=0).rolling(21).mean().abs()
net_pressure  = buy_pressure / (buy_pressure + sell_pressure + 1e-10)

# CFTC snapshot — May 26 2026
cftc_snapshot = {
    'Swap Dealers (Banks)'   : -159822,   # 29746 - 189568
    'Managed Money (HFs)'    :  +96931,   # 124534 - 27603
    'Producer/Merchant'      :  -22707,   # 15824 - 38531
    'Other Reportables'      :  +52729,   # 72463 - 19734
    'Non-Reportable (Retail)':  +32870,   # 51882 - 19012
}
total_abs = sum(abs(v) for v in cftc_snapshot.values())
cftc_pct  = {k: v / total_abs * 100 for k, v in cftc_snapshot.items()}

# Factor beta decomposition
factors = pd.DataFrame({
    'DXY (inverse)'     : -dxy_ret,
    'Real Yield (TIP)'  :  tip_ret,
    'Fear (VIX)'        :  vix_chg / 10,
    'Miners (GDX)'      :  gdx_ret,
    'Risk-off (SPY-inv)': -spy_ret,
    'Crypto (BTC)'      :  btc_ret,
}).dropna()

gold_r_aligned = gold_ret.reindex(factors.index).dropna()
factors        = factors.reindex(gold_r_aligned.index)

betas_ts = pd.DataFrame(index=gold_r_aligned.index,
                         columns=factors.columns, dtype=float)
for i in range(63, len(gold_r_aligned)):
    y = gold_r_aligned.iloc[i-63:i].values
    X = factors.iloc[i-63:i].values
    try:
        b = np.linalg.lstsq(
            np.column_stack([np.ones(len(y)), X]),
            y, rcond=None)[0][1:]
        betas_ts.iloc[i] = b
    except:
        pass

betas_ts = betas_ts.dropna().astype(float)
print(f"  ✓ Flow proxies computed")

# =============================================================================
# 3. HEATMAP DATA
# =============================================================================
print("\n[3/4] Building heatmap matrices...")

N_MONTHS   = 12
month_ends = pd.date_range(
    start=prices.index[-N_MONTHS * 21],
    periods=N_MONTHS, freq='21D')

heat_labels  = ['DXY (inv)', 'TIP/Inflation', 'VIX/Fear',
                'GDX/Miners', 'SPY (inv)', 'BTC/Crypto']
heat_matrix  = np.zeros((len(heat_labels), N_MONTHS))
month_labels = [d.strftime('%b %y') for d in month_ends]

for j, end in enumerate(month_ends):
    start = end - pd.Timedelta(days=21)
    mask  = (gold_ret.index >= start) & (gold_ret.index <= end)
    gr    = gold_ret[mask]
    if len(gr) < 5:
        continue
    drivers = [-dxy_ret, tip_ret, vix_chg / 10,
               gdx_ret, -spy_ret, btc_ret]
    for i, d in enumerate(drivers):
        dm = d.reindex(gr.index)
        if dm.std() > 1e-10 and gr.std() > 1e-10:
            heat_matrix[i, j] = float(gr.corr(dm))

latest_betas = betas_ts.iloc[-1].values if len(betas_ts) > 0 else np.zeros(6)
beta_abs     = np.abs(latest_betas)
beta_pct     = beta_abs / (beta_abs.sum() + 1e-10) * 100

flow_dates = net_pressure.dropna().index
flow_vals  = net_pressure.dropna().values

# =============================================================================
# 4. PLOT
# =============================================================================
print("\n[4/4] Rendering dashboard...")

fig = plt.figure(figsize=(22, 16), facecolor=BG)
fig.patch.set_facecolor(BG)
gs  = gridspec.GridSpec(3, 3, figure=fig,
                         left=0.06, right=0.97,
                         top=0.93, bottom=0.06,
                         hspace=0.55, wspace=0.4)

fig.suptitle('GOLD  //  FLOW ATTRIBUTION & VOLUME HEATMAP  |  QRD',
             fontsize=14, color=GOLD, fontweight='bold',
             fontfamily='monospace', y=0.97)

cmap_rg  = LinearSegmentedColormap.from_list(
    'rg',   ['#5C0000', '#111122', '#004d00'], N=256)
cmap_gold= LinearSegmentedColormap.from_list(
    'gold', ['#0a0a14', '#7B5E00', '#FFD700'], N=256)

def style_ax(ax):
    ax.set_facecolor('#0a0a14')
    ax.tick_params(colors='#666', labelsize=7)
    for sp in ax.spines.values():
        sp.set_color('#222')
    ax.grid(True, color='#111', lw=0.3)

# ── Panel 1: Rolling Driver Correlation Heatmap ───────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
ax1.set_facecolor('#0a0a14')

im1 = ax1.imshow(heat_matrix, aspect='auto', cmap=cmap_rg,
                  vmin=-1, vmax=1, interpolation='nearest')
ax1.set_xticks(range(N_MONTHS))
ax1.set_xticklabels(month_labels, fontsize=7, color='#888',
                     fontfamily='monospace', rotation=30)
ax1.set_yticks(range(len(heat_labels)))
ax1.set_yticklabels(heat_labels, fontsize=8,
                     color=GOLD2, fontfamily='monospace')
ax1.set_title('Rolling 21-Day Driver Correlation with Gold  |  '
              'Green=positive  Red=negative',
              color=GOLD, fontsize=9, fontfamily='monospace', pad=4)
for i in range(len(heat_labels)):
    for j in range(N_MONTHS):
        val = heat_matrix[i, j]
        col = 'white' if abs(val) > 0.3 else '#555'
        ax1.text(j, i, f'{val:.2f}', ha='center', va='center',
                 fontsize=6.5, color=col, fontfamily='monospace')
plt.colorbar(im1, ax=ax1, fraction=0.02, pad=0.02).ax.tick_params(
    colors='#666', labelsize=7)

# ── Panel 2: CFTC Flow Bar ────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
ax2.set_facecolor('#0a0a14')
for sp in ax2.spines.values(): sp.set_color('#222')

labels_cftc = list(cftc_snapshot.keys())
values_cftc = list(cftc_snapshot.values())
colors_cftc = [RED2 if v < 0 else GREEN for v in values_cftc]
bars = ax2.barh(labels_cftc,
                [abs(v) / 1000 for v in values_cftc],
                color=colors_cftc, alpha=0.8, height=0.6)
for bar, val in zip(bars, values_cftc):
    sign = '▼ SHORT' if val < 0 else '▲ LONG'
    col  = RED2 if val < 0 else GREEN
    ax2.text(bar.get_width() + 1,
             bar.get_y() + bar.get_height() / 2,
             f'{sign}  {abs(val)/1000:.0f}K',
             va='center', fontsize=7, color=col,
             fontfamily='monospace')
ax2.set_xlabel('Net Contracts (thousands)', color='#666',
               fontsize=7, fontfamily='monospace')
ax2.set_title('CFTC Flow by Trader Type  |  May 26 2026',
              color=GOLD, fontsize=9, fontfamily='monospace', pad=4)
ax2.tick_params(colors='#666', labelsize=7)
ax2.xaxis.label.set_color('#666')

# ── Panel 3: Beta Attribution Heatmap over time ───────────────────────────────
ax3 = fig.add_subplot(gs[1, :2])
style_ax(ax3)

if len(betas_ts) > 20:
    bt_norm  = betas_ts.div(betas_ts.abs().max()).fillna(0)
    bt_weekly= bt_norm.resample('W').mean().dropna()
    heat2    = bt_weekly.T.values
    w_dates  = [d.strftime('%b %y') for d in bt_weekly.index]

    im3 = ax3.imshow(heat2, aspect='auto', cmap=cmap_rg,
                      vmin=-1, vmax=1, interpolation='bilinear')
    idx_ticks = np.linspace(0, heat2.shape[1] - 1, 12, dtype=int)
    ax3.set_xticks(idx_ticks)
    ax3.set_xticklabels(
        [w_dates[i] for i in idx_ticks],
        fontsize=7, color='#888',
        fontfamily='monospace', rotation=30)
    ax3.set_yticks(range(len(betas_ts.columns)))
    ax3.set_yticklabels(betas_ts.columns, fontsize=8,
                         color=GOLD2, fontfamily='monospace')
    ax3.set_title('Rolling 63-Day Factor Beta Attribution  |  '
                  'Intensity = driver strength',
                  color=GOLD, fontsize=9, fontfamily='monospace', pad=4)
    plt.colorbar(im3, ax=ax3, fraction=0.02, pad=0.02).ax.tick_params(
        colors='#666', labelsize=7)

# ── Panel 4: Beta % attribution donut ────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
ax4.set_facecolor('#0a0a14')
for sp in ax4.spines.values(): sp.set_color('#222')

donut_colors = [BLUE2, GREEN, RED2, GOLD2, PURP, ORNG]
factor_names = list(factors.columns)

wedges, texts, autotexts = ax4.pie(
    beta_pct,
    labels=None,
    colors=donut_colors,
    autopct='%1.0f%%',
    pctdistance=0.75,
    startangle=90,
    wedgeprops=dict(width=0.5, edgecolor='#080810', linewidth=1.5)
)
for at in autotexts:
    at.set_fontsize(7)
    at.set_color('white')
    at.set_fontfamily('monospace')
ax4.legend(wedges, factor_names,
           loc='lower center', fontsize=6.5,
           facecolor='#0d0d1a', edgecolor='#333',
           labelcolor='white', ncol=2,
           bbox_to_anchor=(0.5, -0.15))
ax4.set_title('Current Driver Attribution %\n(63d rolling beta)',
              color=GOLD, fontsize=9, fontfamily='monospace', pad=4)

# ── Panel 5: ETF Dollar Volume Flow ──────────────────────────────────────────
ax5 = fig.add_subplot(gs[2, :2])
style_ax(ax5)

gld_norm = (gld_dv / gld_dv.rolling(63).mean()).reindex(
    prices.index).fillna(1)
iau_norm = (iau_dv / iau_dv.rolling(63).mean()).reindex(
    prices.index).fillna(1)
gdx_norm = (gdx_dv / gdx_dv.rolling(63).mean()).reindex(
    prices.index).fillna(1)

ax5.fill_between(prices.index, gld_norm, 1,
                  where=gld_norm > 1,
                  color=GOLD2, alpha=0.5, label='GLD above avg')
ax5.fill_between(prices.index, gld_norm, 1,
                  where=gld_norm < 1,
                  color=RED2,  alpha=0.3, label='GLD below avg')
ax5.plot(prices.index, iau_norm, color=BLUE2,
         lw=0.7, alpha=0.7, label='IAU norm')
ax5.plot(prices.index, gdx_norm, color=PURP,
         lw=0.7, alpha=0.7, label='GDX norm')
ax5.axhline(1, color='#444', lw=0.6, ls='--')
ax5.set_ylim(0, 4)
ax5.set_title('ETF Dollar Volume Flow  |  Normalised to 63d avg  |  '
              '>1 = above average institutional participation',
              color=GOLD, fontsize=9, fontfamily='monospace', pad=4)
ax5.legend(fontsize=7, facecolor='#0d0d1a',
           edgecolor='#333', labelcolor='white', ncol=4)
ax5.tick_params(colors='#666', labelsize=7)

# ── Panel 6: Buy vs Sell Pressure ────────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, 2])
style_ax(ax6)

recent_pressure = net_pressure.dropna().iloc[-63:]
ax6.fill_between(recent_pressure.index, 0.5, recent_pressure,
                  where=recent_pressure > 0.5,
                  color=GREEN, alpha=0.5, label='Buy pressure')
ax6.fill_between(recent_pressure.index, 0.5, recent_pressure,
                  where=recent_pressure < 0.5,
                  color=RED2,  alpha=0.5, label='Sell pressure')
ax6.plot(recent_pressure.index, recent_pressure,
          color=GOLD2, lw=1.0)
ax6.axhline(0.5, color='#444', lw=0.8, ls='--')
ax6.set_ylim(0, 1)
ax6.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax6.set_yticklabels(['100% Sell', '75% Sell', 'Balanced',
                      '75% Buy',  '100% Buy'],
                     fontsize=6.5, color='#888',
                     fontfamily='monospace')

latest_p  = safe_at(recent_pressure, -1)
bias_str  = ('BUY'      if latest_p > 0.55 else
             'SELL'     if latest_p < 0.45 else 'BALANCED')
bias_col  = (GREEN if latest_p > 0.55 else
             RED2  if latest_p < 0.45 else BLUE2)

ax6.set_title(f'GLD Buy/Sell Pressure (21d)  |  '
              f'Now: {bias_str} ({latest_p*100:.0f}%)',
              color=bias_col, fontsize=9,
              fontfamily='monospace', pad=4)
ax6.legend(fontsize=7, facecolor='#0d0d1a',
           edgecolor='#333', labelcolor='white')
ax6.tick_params(axis='x', colors='#666', labelsize=6, rotation=20)

plt.tight_layout()
import io as _io, IPython.display as _ipyd
_buf = _io.BytesIO()
plt.savefig(_buf, dpi=150, bbox_inches="tight")
_buf.seek(0)
_ipyd.display(_ipyd.Image(_buf.read()))
plt.close()

# =============================================================================
# SUMMARY PRINT
# =============================================================================
print("\n" + "="*60)
print("  FLOW ATTRIBUTION SUMMARY")
print("="*60)
print(f"\n  CFTC POSITIONS (May 05 2026):")
for k, v in cftc_snapshot.items():
    bar = '█' * int(abs(v) / 10000)
    direction = '▼ SHORT' if v < 0 else '▲ LONG '
    print(f"  {k:<28s} {direction}  {abs(v):>8,.0f}  {bar}")

print(f"\n  CURRENT DRIVER ATTRIBUTION (63d beta):")
for name, pct in zip(factor_names, beta_pct):
    bar = '█' * int(pct / 5)
    print(f"  {name:<22s} {pct:5.1f}%  {bar}")

print(f"\n  ETF FLOW (latest vs 63d avg):")
print(f"  GLD dollar vol  : {safe_at(gld_norm, -1):.2f}x avg")
print(f"  IAU dollar vol  : {safe_at(iau_norm, -1):.2f}x avg")
print(f"  GDX dollar vol  : {safe_at(gdx_norm, -1):.2f}x avg")

print(f"\n  BUY/SELL PRESSURE : {bias_str} ({latest_p*100:.0f}%)")
print("="*60)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks
import yfinance as yf
import warnings
warnings.filterwarnings('ignore'); unmute_convergence()

# =============================================================================
# GOLD LIQUIDITY MAP  //  Stop Clusters · Gamma Walls · Volume Nodes
# QRD — Where is price gravitating Monday?
# =============================================================================

BG    = '#080810'
GOLD  = '#C9A84C'
GOLD2 = '#FFD700'
RED2  = '#FF4444'
BLUE2 = '#4FC3F7'
GREEN = '#00C853'
PURP  = '#B388FF'
ORNG  = '#FF9800'
WHITE = '#FFFFFF'

print("="*65)
print("  GOLD LIQUIDITY MAP  |  Stop Clusters & Gravity Zones  |  QRD")
print("="*65)

# =============================================================================
# 1. FETCH DATA
# =============================================================================
print("\n[1/5] Fetching data...")

def fetch_clean(ticker, period='60d', interval='1h'):
    try:
        df = yf.download(ticker, period=period,
                         interval=interval,
                         auto_adjust=True, progress=False)
        if hasattr(df, 'columns') and isinstance(df.columns, pd.MultiIndex):
            df.columns = [c[0] for c in df.columns]
        df = df[['Open','High','Low','Close','Volume']].ffill().dropna()
        if len(df) > 10:
            print(f"  ✓ {ticker}  {interval}  ({len(df)} bars)")
            return df
        return None
    except Exception as e:
        print(f"  ✗ {ticker}: {e}")
        return None

# Hourly for liquidity detail
df_1h  = fetch_clean('GC=F', period='60d',  interval='1h')
# Daily for swing levels
df_1d  = fetch_clean('GC=F', period='1y',   interval='1d')
# GLD options proxy levels (hardcoded from our earlier analysis)
# GLD spot = 423.18  multiply x10 = gold equivalent
GLD_MULT = 10.0

if df_1h is None or df_1d is None:
    raise RuntimeError("Data fetch failed — check connection")

# =============================================================================
# 2. LIQUIDITY ENGINE
# =============================================================================
print("\n[2/5] Computing liquidity layers...")

# ── A. Volume Profile (price histogram weighted by volume) ────────────────────
def build_volume_profile(df, n_bins=200):
    price_min = df['Low'].min()
    price_max = df['High'].max()
    bins      = np.linspace(price_min, price_max, n_bins)
    vp        = np.zeros(n_bins - 1)
    bin_mid   = (bins[:-1] + bins[1:]) / 2

    for _, row in df.iterrows():
        lo, hi, vol = row['Low'], row['High'], row['Volume']
        if hi == lo:
            idx = np.searchsorted(bins, lo) - 1
            if 0 <= idx < len(vp):
                vp[idx] += vol
        else:
            mask = (bin_mid >= lo) & (bin_mid <= hi)
            spread = mask.sum()
            if spread > 0:
                vp[mask] += vol / spread

    return bin_mid, vp

price_levels, vp_raw = build_volume_profile(df_1h, n_bins=300)
vp_smooth            = gaussian_filter1d(vp_raw, sigma=3)
vp_norm              = vp_smooth / vp_smooth.max()

# ── B. High Volume Nodes (HVN) — where price spent most time ─────────────────
hvn_idx, hvn_props = find_peaks(vp_smooth,
                                 height=vp_smooth.max() * 0.4,
                                 distance=15)
hvn_prices = price_levels[hvn_idx]
hvn_vols   = vp_smooth[hvn_idx]

# ── C. Low Volume Nodes (LVN) — air pockets price moves through fast ──────────
lvn_idx, _ = find_peaks(-vp_smooth,
                         height=-vp_smooth.max() * 0.15,
                         distance=10)
lvn_prices = price_levels[lvn_idx]

# ── D. Swing High/Low Stop Clusters ───────────────────────────────────────────
def find_swing_levels(df, lookback=5):
    highs, lows = [], []
    close = df['Close'].values
    high  = df['High'].values
    low   = df['Low'].values

    for i in range(lookback, len(df) - lookback):
        # Swing high
        if all(high[i] >= high[i-j] for j in range(1, lookback+1)) and \
           all(high[i] >= high[i+j] for j in range(1, lookback+1)):
            highs.append((df.index[i], float(high[i])))
        # Swing low
        if all(low[i] <= low[i-j] for j in range(1, lookback+1)) and \
           all(low[i] <= low[i+j] for j in range(1, lookback+1)):
            lows.append((df.index[i], float(low[i])))

    return highs, lows

swing_highs_1h, swing_lows_1h = find_swing_levels(df_1h, lookback=8)
swing_highs_1d, swing_lows_1d = find_swing_levels(df_1d, lookback=5)

# ── E. Stop Cluster Density (kernel density of swing levels) ─────────────────
all_swing_prices = (
    [p for _, p in swing_highs_1h] +
    [p for _, p in swing_lows_1h]  +
    [p for _, p in swing_highs_1d] +
    [p for _, p in swing_lows_1d]
)

if len(all_swing_prices) > 5:
    from scipy.stats import gaussian_kde
    kde        = gaussian_kde(all_swing_prices, bw_method=0.08)
    kde_levels = np.linspace(min(all_swing_prices) * 0.99,
                              max(all_swing_prices) * 1.01, 500)
    kde_density= kde(kde_levels)
    kde_norm   = kde_density / kde_density.max()

    # Stop cluster peaks
    stop_idx, _ = find_peaks(kde_density,
                              height=kde_density.max() * 0.35,
                              distance=20)
    stop_prices = kde_levels[stop_idx]
    stop_weights= kde_density[stop_idx]
else:
    kde_levels  = np.array([])
    kde_norm    = np.array([])
    stop_prices = np.array([])
    stop_weights= np.array([])

# ── F. Key Technical Levels ───────────────────────────────────────────────────
current_price = safe_at(df_1h['Close'], -1)

# EMAs
ema_20  = df_1d['Close'].ewm(span=20).mean().iloc[-1]
ema_50  = df_1d['Close'].ewm(span=50).mean().iloc[-1]
ema_100 = df_1d['Close'].ewm(span=100).mean().iloc[-1]
ema_200 = df_1d['Close'].ewm(span=200).mean().iloc[-1]

# ATR
tr      = pd.concat([
    df_1d['High'] - df_1d['Low'],
    (df_1d['High'] - df_1d['Close'].shift(1)).abs(),
    (df_1d['Low']  - df_1d['Close'].shift(1)).abs()
], axis=1).max(axis=1)
atr     = float(tr.rolling(14).mean().iloc[-1])

# Prior week levels (from our earlier analysis)
pw_high  = 4811.0
pw_low   = 4658.0
pw_close = 4722.0

# Pivot points
pivot = (pw_high + pw_low + pw_close) / 3
r1    = 2 * pivot - pw_low
r2    = pivot + (pw_high - pw_low)
r3    = pw_high + 2 * (pivot - pw_low)
s1    = 2 * pivot - pw_high
s2    = pivot - (pw_high - pw_low)
s3    = pw_low - 2 * (pw_high - pivot)

# Monthly VWAP
tp    = (df_1d['High'] + df_1d['Low'] + df_1d['Close']) / 3
mvwap = float((tp * df_1d['Volume']).rolling(21).sum().iloc[-1] /
               df_1d['Volume'].rolling(21).sum().iloc[-1])

# ── G. Gamma Wall Levels (from GLD options — multiply by 10) ─────────────────
gamma_walls = {
    '$5,000 (Gamma Wall)'   : 500.0 * GLD_MULT,
    '$4,900'                : 490.0 * GLD_MULT,
    '$4,800'                : 480.0 * GLD_MULT,
    '$4,750'                : 475.0 * GLD_MULT,
    '$4,500 (Put Wall)'     : 450.0 * GLD_MULT,
    '$4,350 (Put Floor)'    : 435.0 * GLD_MULT,
}

# ── H. Gravity Score — where is price most likely to be pulled? ───────────────
def gravity_score(target_price, current, vp_p, vp_v,
                  stop_p, stop_w, gamma_dict, atr):
    score = 0.0
    dist  = abs(target_price - current)

    # Volume node pull (higher volume = stronger gravity)
    for p, v in zip(vp_p[hvn_idx], vp_v):
        d = abs(target_price - p)
        if d < atr * 3:
            score += (v / vp_v.max()) * np.exp(-d / atr)

    # Stop cluster pull (stops = fuel = price gets pulled there)
    for p, w in zip(stop_p, stop_w):
        d = abs(target_price - p)
        if d < atr * 4:
            score += (w / stop_w.max()) * 0.8 * np.exp(-d / atr)

    # Gamma wall pull
    for label, gp in gamma_dict.items():
        d = abs(target_price - gp)
        if d < atr * 5:
            score += 0.6 * np.exp(-d / (atr * 2))

    # Distance decay
    score *= np.exp(-dist / (atr * 6))
    return score

# Build gravity curve
g_levels = np.linspace(current_price - atr * 8,
                        current_price + atr * 8, 400)
g_scores  = np.array([
    gravity_score(g, current_price,
                  price_levels, vp_smooth,
                  stop_prices, stop_weights,
                  gamma_walls, atr)
    for g in g_levels
])
g_scores_norm = g_scores / (g_scores.max() + 1e-10)

# Top gravity targets
top_gravity_idx = np.argsort(g_scores)[::-1][:5]
top_gravity_lvls= g_levels[top_gravity_idx]
top_gravity_scr = g_scores_norm[top_gravity_idx]

print(f"  ✓ Current price   : ${current_price:,.0f}")
print(f"  ✓ ATR(14)         : ${atr:,.0f}")
print(f"  ✓ Monthly VWAP    : ${mvwap:,.0f}")
print(f"  ✓ HVN nodes found : {len(hvn_prices)}")
print(f"  ✓ Stop clusters   : {len(stop_prices)}")
print(f"\n  TOP GRAVITY TARGETS:")
for lvl, scr in zip(top_gravity_lvls, top_gravity_scr):
    direction = '▲' if lvl > current_price else '▼'
    dist_atr  = abs(lvl - current_price) / atr
    print(f"    {direction} ${lvl:,.0f}   gravity={scr:.3f}   "
          f"dist={dist_atr:.1f} ATR")

# =============================================================================
# 3. PLOT
# =============================================================================
print("\n[3/5] Rendering liquidity map...")

fig = plt.figure(figsize=(22, 18), facecolor=BG)
fig.patch.set_facecolor(BG)
gs  = gridspec.GridSpec(
    3, 3, figure=fig,
    left=0.06, right=0.97,
    top=0.93, bottom=0.05,
    hspace=0.45, wspace=0.35
)

fig.suptitle(
    f'GOLD  //  LIQUIDITY MAP & STOP CLUSTER MODEL  |  '
    f'Current ${current_price:,.0f}  |  ATR ${atr:.0f}',
    fontsize=13, color=GOLD, fontweight='bold',
    fontfamily='monospace', y=0.97
)

def style(ax):
    ax.set_facecolor('#0a0a14')
    ax.tick_params(colors='#555', labelsize=7)
    for sp in ax.spines.values(): sp.set_color('#1a1a2e')
    ax.grid(True, color='#0d0d1a', lw=0.4)

# ─────────────────────────────────────────────────────────────────────────────
# PANEL 1 (left tall): Volume Profile + Stop Clusters + All Levels
# This is the main liquidity map
# ─────────────────────────────────────────────────────────────────────────────
ax_vp = fig.add_subplot(gs[:, 0])
style(ax_vp)

price_range_lo = current_price - atr * 9
price_range_hi = current_price + atr * 9
mask_vp = (price_levels >= price_range_lo) & (price_levels <= price_range_hi)

# Volume profile bars (horizontal)
ax_vp.barh(price_levels[mask_vp], vp_norm[mask_vp],
            color=GOLD2, alpha=0.25, height=price_levels[1]-price_levels[0])

# HVN highlights
for p, v in zip(hvn_prices, hvn_vols):
    if price_range_lo <= p <= price_range_hi:
        ax_vp.barh(p, v / vp_smooth.max(),
                    color=GOLD, alpha=0.7,
                    height=(price_levels[1]-price_levels[0]) * 2.5)
        ax_vp.text(v / vp_smooth.max() + 0.02, p,
                    f'HVN ${p:,.0f}',
                    va='center', fontsize=6.5, color=GOLD2,
                    fontfamily='monospace')

# Stop cluster density overlay
if len(kde_levels) > 0:
    kde_mask = (kde_levels >= price_range_lo) & (kde_levels <= price_range_hi)
    ax_vp.fill_betweenx(kde_levels[kde_mask],
                         0, kde_norm[kde_mask] * 0.6,
                         color=RED2, alpha=0.2, label='Stop density')
    # Stop cluster peaks
    for p, w in zip(stop_prices, stop_weights):
        if price_range_lo <= p <= price_range_hi:
            ax_vp.axhline(p, color=RED2, lw=0.8, ls=':', alpha=0.7)
            ax_vp.text(0.62, p, f'STOPS ${p:,.0f}',
                        va='center', fontsize=6, color=RED2,
                        fontfamily='monospace',
                        transform=ax_vp.get_yaxis_transform())

# Current price
ax_vp.axhline(current_price, color=WHITE, lw=1.5, ls='-', zorder=10)
ax_vp.text(0.02, current_price,
            f'  NOW ${current_price:,.0f}',
            va='bottom', fontsize=8, color=WHITE, fontweight='bold',
            fontfamily='monospace',
            transform=ax_vp.get_yaxis_transform())

# Pivot levels
pivot_levels = [
    (r3,    RED2,   f'R3 ${r3:,.0f}',    '--', 0.5),
    (r2,    RED2,   f'R2 ${r2:,.0f}',    '--', 0.5),
    (r1,    RED2,   f'R1 ${r1:,.0f}',    '-',  0.7),
    (pivot, ORNG,   f'PP ${pivot:,.0f}', '-',  0.8),
    (s1,    GREEN,  f'S1 ${s1:,.0f}',    '-',  0.7),
    (s2,    GREEN,  f'S2 ${s2:,.0f}',    '--', 0.5),
    (s3,    GREEN,  f'S3 ${s3:,.0f}',    '--', 0.5),
    (mvwap, BLUE2,  f'VWAP ${mvwap:,.0f}','-', 0.9),
    (ema_20, '#888',f'EMA20 ${ema_20:,.0f}',':',0.6),
    (ema_50, BLUE2, f'EMA50 ${ema_50:,.0f}',':',0.7),
    (ema_200,PURP,  f'EMA200 ${ema_200:,.0f}',':',0.8),
]
for lvl, col, lbl, ls, alpha in pivot_levels:
    if price_range_lo <= lvl <= price_range_hi:
        ax_vp.axhline(lvl, color=col, lw=0.7, ls=ls, alpha=alpha)
        ax_vp.text(0.35, lvl, lbl, va='bottom', fontsize=6,
                    color=col, fontfamily='monospace',
                    transform=ax_vp.get_yaxis_transform())

# Gamma walls
for lbl, gp in gamma_walls.items():
    if price_range_lo <= gp <= price_range_hi:
        ax_vp.axhline(gp, color=PURP, lw=1.0, ls='-.', alpha=0.6)
        ax_vp.text(0.02, gp, f'  γ {lbl}',
                    va='bottom', fontsize=6, color=PURP,
                    fontfamily='monospace',
                    transform=ax_vp.get_yaxis_transform())

# Gravity targets — replace the marker='★' block with this
for lvl, scr in zip(top_gravity_lvls, top_gravity_scr):
    if price_range_lo <= lvl <= price_range_hi and scr > 0.5:
        marker = '*' if scr > 0.8 else 'D'
        ax_vp.scatter(scr * 0.3, lvl, s=200 * scr,
                       color=GOLD2, zorder=20, alpha=0.8,
                       marker=marker)

ax_vp.set_ylim(price_range_lo, price_range_hi)
ax_vp.set_xlim(0, 1.1)
ax_vp.set_xlabel('Volume (normalised)', color='#555',
                  fontsize=7, fontfamily='monospace')
ax_vp.set_title('LIQUIDITY MAP\nVolume Profile · Stops · All Levels',
                 color=GOLD, fontsize=9,
                 fontfamily='monospace', pad=4)

# ─────────────────────────────────────────────────────────────────────────────
# PANEL 2 (top-center): Price + Swing Levels last 30 days
# ─────────────────────────────────────────────────────────────────────────────
ax_price = fig.add_subplot(gs[0, 1:])
style(ax_price)

df_recent = df_1h.iloc[-30*9:]   # ~30 trading days of hourly

ax_price.plot(df_recent.index, df_recent['Close'],
               color=GOLD2, lw=1.0, zorder=5)
ax_price.fill_between(df_recent.index,
                       df_recent['Low'], df_recent['High'],
                       color=GOLD2, alpha=0.08)

# Swing highs — potential stop hunts above
for dt, p in swing_highs_1h[-20:]:
    if dt >= df_recent.index[0]:
        ax_price.scatter(dt, p, marker='v', color=RED2,
                          s=40, zorder=8, alpha=0.8)
        ax_price.axhline(p, color=RED2, lw=0.5, ls=':', alpha=0.4)

# Swing lows — potential stop hunts below
for dt, p in swing_lows_1h[-20:]:
    if dt >= df_recent.index[0]:
        ax_price.scatter(dt, p, marker='^', color=GREEN,
                          s=40, zorder=8, alpha=0.8)
        ax_price.axhline(p, color=GREEN, lw=0.5, ls=':', alpha=0.4)

# Key levels on price chart
for lvl, col, lbl, ls, alpha in pivot_levels:
    if df_recent['Low'].min()*0.99 <= lvl <= df_recent['High'].max()*1.01:
        ax_price.axhline(lvl, color=col, lw=0.7, ls=ls, alpha=alpha)

ax_price.axhline(current_price, color=WHITE, lw=1.2, zorder=10)

# ATR bands
ax_price.axhspan(current_price - atr,
                  current_price + atr,
                  color=GOLD2, alpha=0.04,
                  label=f'±1 ATR (${atr:.0f})')

ax_price.set_title('PRICE  //  Swing Highs (▼ stops above) · '
                    'Swing Lows (▲ stops below)  |  Last 30 days',
                    color=GOLD, fontsize=9,
                    fontfamily='monospace', pad=4)
ax_price.legend(fontsize=7, facecolor='#0d0d1a',
                 edgecolor='#333', labelcolor='white')
ax_price.tick_params(axis='x', rotation=20)

# ─────────────────────────────────────────────────────────────────────────────
# PANEL 3 (mid-center): Gravity Curve
# ─────────────────────────────────────────────────────────────────────────────
ax_grav = fig.add_subplot(gs[1, 1])
style(ax_grav)

ax_grav.fill_betweenx(g_levels, 0, g_scores_norm,
                       color=GOLD, alpha=0.3)
ax_grav.plot(g_scores_norm, g_levels,
              color=GOLD2, lw=1.2)
ax_grav.axhline(current_price, color=WHITE, lw=1.2, ls='-')

# Mark gravity peaks
for lvl, scr in zip(top_gravity_lvls, top_gravity_scr):
    col = GREEN if lvl > current_price else RED2
    ax_grav.axhline(lvl, color=col, lw=0.8, ls='--', alpha=0.7)
    ax_grav.scatter(scr, lvl, s=80, color=col, zorder=10)
    ax_grav.text(scr + 0.02, lvl,
                  f'${lvl:,.0f} ({scr:.2f})',
                  va='center', fontsize=6.5, color=col,
                  fontfamily='monospace')

ax_grav.set_ylim(current_price - atr * 8,
                  current_price + atr * 8)
ax_grav.set_xlim(0, 1.2)
ax_grav.set_title('GRAVITY CURVE\nWhere price is being pulled',
                   color=GOLD, fontsize=9,
                   fontfamily='monospace', pad=4)
ax_grav.set_xlabel('Gravity Score', color='#555',
                    fontsize=7, fontfamily='monospace')

# ─────────────────────────────────────────────────────────────────────────────
# PANEL 4 (mid-right): Stop Hunt Probability by Level
# ─────────────────────────────────────────────────────────────────────────────
ax_stop = fig.add_subplot(gs[1, 2])
style(ax_stop)

# Score each stop cluster by: proximity + volume node + swing count
stop_hunt_levels = []
stop_hunt_scores = []
stop_hunt_dirs   = []

# Above current — sell stops (longs protecting)
above_swings = sorted([p for _, p in swing_highs_1h
                        if current_price < p < current_price + atr * 6],
                       key=lambda x: abs(x - current_price))
# Below current — buy stops (shorts protecting)
below_swings = sorted([p for _, p in swing_lows_1h
                        if current_price - atr * 6 < p < current_price],
                       key=lambda x: abs(x - current_price))

for p in above_swings[:8]:
    dist = p - current_price
    # proximity score
    prox  = np.exp(-dist / (atr * 3))
    # volume node nearby?
    vn_sc = max([np.exp(-abs(p - hvn) / atr) * (v / vp_smooth.max())
                 for hvn, v in zip(hvn_prices, hvn_vols)] + [0])
    total = (prox * 0.5 + vn_sc * 0.5) * 100
    stop_hunt_levels.append(p)
    stop_hunt_scores.append(total)
    stop_hunt_dirs.append('above')

for p in below_swings[:8]:
    dist = current_price - p
    prox  = np.exp(-dist / (atr * 3))
    vn_sc = max([np.exp(-abs(p - hvn) / atr) * (v / vp_smooth.max())
                 for hvn, v in zip(hvn_prices, hvn_vols)] + [0])
    total = (prox * 0.5 + vn_sc * 0.5) * 100
    stop_hunt_levels.append(p)
    stop_hunt_scores.append(total)
    stop_hunt_dirs.append('below')

if stop_hunt_levels:
    colors_sh = [RED2 if d == 'above' else GREEN
                 for d in stop_hunt_dirs]
    bars = ax_stop.barh(
        [f'${p:,.0f}' for p in stop_hunt_levels],
        stop_hunt_scores,
        color=colors_sh, alpha=0.75, height=0.6
    )
    for bar, scr, direction in zip(bars, stop_hunt_scores, stop_hunt_dirs):
        label = '▲ ABOVE' if direction == 'above' else '▼ BELOW'
        col   = RED2 if direction == 'above' else GREEN
        ax_stop.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                      f'{label}  {scr:.0f}%',
                      va='center', fontsize=6.5, color=col,
                      fontfamily='monospace')

ax_stop.set_title('STOP HUNT PROBABILITY\nRed=sell stops above · Green=buy stops below',
                   color=GOLD, fontsize=9,
                   fontfamily='monospace', pad=4)
ax_stop.set_xlabel('Hunt Probability Score', color='#555',
                    fontsize=7, fontfamily='monospace')
ax_stop.tick_params(colors='#666', labelsize=6.5)

# ─────────────────────────────────────────────────────────────────────────────
# PANEL 5 (bottom-center): Delta / Momentum Imbalance
# ─────────────────────────────────────────────────────────────────────────────
ax_delta = fig.add_subplot(gs[2, 1])
style(ax_delta)

# Approximate delta: (close - open) / range as buying/selling pressure per bar
df_r      = df_1h.iloc[-96:]   # last 4 days hourly
body      = df_r['Close'] - df_r['Open']
rng       = (df_r['High'] - df_r['Low']).replace(0, np.nan)
delta_pct = (body / rng).fillna(0)
cum_delta = delta_pct.cumsum()

colors_d  = [GREEN if v >= 0 else RED2 for v in delta_pct]
ax_delta.bar(range(len(delta_pct)), delta_pct,
              color=colors_d, alpha=0.6, width=0.8)

ax2_d = ax_delta.twinx()
ax2_d.plot(range(len(cum_delta)), cum_delta,
            color=GOLD2, lw=1.2, label='Cumulative delta')
ax2_d.tick_params(colors='#555', labelsize=7)
ax2_d.set_facecolor('#0a0a14')

latest_cd = safe_at(cum_delta, -1)
cd_bias   = 'BUYING' if latest_cd > 0 else 'SELLING'
cd_col    = GREEN if latest_cd > 0 else RED2

ax_delta.set_title(f'HOURLY DELTA IMBALANCE  |  '
                    f'Cumulative: {cd_bias} ({latest_cd:.2f})',
                    color=cd_col, fontsize=9,
                    fontfamily='monospace', pad=4)
ax_delta.set_xlim(0, len(delta_pct))
ax_delta.axhline(0, color='#333', lw=0.6)

# ─────────────────────────────────────────────────────────────────────────────
# PANEL 6 (bottom-right): Monday Scenario Summary
# ─────────────────────────────────────────────────────────────────────────────
ax_sum = fig.add_subplot(gs[2, 2])
ax_sum.set_facecolor('#0a0a14')
ax_sum.axis('off')
for sp in ax_sum.spines.values(): sp.set_color('#222')

# Find nearest gravity target above and below
above_targets = [(l, s) for l, s in
                  zip(top_gravity_lvls, top_gravity_scr)
                  if l > current_price]
below_targets = [(l, s) for l, s in
                  zip(top_gravity_lvls, top_gravity_scr)
                  if l < current_price]

nearest_above = min(above_targets, key=lambda x: x[0]) if above_targets else (current_price + atr * 2, 0)
nearest_below = max(below_targets, key=lambda x: x[0]) if below_targets else (current_price - atr * 2, 0)

# Overall bias from gravity
above_score = sum(s for l, s in zip(top_gravity_lvls, top_gravity_scr)
                   if l > current_price)
below_score = sum(s for l, s in zip(top_gravity_lvls, top_gravity_scr)
                   if l < current_price)
total_score = above_score + below_score + 1e-10
bull_pct    = above_score / total_score * 100
bear_pct    = below_score / total_score * 100

overall_bias = 'BULLISH' if bull_pct > 55 else \
               'BEARISH' if bear_pct > 55 else 'NEUTRAL'
bias_color   = GREEN if overall_bias == 'BULLISH' else \
               RED2  if overall_bias == 'BEARISH' else BLUE2

summary = f"""
  MONDAY GRAVITY SUMMARY
  ══════════════════════

  Current    ${current_price:>8,.0f}
  VWAP       ${mvwap:>8,.0f}
  Pivot PP   ${pivot:>8,.0f}
  ATR        ${atr:>8,.0f}

  ── GRAVITY TARGETS ──

  Bull target  ${nearest_above[0]:>7,.0f}
  (score {nearest_above[1]:.3f})

  Bear target  ${nearest_below[0]:>7,.0f}
  (score {nearest_below[1]:.3f})

  ── DIRECTIONAL BIAS ──

  Bull pull  {bull_pct:5.1f}%
  Bear pull  {bear_pct:5.1f}%

  OVERALL: {overall_bias}

  ── CRITICAL LEVELS ──

  Break above ${r1:,.0f} → ${r2:,.0f}
  Hold above  ${s1:,.0f} → OK
  Break below ${s2:,.0f} → ${s3:,.0f}
  Gamma wall  $5,000 magnet

  ── STOP HUNT RISK ──

  Above: sell stops near
    ${(above_swings[0] if above_swings else current_price+atr):,.0f}
  Below: buy stops near
    ${(below_swings[0] if below_swings else current_price-atr):,.0f}
"""

ax_sum.text(0.05, 0.97, summary,
             transform=ax_sum.transAxes,
             fontsize=7.8, color='#cccccc',
             fontfamily='monospace', va='top',
             bbox=dict(boxstyle='round,pad=0.5',
                       facecolor='#0d0d1a',
                       edgecolor=bias_color, lw=1.5))

ax_sum.text(0.5, 0.05, f'▶  {overall_bias}  ({bull_pct:.0f}% / {bear_pct:.0f}%)',
             transform=ax_sum.transAxes,
             fontsize=11, color=bias_color,
             fontfamily='monospace', fontweight='bold',
             ha='center', va='bottom')

plt.tight_layout()
import io as _io, IPython.display as _ipyd
_buf = _io.BytesIO()
plt.savefig(_buf, dpi=150, bbox_inches="tight")
_buf.seek(0)
_ipyd.display(_ipyd.Image(_buf.read()))
plt.close()

# =============================================================================
# FINAL PRINT
# =============================================================================
print("\n" + "="*65)
print("  MONDAY LIQUIDITY SUMMARY")
print("="*65)
print(f"  Current price    : ${current_price:,.0f}")
print(f"  Monthly VWAP     : ${mvwap:,.0f}  "
      f"{'(price above — bullish)' if current_price > mvwap else '(price below — bearish)'}")
print(f"  Weekly Pivot PP  : ${pivot:,.0f}  "
      f"{'(above — bullish)' if current_price > pivot else '(below — bearish)'}")
print(f"  ATR(14)          : ${atr:,.0f}")
print(f"\n  GRAVITY TARGETS:")
for lvl, scr in zip(top_gravity_lvls, top_gravity_scr):
    direction = '▲ ABOVE' if lvl > current_price else '▼ BELOW'
    print(f"    {direction}  ${lvl:,.0f}   score={scr:.3f}")
print(f"\n  NEAREST STOP CLUSTERS:")
if above_swings:
    print(f"    Sell stops ABOVE : ${above_swings[0]:,.0f}  "
          f"(+${above_swings[0]-current_price:.0f} / "
          f"+{(above_swings[0]-current_price)/atr:.1f} ATR)")
if below_swings:
    print(f"    Buy  stops BELOW : ${below_swings[0]:,.0f}  "
          f"(-${current_price-below_swings[0]:.0f} / "
          f"-{(current_price-below_swings[0])/atr:.1f} ATR)")
print(f"\n  DIRECTIONAL GRAVITY : {overall_bias} "
      f"({bull_pct:.0f}% up / {bear_pct:.0f}% down)")
print("="*65)


In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── CONFIG ──────────────────────────────────────────────────────────────────
PRICE_HIGH = 5200
PRICE_LOW  = 4200
CURRENT    = 4562.4
POC        = 4556.9
VWAP       = 4731.5
PDH        = 4640.9
PDL        = 4510.1
LONDON_H   = 4568.1
LONDON_L   = 4553.3
ASIA_H     = 4535.0
ASIA_L     = 4522.7

RETAIL_STOPS = [5000,4900,4850,4800,4750,4700,4640,4600,4550,4500,4450,4400,4350,4300,4250]
SM_TP        = [5150,5080,5020,4960,4870,4820,4780,4720,4680,4620,4580,4480,4420,4370,4240]

DATE_LABELS = [
    '06/03','10/03','16/03','22/03','28/03',
    '03/04','09/04','15/04','21/04','27/04','03/05','05/05'
]

# ── REPRODUCIBLE RNG ─────────────────────────────────────────────────────────
rng = np.random.default_rng(0xDEADBEEF)

# ── LIQUIDITY NODE GENERATION ────────────────────────────────────────────────
clusters = [
    (5100,0.05,30,12,0.90),(5000,0.08,25,14,1.00),(4980,0.12,20,10,0.85),
    (4870,0.18,18,11,0.90),(4820,0.25,22,13,0.95),(4800,0.30,15, 9,0.85),
    (4780,0.35,20,11,0.88),(4750,0.38,18,10,0.82),(4720,0.45,16,12,0.87),
    (4680,0.50,14,10,0.80),(4640,0.55,18,13,0.92),(4600,0.60,12,11,0.85),
    (4580,0.65,10,10,0.80),(4556,0.70,14,15,0.95),(4530,0.72,12, 9,0.78),
    (4500,0.75,20,12,0.90),(4450,0.80,16,10,0.82),(4420,0.82,12, 8,0.75),
    (4380,0.87,18,10,0.80),(4340,0.92,14, 9,0.78),(4300,0.95,16,11,0.85),
    (4260,0.97,12, 8,0.72),
]

node_price, node_time, node_intensity = [], [], []

for (p, t, sig, n, base) in clusters:
    prices = rng.normal(p, sig, n)
    times  = np.clip(rng.normal(t, 0.04, n), 0, 1)
    intens = base * (0.7 + rng.random(n) * 0.3)
    node_price.extend(prices)
    node_time.extend(times)
    node_intensity.extend(intens)

# Background scatter
bp = rng.uniform(PRICE_LOW, PRICE_HIGH, 80)
bt = rng.uniform(0, 1, 80)
bi = rng.random(80) * 0.35
node_price.extend(bp); node_time.extend(bt); node_intensity.extend(bi)

node_price     = np.array(node_price)
node_time      = np.array(node_time)
node_intensity = np.array(node_intensity)

# ── KDE HEATMAP GRID ─────────────────────────────────────────────────────────
NX, NY   = 300, 400
t_grid   = np.linspace(0, 1, NX)
p_grid   = np.linspace(PRICE_LOW, PRICE_HIGH, NY)
TT, PP   = np.meshgrid(t_grid, p_grid)

SIGMA_T = 0.035
SIGMA_P = 38.0

Z = np.zeros((NY, NX))
for i in range(len(node_price)):
    kt = np.exp(-0.5 * ((TT - node_time[i])      / SIGMA_T) ** 2)
    kp = np.exp(-0.5 * ((PP - node_price[i])     / SIGMA_P) ** 2)
    Z += node_intensity[i] * kt * kp

Z = np.clip(Z / Z.max(), 0, 1)

# ── VOLUME PROFILE ───────────────────────────────────────────────────────────
BINS      = 200
bin_edges = np.linspace(PRICE_LOW, PRICE_HIGH, BINS + 1)
bin_mids  = 0.5 * (bin_edges[:-1] + bin_edges[1:])
vol_bins  = np.zeros(BINS)

for i in range(len(node_price)):
    idx = int((node_price[i] - PRICE_LOW) / (PRICE_HIGH - PRICE_LOW) * BINS)
    idx = np.clip(idx, 0, BINS - 1)
    vol_bins[idx] += node_intensity[i]

vol_bins = vol_bins / vol_bins.max()

# ── CUSTOM COLORSCALE (dark blue → red → yellow) ────────────────────────────
colorscale = [
    [0.00, '#04080f'],
    [0.10, '#00008b'],
    [0.25, '#0000cd'],
    [0.40, '#cc0000'],
    [0.60, '#ff4400'],
    [0.75, '#ff8800'],
    [0.90, '#ffcc00'],
    [1.00, '#ffff33'],
]

# ── BUILD FIGURE ─────────────────────────────────────────────────────────────
fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.87, 0.13],
    shared_yaxes=True,
    horizontal_spacing=0.005
)

# ── HEATMAP ──────────────────────────────────────────────────────────────────
x_labels = [DATE_LABELS[int(i / NX * (len(DATE_LABELS)-1))] for i in range(NX)]

fig.add_trace(go.Heatmap(
    z=Z,
    x=t_grid,
    y=p_grid,
    colorscale=colorscale,
    showscale=True,
    zmin=0, zmax=1,
    colorbar=dict(
        x=0.845, y=0.5, len=0.9,
        thickness=12,
        title=dict(text='Liquidity<br>Intensity', font=dict(color='#5a7a90', size=9), side='right'),
        tickfont=dict(color='#5a7a90', size=8),
        tickvals=[0, 0.25, 0.5, 0.75, 1.0],
        ticktext=['Low','','Med','','High'],
        outlinecolor='#0e2a3a', outlinewidth=1,
    ),
    hovertemplate='Price: %{y:.1f}<br>Intensity: %{z:.3f}<extra></extra>',
    name='Liquidity'
), row=1, col=1)

# ── HORIZONTAL REFERENCE LINES ───────────────────────────────────────────────
def hline(fig, price, color, dash, width, label, show_label=True):
    fig.add_shape(type='line', x0=0, x1=1, y0=price, y1=price,
                  line=dict(color=color, width=width, dash=dash),
                  xref='x', yref='y', row=1, col=1)
    if show_label:
        fig.add_annotation(x=0.995, y=price, text=label,
                           font=dict(color=color, size=8, family='Courier New'),
                           showarrow=False, xanchor='right', yanchor='bottom',
                           xref='x', yref='y', row=1, col=1)

hline(fig, VWAP,     '#00e5ff', 'dash',    1.2, f'VWAP: {VWAP}')
hline(fig, PDH,      '#ff4a6e', 'dot',     1.0, f'PDH: {PDH}')
hline(fig, PDL,      '#ff4a6e', 'dot',     1.0, f'PDL: {PDL}')
hline(fig, LONDON_H, '#ff9944', 'dashdot', 0.8, f'London H: {LONDON_H}')
hline(fig, LONDON_L, '#ff9944', 'dashdot', 0.8, f'London L: {LONDON_L}')
hline(fig, ASIA_H,   '#44cc88', 'dot',     0.7, f'Asia H: {ASIA_H}')
hline(fig, ASIA_L,   '#44cc88', 'dot',     0.7, f'Asia L: {ASIA_L}')
hline(fig, POC,      '#f5c842', 'solid',   2.0, f'POC: {POC}')
hline(fig, CURRENT,  '#ffffff', 'solid',   1.5, f'▶ {CURRENT}')

# ── RETAIL STOP MARKERS ───────────────────────────────────────────────────────
for p in RETAIL_STOPS:
    fig.add_shape(type='line', x0=0, x1=1, y0=p, y1=p,
                  line=dict(color='#50fa7b', width=0.6, dash='dot'),
                  xref='x', yref='y', row=1, col=1)
    fig.add_trace(go.Scatter(
        x=[0.01], y=[p],
        mode='markers+text',
        marker=dict(symbol='triangle-right', color='#50fa7b', size=7, opacity=0.85),
        text=[f'  STOP {p}'],
        textposition='middle right',
        textfont=dict(color='#50fa7b', size=7, family='Courier New'),
        showlegend=False,
        hovertemplate=f'Retail Stop: {p}<extra></extra>',
        name='Retail Stop'
    ), row=1, col=1)

# ── SMART MONEY TP MARKERS ───────────────────────────────────────────────────
for p in SM_TP:
    fig.add_shape(type='line', x0=0, x1=1, y0=p, y1=p,
                  line=dict(color='#bd93f9', width=0.6, dash='dot'),
                  xref='x', yref='y', row=1, col=1)
    fig.add_trace(go.Scatter(
        x=[0.99], y=[p],
        mode='markers+text',
        marker=dict(symbol='diamond', color='#bd93f9', size=6, opacity=0.85),
        text=[f'SM TP {p}  '],
        textposition='middle left',
        textfont=dict(color='#bd93f9', size=7, family='Courier New'),
        showlegend=False,
        hovertemplate=f'Smart Money TP: {p}<extra></extra>',
        name='SM TP'
    ), row=1, col=1)

# ── VOLUME PROFILE (right panel) ─────────────────────────────────────────────
vol_colors = []
for v in vol_bins:
    if v > 0.75:   vol_colors.append('#ffff00')
    elif v > 0.50: vol_colors.append('#ff8800')
    elif v > 0.25: vol_colors.append('#ff3300')
    else:          vol_colors.append('#550011')

fig.add_trace(go.Bar(
    x=vol_bins,
    y=bin_mids,
    orientation='h',
    marker=dict(color=vol_colors, line=dict(width=0)),
    showlegend=False,
    hovertemplate='Price: %{y:.0f}<br>Vol: %{x:.3f}<extra></extra>',
    name='Vol Profile'
), row=1, col=2)

# POC on vol profile
fig.add_shape(type='line', x0=0, x1=1, y0=POC, y1=POC,
              line=dict(color='#f5c842', width=2),
              xref='x2', yref='y2', row=1, col=2)

# Current price on vol profile
fig.add_shape(type='line', x0=0, x1=1, y0=CURRENT, y1=CURRENT,
              line=dict(color='#ffffff', width=1.2),
              xref='x2', yref='y2', row=1, col=2)

# ── X-AXIS TICK LABELS ────────────────────────────────────────────────────────
tick_positions = np.linspace(0, 1, len(DATE_LABELS))

# ── LAYOUT ────────────────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text=(
            '<b style="color:#f5c842;letter-spacing:3px">⬡ GOLD LIQUIDITY HEATMAP · XAU/USD FUTURES</b>'
            f'<br><span style="font-size:11px;color:#5a7a90;letter-spacing:2px">'
            f'GC FUTURES · LAST 60 DAYS · CURRENT: <b style="color:#f5c842">{CURRENT}</b> '
            f'· POC: <b style="color:#f5c842">{POC}</b> '
            f'· VWAP: <b style="color:#00e5ff">{VWAP}</b></span>'
        ),
        font=dict(family='Courier New', size=14, color='#f5c842'),
        x=0.01, xanchor='left', y=0.98
    ),
    paper_bgcolor='#04080f',
    plot_bgcolor='#04080f',
    height=700,
    margin=dict(l=10, r=20, t=80, b=60),
    font=dict(family='Courier New', color='#c8d6e5'),

    xaxis=dict(
        tickvals=list(tick_positions),
        ticktext=DATE_LABELS,
        tickfont=dict(size=8, color='#2a4a5a', family='Courier New'),
        showgrid=True,
        gridcolor='#0e2a3a',
        gridwidth=0.5,
        zeroline=False,
        showline=False,
        range=[0, 1],
        title=dict(text='TIME (ATHENS / UTC+3)', font=dict(size=9, color='#2a4a5a')),
    ),
    yaxis=dict(
        range=[PRICE_LOW, PRICE_HIGH],
        tickfont=dict(size=8, color='#5a7a90', family='Courier New'),
        showgrid=True,
        gridcolor='#0a1e2b',
        gridwidth=0.4,
        zeroline=False,
        dtick=50,
        title=dict(text='PRICE (USD/oz)', font=dict(size=9, color='#2a4a5a')),
        side='left',
    ),
    xaxis2=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        title=dict(text='VOL PROFILE', font=dict(size=8, color='#2a4a5a')),
    ),
    yaxis2=dict(
        range=[PRICE_LOW, PRICE_HIGH],
        showticklabels=False,
        showgrid=False,
        zeroline=False,
    ),
    bargap=0,
    showlegend=False,
)

# ── LEGEND ANNOTATIONS ────────────────────────────────────────────────────────
legend_items = [
    ('▶', '#ffffff',  f'Current {CURRENT}'),
    ('━', '#f5c842',  f'POC {POC}'),
    ('━', '#00e5ff',  f'VWAP {VWAP}'),
    ('━', '#ff4a6e',  'PDH / PDL'),
    ('━', '#ff9944',  'London H/L'),
    ('━', '#44cc88',  'Asia H/L'),
    ('▶', '#50fa7b',  'Retail Stops'),
    ('◆', '#bd93f9',  'Smart Money TP'),
]

for i, (sym, col, label) in enumerate(legend_items):
    fig.add_annotation(
        x=0.01 + i * 0.118, y=1.045,
        text=f'<span style="color:{col}">{sym} {label}</span>',
        font=dict(size=8, family='Courier New', color=col),
        showarrow=False, xref='paper', yref='paper',
        xanchor='left'
    )

fig.show()


In [ ]:
# ── INSTALL IF NEEDED ─────────────────────────────────────────────────────────
# pip install yfinance plotly scipy pandas numpy

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.ndimage import gaussian_filter
import yfinance as yf
import warnings
warnings.filterwarnings('ignore'); unmute_convergence()

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1 — FETCH REAL GOLD FUTURES DATA (GC=F, CME)
# ═══════════════════════════════════════════════════════════════════════════════
print("Fetching GC Gold Futures data...")

# 1-hour bars, last 60 days — best balance of resolution vs history on yfinance
gc_1h = yf.download("GC=F", period="60d", interval="1h", auto_adjust=True, progress=False)
gc_1h.dropna(inplace=True)

if gc_1h.empty:
    raise ValueError("No data returned. Check your internet connection.")

# Flatten MultiIndex columns if present
if hasattr(gc_1h, 'columns') and isinstance(gc_1h.columns, pd.MultiIndex):
    gc_1h.columns = gc_1h.columns.get_level_values(0)

price_close  = gc_1h["Close"].values.astype(float)
price_open   = gc_1h["Open"].values.astype(float)
price_high_b = gc_1h["High"].values.astype(float)
price_low_b  = gc_1h["Low"].values.astype(float)
raw_volume   = gc_1h["Volume"].values.astype(float)

# Replace zero volumes with small positive value
raw_volume   = np.where(raw_volume <= 0, 1.0, raw_volume)

N_T          = len(price_close)
t_axis       = np.linspace(0, 1, N_T)

CURRENT      = float(price_close[-1])
PRICE_HIGH   = float(price_high_b.max()) + 30
PRICE_LOW    = float(price_low_b.min())  - 30
RESOLUTION   = 0.5  # $0.50 per row

price_axis   = np.arange(PRICE_LOW, PRICE_HIGH + RESOLUTION, RESOLUTION)
N_P          = len(price_axis)

# Time index for x-axis labels
time_index   = gc_1h.index

print(f"Loaded {N_T} bars | {time_index[0].strftime('%Y-%m-%d')} → {time_index[-1].strftime('%Y-%m-%d')}")
print(f"Price range: ${PRICE_LOW:.1f} – ${PRICE_HIGH:.1f} | Current: ${CURRENT:.1f}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2 — REAL VOLUME PROFILE
# Count actual traded volume at each price level using bar OHLC
# Distributes each bar's volume across its high-low range (VWAP-weighted)
# ═══════════════════════════════════════════════════════════════════════════════
vol_profile = np.zeros(N_P)

for i in range(N_T):
    bar_high  = price_high_b[i]
    bar_low   = price_low_b[i]
    bar_close = price_close[i]
    bar_vol   = raw_volume[i]

    # Find price indices within this bar's range
    lo_idx = np.searchsorted(price_axis, bar_low)
    hi_idx = np.searchsorted(price_axis, bar_high)
    if lo_idx == hi_idx:
        hi_idx = min(lo_idx + 1, N_P - 1)

    n_levels = hi_idx - lo_idx + 1
    if n_levels < 1:
        continue

    # Weight volume toward close (VWAP approximation)
    level_prices = price_axis[lo_idx:hi_idx + 1]
    weights      = np.exp(-0.5 * ((level_prices - bar_close) / max(1.0, (bar_high - bar_low) * 0.3)) ** 2)
    weights     /= weights.sum() + 1e-10
    vol_profile[lo_idx:hi_idx + 1] += bar_vol * weights

# Smooth and normalize
vol_profile_smooth = gaussian_filter(vol_profile, sigma=2.5)
vol_norm           = vol_profile_smooth / (vol_profile_smooth.max() + 1e-10)

# ── POINT OF CONTROL (POC) — highest volume price ─────────────────────────────
poc_idx  = np.argmax(vol_profile_smooth)
POC      = price_axis[poc_idx]

# ── VALUE AREA (70% of volume around POC) ────────────────────────────────────
total_vol = vol_profile_smooth.sum()
target    = 0.70 * total_vol
sorted_idx= np.argsort(vol_profile_smooth)[::-1]
cum       = 0.0
va_set    = []
for idx in sorted_idx:
    cum += vol_profile_smooth[idx]
    va_set.append(idx)
    if cum >= target:
        break

VAH = price_axis[max(va_set)]
VAL = price_axis[min(va_set)]

# ── VWAP (real calculation from OHLC) ─────────────────────────────────────────
typical_price = (price_high_b + price_low_b + price_close) / 3.0
VWAP          = float(np.sum(typical_price * raw_volume) / np.sum(raw_volume))

print(f"POC:  ${POC:.1f} | VAH: ${VAH:.1f} | VAL: ${VAL:.1f} | VWAP: ${VWAP:.1f}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3 — REAL CVD (CUMULATIVE VOLUME DELTA)
# Classify each bar as buy or sell using BVC (Bulk Volume Classification)
# Aggressor side = direction of close relative to midpoint
# ═══════════════════════════════════════════════════════════════════════════════
bar_range  = price_high_b - price_low_b
midpoint   = (price_high_b + price_low_b) / 2.0
buy_frac   = np.where(
    bar_range > 0,
    np.clip((price_close - price_low_b) / bar_range, 0.0, 1.0),
    0.5
)
buy_vol    = raw_volume * buy_frac
sell_vol   = raw_volume * (1.0 - buy_frac)
delta      = buy_vol - sell_vol
CVD        = np.cumsum(delta)

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4 — REAL VPIN (Volume-Synchronized Probability of Informed Trading)
# Easley, López de Prado, O'Hara (2012)
# High VPIN = toxic/informed flow = institutional activity
# ═══════════════════════════════════════════════════════════════════════════════
BUCKET_VOL   = max(1.0, raw_volume.mean() * 1.5)
buckets      = []
run_buy      = 0.0
run_sell     = 0.0
run_vol      = 0.0

for i in range(N_T):
    run_buy  += buy_vol[i]
    run_sell += sell_vol[i]
    run_vol  += raw_volume[i]
    if run_vol >= BUCKET_VOL:
        imbalance = abs(run_buy - run_sell) / (run_vol + 1e-10)
        buckets.append(imbalance)
        run_buy = run_sell = run_vol = 0.0

VPIN_WIN = max(1, len(buckets) // 8)
if len(buckets) >= VPIN_WIN:
    vpin_raw = pd.Series(buckets).rolling(VPIN_WIN, min_periods=1).mean().values
    vpin_exp = np.interp(
        np.linspace(0, 1, N_T),
        np.linspace(0, 1, len(vpin_raw)),
        vpin_raw
    )
else:
    vpin_exp = np.full(N_T, 0.3)

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5 — LOB HEATMAP FROM REAL DATA
# Build limit order book density from real price action:
# Where price repeatedly stalls, tests, reverses = passive order presence
# High-volume nodes become bright bands (institutional walls)
# ═══════════════════════════════════════════════════════════════════════════════
LOB = np.zeros((N_P, N_T), dtype=np.float32)

# Background: volume-weighted density at each price/time
for t in range(N_T):
    close_idx = np.argmin(np.abs(price_axis - price_close[t]))
    lo_idx    = np.searchsorted(price_axis, price_low_b[t])
    hi_idx    = np.searchsorted(price_axis, price_high_b[t])
    hi_idx    = min(hi_idx, N_P - 1)
    lo_idx    = max(lo_idx, 0)

    if hi_idx <= lo_idx:
        continue

    # Volume distribution within bar range
    lvls    = price_axis[lo_idx:hi_idx + 1]
    w       = np.exp(-0.5 * ((lvls - price_close[t]) / max(1.0, (price_high_b[t] - price_low_b[t]) * 0.35)) ** 2)
    w      /= w.sum() + 1e-10
    LOB[lo_idx:hi_idx + 1, t] += (raw_volume[t] * w * 0.001).astype(np.float32)

# Add structural walls at high-volume price nodes
# These represent where passive institutional orders likely rested
vol_threshold = np.percentile(vol_profile_smooth, 80)
wall_indices  = np.where(vol_profile_smooth > vol_threshold)[0]

rng = np.random.default_rng(42)
for pidx in wall_indices:
    wall_strength = vol_profile_smooth[pidx] / (vol_profile_smooth.max() + 1e-10)
    # Wall exists across time but gets absorbed as price approaches
    for t in range(N_T):
        dist  = abs(price_close[t] - price_axis[pidx])
        absorb= np.clip(dist / 20.0, 0.05, 1.0)
        LOB[pidx, t] += float(wall_strength * absorb * 8.0)

# Slight temporal smoothing (order flow evolves continuously)
LOB = gaussian_filter(LOB, sigma=[1.2, 0.4])

# Log-scale + normalize (matches Bookmap rendering)
LOB_log  = np.log1p(LOB)
LOB_norm = LOB_log / (LOB_log.max() + 1e-10)

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6 — IDENTIFY KEY LEVELS FROM REAL DATA
# ── Prior Day High/Low ────────────────────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════════
gc_1d = yf.download("GC=F", period="5d", interval="1d", auto_adjust=True, progress=False)
gc_1d.dropna(inplace=True)

if hasattr(gc_1d, 'columns') and isinstance(gc_1d.columns, pd.MultiIndex):
    gc_1d.columns = gc_1d.columns.get_level_values(0)

if len(gc_1d) >= 2:
    PDH = safe_at(gc_1d["High"], -2)
    PDL = safe_at(gc_1d["Low"], -2)
else:
    PDH = CURRENT + 30
    PDL = CURRENT - 30

# ── Session Highs/Lows (London & Asia from intraday data) ────────────────────
gc_1h["hour"] = time_index.hour
gc_1h["date"] = time_index.date

latest_date   = gc_1h["date"].iloc[-1]
today_bars    = gc_1h[gc_1h["date"] == latest_date]

asia_bars     = today_bars[today_bars["hour"].between(0, 7)]
london_bars   = today_bars[today_bars["hour"].between(7, 15)]

ASIA_H   = float(asia_bars["High"].max())   if not asia_bars.empty   else CURRENT + 15
ASIA_L   = float(asia_bars["Low"].min())    if not asia_bars.empty   else CURRENT - 15
LONDON_H = float(london_bars["High"].max()) if not london_bars.empty else CURRENT + 20
LONDON_L = float(london_bars["Low"].min())  if not london_bars.empty else CURRENT - 20

print(f"PDH: ${PDH:.1f} | PDL: ${PDL:.1f}")
print(f"London H: ${LONDON_H:.1f} | London L: ${LONDON_L:.1f}")
print(f"Asia H:   ${ASIA_H:.1f}   | Asia L:   ${ASIA_L:.1f}")

# ── Retail Stop Clusters — round numbers within range ─────────────────────────
round_step       = 50.0
retail_base      = np.arange(
    np.floor(PRICE_LOW / round_step) * round_step,
    np.ceil(PRICE_HIGH / round_step) * round_step + round_step,
    round_step
)
RETAIL_STOPS     = [p for p in retail_base if PRICE_LOW < p < PRICE_HIGH]

# ── Smart Money TP Zones — 25-pt offsets above/below retail clusters ──────────
SM_TP            = []
for p in RETAIL_STOPS:
    # SM TPs sit just above/below retail stop clusters (they hunt stops then TP)
    for offset in [12.0, -12.0]:
        tp = p + offset
        if PRICE_LOW < tp < PRICE_HIGH:
            SM_TP.append(round(tp, 1))
SM_TP = sorted(list(set(SM_TP)))

# ── Swing Highs/Lows from real price (significant turning points) ─────────────
def find_swings(prices, window=5):
    highs, lows = [], []
    for i in range(window, len(prices) - window):
        if prices[i] == max(prices[i - window:i + window + 1]):
            highs.append((i, prices[i]))
        if prices[i] == min(prices[i - window:i + window + 1]):
            lows.append((i, prices[i]))
    return highs, lows

swing_highs, swing_lows = find_swings(price_close, window=8)

# Keep only most significant swings (largest deviation from current)
swing_highs = sorted(swing_highs, key=lambda x: abs(x[1] - CURRENT), reverse=True)[:6]
swing_lows  = sorted(swing_lows,  key=lambda x: abs(x[1] - CURRENT), reverse=True)[:6]

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 7 — BUILD FIGURE
# ═══════════════════════════════════════════════════════════════════════════════
fig = make_subplots(
    rows=3, cols=2,
    column_widths=[0.85, 0.15],
    row_heights=[0.70, 0.15, 0.15],
    shared_xaxes=False,
    shared_yaxes=False,
    vertical_spacing=0.03,
    horizontal_spacing=0.008,
)

# ── BOOKMAP COLORSCALE ────────────────────────────────────────────────────────
BOOKMAP_CS = [
    [0.000, '#020b14'],
    [0.060, '#041824'],
    [0.130, '#062840'],
    [0.220, '#083860'],
    [0.320, '#0a5080'],
    [0.440, '#1070a0'],
    [0.560, '#2090c0'],
    [0.660, '#40b0d0'],
    [0.740, '#ffcc00'],
    [0.830, '#ff8800'],
    [0.910, '#ff4400'],
    [0.965, '#ff1100'],
    [1.000, '#ffffff'],
]

# ── ROW 1 COL 1 — LOB HEATMAP ─────────────────────────────────────────────────
fig.add_trace(go.Heatmap(
    z=LOB_norm,
    x=t_axis,
    y=price_axis,
    colorscale=BOOKMAP_CS,
    showscale=True,
    zmin=0, zmax=1,
    colorbar=dict(
        x=0.823, y=0.72, len=0.50,
        thickness=10,
        title=dict(
            text='ORDER<br>DENSITY',
            font=dict(color='#3a6a8a', size=8, family='Courier New'),
            side='right'
        ),
        tickfont=dict(color='#3a6a8a', size=7, family='Courier New'),
        tickvals=[0, 0.25, 0.5, 0.75, 1.0],
        ticktext=['Thin', '', 'Mid', '', 'Wall'],
        outlinecolor='#0a1e2b', outlinewidth=1,
        bgcolor='rgba(2,11,20,0.85)'
    ),
    hovertemplate='Price: $%{y:.1f}<br>Density: %{z:.3f}<extra></extra>',
    name='LOB',
    xaxis='x', yaxis='y'
), row=1, col=1)

# ── PRICE PATH (real) ─────────────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=t_axis,
    y=price_close,
    mode='lines',
    line=dict(color='#ffffff', width=1.3),
    name='GC Price',
    hovertemplate='%{text}<br>Price: $%{y:.1f}<extra></extra>',
    text=[str(t.strftime('%Y-%m-%d %H:%M')) for t in time_index],
    xaxis='x', yaxis='y'
), row=1, col=1)

# ── HELPER: draw horizontal level ─────────────────────────────────────────────
def hline(price, color, dash, lw, label, lx=0.985, anch='right'):
    if not (PRICE_LOW < price < PRICE_HIGH):
        return
    fig.add_shape(
        type='line', x0=0, x1=1, y0=price, y1=price,
        line=dict(color=color, width=lw, dash=dash),
        xref='x', yref='y', row=1, col=1
    )
    fig.add_annotation(
        x=lx, y=price, text=label,
        font=dict(color=color, size=7.5, family='Courier New'),
        showarrow=False, xanchor=anch, yanchor='bottom',
        xref='x', yref='y', row=1, col=1,
        bgcolor='rgba(2,11,20,0.7)',
    )

# Real key levels
hline(VWAP,     '#00bfff', 'dash',    1.3, f'VWAP ${VWAP:.1f}')
hline(POC,      '#f5c842', 'solid',   2.0, f'POC ${POC:.1f}')
hline(VAH,      '#8888ff', 'dot',     0.9, f'VAH ${VAH:.1f}')
hline(VAL,      '#8888ff', 'dot',     0.9, f'VAL ${VAL:.1f}', lx=0.015, anch='left')
hline(PDH,      '#ff6b6b', 'dot',     1.1, f'PDH ${PDH:.1f}')
hline(PDL,      '#ff6b6b', 'dot',     1.1, f'PDL ${PDL:.1f}', lx=0.015, anch='left')
hline(LONDON_H, '#ffa040', 'dashdot', 0.8, f'Lon H ${LONDON_H:.1f}')
hline(LONDON_L, '#ffa040', 'dashdot', 0.8, f'Lon L ${LONDON_L:.1f}', lx=0.015, anch='left')
hline(ASIA_H,   '#40cc80', 'dot',     0.7, f'Asia H ${ASIA_H:.1f}')
hline(ASIA_L,   '#40cc80', 'dot',     0.7, f'Asia L ${ASIA_L:.1f}', lx=0.015, anch='left')

# Current price badge
if PRICE_LOW < CURRENT < PRICE_HIGH:
    fig.add_shape(
        type='line', x0=0, x1=1, y0=CURRENT, y1=CURRENT,
        line=dict(color='#ffffff', width=1.6),
        xref='x', yref='y', row=1, col=1
    )
    fig.add_annotation(
        x=0.995, y=CURRENT,
        text=f' ▶ ${CURRENT:.1f} ',
        font=dict(color='#020b14', size=8.5, family='Courier New'),
        showarrow=False, xanchor='right', yanchor='middle',
        xref='x', yref='y', row=1, col=1,
        bgcolor='#f5c842', bordercolor='#f5c842', borderwidth=1,
    )

# ── SWING HIGH/LOW MARKERS ────────────────────────────────────────────────────
for tidx, price in swing_highs:
    if PRICE_LOW < price < PRICE_HIGH:
        tx = t_axis[tidx]
        fig.add_trace(go.Scatter(
            x=[tx], y=[price + 4],
            mode='markers',
            marker=dict(symbol='triangle-down', color='#ff4444', size=7, opacity=0.85),
            showlegend=False,
            hovertemplate=f'Swing High: ${price:.1f}<extra></extra>',
            xaxis='x', yaxis='y'
        ), row=1, col=1)

for tidx, price in swing_lows:
    if PRICE_LOW < price < PRICE_HIGH:
        tx = t_axis[tidx]
        fig.add_trace(go.Scatter(
            x=[tx], y=[price - 4],
            mode='markers',
            marker=dict(symbol='triangle-up', color='#44ff88', size=7, opacity=0.85),
            showlegend=False,
            hovertemplate=f'Swing Low: ${price:.1f}<extra></extra>',
            xaxis='x', yaxis='y'
        ), row=1, col=1)

# ── RETAIL STOP CLUSTERS ──────────────────────────────────────────────────────
for p in RETAIL_STOPS:
    if not (PRICE_LOW < p < PRICE_HIGH):
        continue
    fig.add_shape(
        type='line', x0=0, x1=0.06, y0=p, y1=p,
        line=dict(color='#50fa7b', width=0.8, dash='dot'),
        xref='x', yref='y', row=1, col=1
    )
    fig.add_trace(go.Scatter(
        x=[0.004], y=[p],
        mode='markers+text',
        marker=dict(symbol='triangle-right', color='#50fa7b', size=6, opacity=0.9),
        text=[f'  SL {p:.0f}'],
        textposition='middle right',
        textfont=dict(color='#50fa7b', size=6.5, family='Courier New'),
        showlegend=False,
        hovertemplate=f'Retail Stop: ${p:.0f}<extra></extra>',
        xaxis='x', yaxis='y'
    ), row=1, col=1)

# ── SMART MONEY TP ZONES ──────────────────────────────────────────────────────
for p in SM_TP:
    if not (PRICE_LOW < p < PRICE_HIGH):
        continue
    fig.add_shape(
        type='line', x0=0.94, x1=1.0, y0=p, y1=p,
        line=dict(color='#bd93f9', width=0.7, dash='dot'),
        xref='x', yref='y', row=1, col=1
    )
    fig.add_trace(go.Scatter(
        x=[0.996], y=[p],
        mode='markers+text',
        marker=dict(symbol='diamond', color='#bd93f9', size=5, opacity=0.9),
        text=[f'SM {p:.0f}  '],
        textposition='middle left',
        textfont=dict(color='#bd93f9', size=6, family='Courier New'),
        showlegend=False,
        hovertemplate=f'SM TP Zone: ${p:.0f}<extra></extra>',
        xaxis='x', yaxis='y'
    ), row=1, col=1)

# ── ROW 1 COL 2 — VOLUME PROFILE ─────────────────────────────────────────────
vp_colors = []
for i, v in enumerate(vol_norm):
    p = price_axis[i]
    if abs(p - POC) < RESOLUTION * 2:
        vp_colors.append('#f5c842')
    elif VAL <= p <= VAH:
        vp_colors.append('#ff6600')
    elif v > 0.65:
        vp_colors.append('#ff3300')
    elif v > 0.35:
        vp_colors.append('#aa2200')
    else:
        vp_colors.append('#2a0008')

fig.add_trace(go.Bar(
    x=vol_norm,
    y=price_axis,
    orientation='h',
    marker=dict(color=vp_colors, line=dict(width=0)),
    showlegend=False,
    hovertemplate='$%{y:.1f}<br>Volume: %{x:.3f}<extra></extra>',
    name='Volume Profile',
    xaxis='x2', yaxis='y2'
), row=1, col=2)

for ref_price, col, lw in [
    (POC,     '#f5c842', 1.8),
    (CURRENT, '#ffffff', 1.2),
    (VAH,     '#8888ff', 0.8),
    (VAL,     '#8888ff', 0.8),
]:
    if PRICE_LOW < ref_price < PRICE_HIGH:
        fig.add_shape(type='line', x0=0, x1=1, y0=ref_price, y1=ref_price,
                      line=dict(color=col, width=lw),
                      xref='x2', yref='y2', row=1, col=2)

# ── ROW 2 — CVD ───────────────────────────────────────────────────────────────
cvd_delta = np.diff(CVD, prepend=CVD[0])
cvd_cols  = ['#00cc44' if d >= 0 else '#cc2200' for d in cvd_delta]

fig.add_trace(go.Bar(
    x=t_axis, y=CVD,
    marker_color=cvd_cols, marker_line_width=0,
    showlegend=False,
    hovertemplate='CVD: %{y:,.0f}<extra></extra>',
    name='CVD', xaxis='x3', yaxis='y3'
), row=2, col=1)

fig.add_shape(type='line', x0=0, x1=1, y0=0, y1=0,
              line=dict(color='#3a6a8a', width=0.8),
              xref='x3', yref='y3', row=2, col=1)

# ── ROW 3 — VPIN ──────────────────────────────────────────────────────────────
VPIN_THRESH = 0.5
vpin_cols   = ['#ff2200' if v > VPIN_THRESH else '#1a6a9a' for v in vpin_exp]

fig.add_trace(go.Bar(
    x=t_axis, y=vpin_exp,
    marker_color=vpin_cols, marker_line_width=0,
    showlegend=False,
    hovertemplate='VPIN: %{y:.3f}<extra></extra>',
    name='VPIN', xaxis='x4', yaxis='y4'
), row=3, col=1)

fig.add_shape(type='line', x0=0, x1=1, y0=VPIN_THRESH, y1=VPIN_THRESH,
              line=dict(color='#ff8800', width=1.0, dash='dash'),
              xref='x4', yref='y4', row=3, col=1)

# ── X-AXIS TICKS (real timestamps) ───────────────────────────────────────────
n_ticks    = min(12, N_T)
tick_idx   = np.linspace(0, N_T - 1, n_ticks, dtype=int)
tick_xvals = [t_axis[i] for i in tick_idx]
tick_texts = [time_index[i].strftime('%d/%m\n%H:%M') for i in tick_idx]

# ── LEGEND ────────────────────────────────────────────────────────────────────
legend_items = [
    ('▶', '#50fa7b', 'Retail SL'),
    ('◆', '#bd93f9', 'SM TP'),
    ('━', '#f5c842', f'POC ${POC:.0f}'),
    ('━', '#00bfff', f'VWAP ${VWAP:.0f}'),
    ('━', '#ff6b6b', 'PDH/PDL'),
    ('━', '#8888ff', f'VA {VAL:.0f}–{VAH:.0f}'),
    ('━', '#ffa040', 'London H/L'),
    ('━', '#40cc80', 'Asia H/L'),
    ('▼', '#ff4444', 'Swing H'),
    ('▲', '#44ff88', 'Swing L'),
]
for i, (sym, col, lbl) in enumerate(legend_items):
    fig.add_annotation(
        x=0.001 + i * 0.098, y=1.013,
        text=f'<span style="color:{col};font-family:Courier New;font-size:8.5px">'
             f'{sym} {lbl}</span>',
        showarrow=False, xref='paper', yref='paper', xanchor='left'
    )

# ── PANEL LABELS ──────────────────────────────────────────────────────────────
for txt, xp, yp in [
    ('LOB HEATMAP  ·  LIMIT ORDER BOOK DENSITY', 0.001, 0.997),
    ('VOL PROFILE',                               0.858, 0.997),
    ('CVD  ·  CUMULATIVE VOLUME DELTA',           0.001, 0.280),
    ('VPIN  ·  INFORMED FLOW TOXICITY  (Easley, López de Prado, O\'Hara 2012)',
                                                  0.001, 0.138),
]:
    fig.add_annotation(
        x=xp, y=yp, text=txt,
        font=dict(color='#3a6a8a', size=7.5, family='Courier New'),
        showarrow=False, xref='paper', yref='paper', xanchor='left'
    )

# ═══════════════════════════════════════════════════════════════════════════════
# LAYOUT
# ═══════════════════════════════════════════════════════════════════════════════
axis_common = dict(
    showgrid=True, gridcolor='#06182a', gridwidth=0.5,
    zeroline=False, showline=False,
    tickfont=dict(size=7.5, color='#4a7a9a', family='Courier New'),
)

fig.update_layout(
    title=dict(
        text=(
            '<b style="color:#f5c842;letter-spacing:2px;font-family:Courier New">'
            'XAU/USD  ·  GOLD FUTURES  ·  LIMIT ORDER BOOK LIQUIDITY HEATMAP</b>'
            f'<br><span style="color:#3a6a8a;font-size:10px;font-family:Courier New">'
            f'GC FUTURES  ·  REAL CME DATA  ·  '
            f'CURRENT: <b style="color:#f5c842">${CURRENT:.1f}</b>  ·  '
            f'POC: <b style="color:#f5c842">${POC:.1f}</b>  ·  '
            f'VWAP: <b style="color:#00bfff">${VWAP:.1f}</b>  ·  '
            f'VAH: <b style="color:#8888ff">${VAH:.1f}</b>  ·  '
            f'VAL: <b style="color:#8888ff">${VAL:.1f}</b>  ·  '
            f'PDH: <b style="color:#ff6b6b">${PDH:.1f}</b>  ·  '
            f'PDL: <b style="color:#ff6b6b">${PDL:.1f}</b>'
            '</span>'
        ),
        font=dict(family='Courier New', size=13, color='#f5c842'),
        x=0.0, xanchor='left', y=0.99
    ),
    paper_bgcolor='#020b14',
    plot_bgcolor='#020b14',
    height=860,
    margin=dict(l=10, r=20, t=80, b=45),
    font=dict(family='Courier New', color='#3a6a8a', size=8),
    bargap=0,
    showlegend=False,

    xaxis=dict(**axis_common,
        range=[0, 1],
        tickvals=tick_xvals,
        ticktext=tick_texts,
        title=dict(text='TIME (UTC)', font=dict(size=8, color='#2a4a6a')),
    ),
    yaxis=dict(**axis_common,
        range=[PRICE_LOW, PRICE_HIGH],
        dtick=50,
        title=dict(text='PRICE  (USD / troy oz)', font=dict(size=8, color='#2a4a6a')),
    ),
    xaxis2=dict(showticklabels=False, showgrid=False, zeroline=False),
    yaxis2=dict(range=[PRICE_LOW, PRICE_HIGH], showticklabels=False,
                showgrid=False, zeroline=False),
    xaxis3=dict(**axis_common, range=[0, 1], showticklabels=False),
    yaxis3=dict(**axis_common,
        title=dict(text='CVD', font=dict(size=7, color='#2a4a6a')),
    ),
    xaxis4=dict(**axis_common, range=[0, 1], showticklabels=False),
    yaxis4=dict(**axis_common,
        range=[0, max(1.0, float(vpin_exp.max()) * 1.1)],
        title=dict(text='VPIN', font=dict(size=7, color='#2a4a6a')),
        tickvals=[0, 0.25, 0.5, 0.75, 1.0],
        ticktext=['0', '', '0.5', '', '1'],
    ),
)

fig.show()
print(f"\nData: {time_index[0].strftime('%Y-%m-%d %H:%M')} → {time_index[-1].strftime('%Y-%m-%d %H:%M')} UTC")
print(f"Bars: {N_T} × 1h | POC: ${POC:.1f} | VAH: ${VAH:.1f} | VAL: ${VAL:.1f}")
print(f"VWAP: ${VWAP:.1f} | PDH: ${PDH:.1f} | PDL: ${PDL:.1f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from matplotlib.colors import TwoSlopeNorm
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore'); unmute_convergence()

# ── CONFIG — update these every Sunday ───────────────────────────────────────
# ── Live spot price fetch (auto-updated every run) ───────────────────────
import yfinance as _yf_live
GLD_SPOT     = float(_yf_live.Ticker('GLD').fast_info['lastPrice'])
GOLD_SPOT    = float(_yf_live.Ticker('GC=F').fast_info['lastPrice'])
BASELINE_VIX = float(_yf_live.Ticker('^VIX').fast_info['lastPrice'])
RATIO        = GOLD_SPOT / GLD_SPOT
# FLIP_GOLD is model-derived; update manually or keep prior estimate
print(f"[LIVE] GLD=${GLD_SPOT:.2f}  Gold=${GOLD_SPOT:,.0f}  VIX={BASELINE_VIX:.2f}  Ratio={RATIO:.3f}")

FLIP_GOLD    = 4266.0
BASELINE_GVZ = 18.5

# ── BUILD SYNTHETIC GEX/VEX/CHARM FROM BS ────────────────────────────────────
# (replaces the "cells 46-49" data — computed fresh each run)
from scipy.stats import norm as N

def bs_greeks(S, K, T, r, sigma, flag='call'):
    if T <= 1e-6 or sigma <= 1e-6:
        return dict(gamma=0, vanna=0, charm=0, vega=0, delta=0)
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    phi = N.pdf(d1)
    gamma = phi / (S*sigma*np.sqrt(T))
    vanna = -phi*d2/sigma
    charm = -phi*(2*0.05*T - d2*sigma*np.sqrt(T))/(2*T*sigma*np.sqrt(T))
    vega  = S*phi*np.sqrt(T)/100
    delta = N.cdf(d1) if flag=='call' else N.cdf(-d1)
    return dict(gamma=gamma, vanna=vanna, charm=charm, vega=vega, delta=delta)

ATM_IV = BASELINE_GVZ / 100
R      = 0.053
T30    = 30/365
T7     = 7/365
OI_SCALE = 50000

# Strike grid in GLD terms
strikes_gld  = np.arange(360, 480, 5, dtype=float)
strikes_gold = strikes_gld * RATIO

GEX_BY_STRIKE   = {}
VEX_BY_STRIKE   = {}
CHARM_BY_STRIKE = {}

np.random.seed(42)
for K in strikes_gld:
    m = K / GLD_SPOT
    iv = ATM_IV + 0.04*(1-m)*6 if m < 1 else ATM_IV + 0.015*(m-1)*3
    iv = max(iv, 0.08)
    oi = int(np.random.lognormal(8.2, 1.1) * np.exp(-5*(m-1)**2))
    coi = max(oi + np.random.randint(-500,500), 10)
    poi = max(oi + np.random.randint(-500,500), 10)
    cg = bs_greeks(GLD_SPOT, K, T30, R, iv, 'call')
    pg = bs_greeks(GLD_SPOT, K, T7,  R, iv, 'put')
    GEX_BY_STRIKE[K]   = (cg['gamma']*coi - pg['gamma']*poi)*100*GLD_SPOT
    VEX_BY_STRIKE[K]   = (cg['vega']*coi  + pg['vega']*poi)*100
    CHARM_BY_STRIKE[K] = (cg['charm']*coi - pg['charm']*poi)*100

NET_GEX   = sum(GEX_BY_STRIKE.values())
NET_VANNA = sum(v*(bs_greeks(GLD_SPOT,k,T30,R,ATM_IV)['vanna'])
                for k,v in VEX_BY_STRIKE.items())
NET_CHARM = sum(CHARM_BY_STRIKE.values())

print(f"Net GEX   : {NET_GEX/1e6:+.2f}M")
print(f"Net Charm : {NET_CHARM/1e3:+.1f}K delta")
print(f"Net Vanna : {NET_VANNA/1e3:+.1f}K")
print(f"GEX Flip  : ${FLIP_GOLD:,.0f}")
print(f"Spot      : ${GOLD_SPOT:,.0f} — {'ABOVE' if GOLD_SPOT>FLIP_GOLD else 'BELOW'} flip → "
      f"{'LONG γ' if GOLD_SPOT>FLIP_GOLD else 'SHORT γ'}")

# ── STYLE ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#080e1a',
    'axes.facecolor':   '#080e1a',
    'text.color':       '#c8d0e0',
    'font.family':      'monospace',
    'font.size':        8,
})
GOLD_C  = '#C9A84C'
CYAN_C  = '#1abc9c'
RED_C   = '#e74c3c'
GREEN_C = '#2ecc71'
BG      = '#080e1a'

fig = plt.figure(figsize=(22, 18), facecolor=BG)
gs  = gridspec.GridSpec(2, 3, figure=fig,
                        hspace=0.48, wspace=0.35,
                        left=0.05, right=0.97,
                        top=0.93, bottom=0.05)

bias_str = 'LONG γ — BULLISH LEAN' if GOLD_SPOT > FLIP_GOLD else 'SHORT γ — BEARISH LEAN'
fig.suptitle(
    f'3D QUANT SURFACES  //  GOLD MONDAY SCENARIO  |  '
    f'Spot ${GOLD_SPOT:,.0f}   Flip ${FLIP_GOLD:,.0f}   |   {bias_str}',
    fontsize=12, color=GOLD_C, fontfamily='monospace', y=0.97
)

def style_3d(ax):
    ax.set_facecolor(BG)
    ax.tick_params(colors='#6a7490', labelsize=6)
    for pane in [ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane]:
        pane.fill = False
        pane.set_edgecolor('#111827')
    ax.grid(True, color='#111827', lw=0.3)

# =============================================================================
# SURFACE 1 — VANNA-VIX SIMULATION  (top left, wide)
# =============================================================================
ax1 = fig.add_subplot(gs[0, :2], projection='3d')
style_3d(ax1)

vix_scenarios = np.linspace(-6, 6, 50)
S_grid, V_grid = np.meshgrid(strikes_gold, vix_scenarios)
Z_vanna = np.zeros_like(S_grid)

for j, vd in enumerate(vix_scenarios):
    for i, sg in enumerate(strikes_gld):
        vex  = VEX_BY_STRIKE.get(sg, 0)
        flow = vex * (-vd / BASELINE_GVZ) * 0.01 / 1e3
        Z_vanna[j, i] = flow

norm_v   = TwoSlopeNorm(vmin=min(Z_vanna.min(),-0.01), vcenter=0,
                        vmax=max(Z_vanna.max(), 0.01))
colors_v = cm.RdYlGn(norm_v(Z_vanna))
ax1.plot_surface(S_grid, V_grid, Z_vanna,
                 facecolors=colors_v, alpha=0.88,
                 linewidth=0, antialiased=True)

flat_idx = np.argmin(np.abs(vix_scenarios))
ax1.plot(strikes_gold, [0]*len(strikes_gld), Z_vanna[flat_idx],
         color=GOLD_C, lw=2.5, zorder=10, label='VIX flat')

ax1.set_xlabel('Gold price ($)', fontsize=8, labelpad=8)
ax1.set_ylabel('VIX Δ from baseline', fontsize=8, labelpad=8)
ax1.set_zlabel('Dealer flow (K delta)', fontsize=8, labelpad=8)
ax1.set_title('Surface 1 — Vanna-VIX simulation\n'
              'Green = dealers BUY    Red = dealers SELL    '
              f'Baseline VIX {BASELINE_VIX}',
              fontsize=9, color='#aaa', pad=6)
ax1.view_init(elev=28, azim=-55)

for gold_lv, lbl, col in [
    (GOLD_SPOT,  f'SPOT\n${GOLD_SPOT:,.0f}', GOLD_C),
    (FLIP_GOLD,  f'FLIP\n${FLIP_GOLD:,.0f}', RED_C),
    (GOLD_SPOT*1.033, '$4,750\nresist',       RED_C),
    (GOLD_SPOT*0.97,  '$4,455\nsupport',      GREEN_C),
]:
    idx = np.argmin(np.abs(strikes_gold - gold_lv))
    z   = Z_vanna[flat_idx, idx]
    ax1.text(gold_lv, 0, z+1.5, lbl, color=col,
             fontsize=6.5, ha='center', fontfamily='monospace')

ax1.legend(fontsize=7, facecolor='#111', labelcolor='white', loc='upper left')

# =============================================================================
# SURFACE 2 — GAMMA WALL  (top right)
# =============================================================================
ax2 = fig.add_subplot(gs[0, 2], projection='3d')
style_3d(ax2)

gex_keys  = np.array(sorted(GEX_BY_STRIKE.keys()), dtype=float)
gex_gold  = gex_keys * RATIO
gex_vals  = np.array([GEX_BY_STRIKE[k] for k in sorted(GEX_BY_STRIKE)]) / 1e6
dist_pct  = ((gex_gold - GOLD_SPOT) / GOLD_SPOT) * 100

for gp, gv, dp in zip(gex_gold, gex_vals, dist_pct):
    xs = np.linspace(gp-25, gp+25, 6)
    ys = np.linspace(dp-0.25, dp+0.25, 6)
    Xg, Yg = np.meshgrid(xs, ys)
    Zg = np.full_like(Xg, gv)
    color = GREEN_C if gv > 0 else RED_C
    ax2.plot_surface(Xg, Yg, Zg, color=color, alpha=0.75,
                     linewidth=0, antialiased=True)

ax2.set_xlabel('Gold ($)', fontsize=7, labelpad=6)
ax2.set_ylabel('Dist from spot %', fontsize=7, labelpad=6)
ax2.set_zlabel('GEX ($M)', fontsize=7, labelpad=6)
ax2.set_title('Surface 2 — Gamma wall\n'
              'Green = long γ (suppresses)   Red = short γ (amplifies)',
              fontsize=9, color='#aaa', pad=6)
ax2.view_init(elev=30, azim=-40)

# =============================================================================
# SURFACE 3 — GRAVITY WELL  (bottom left)
# =============================================================================
ax3 = fig.add_subplot(gs[1, 0], projection='3d')
style_3d(ax3)

all_s_gld  = np.array(sorted(set(list(GEX_BY_STRIKE)+list(VEX_BY_STRIKE)+list(CHARM_BY_STRIKE))), dtype=float)
all_s_gold = all_s_gld * RATIO
time_to_open = np.linspace(60, 0, 40)

T_grid, St_grid = np.meshgrid(time_to_open, all_s_gold)
Z_grav = np.zeros_like(T_grid)

gex_max   = max(abs(v) for v in GEX_BY_STRIKE.values())
vex_max   = max(abs(v) for v in VEX_BY_STRIKE.values())
charm_max = max(abs(v) for v in CHARM_BY_STRIKE.values()) if CHARM_BY_STRIKE else 1

for i, sg in enumerate(all_s_gld):
    gn = abs(GEX_BY_STRIKE.get(sg, 0))   / gex_max
    vn = abs(VEX_BY_STRIKE.get(sg, 0))   / vex_max
    cn = abs(CHARM_BY_STRIKE.get(sg, 0)) / charm_max
    score = gn*0.40 + vn*0.35 + cn*0.25
    for j, t in enumerate(time_to_open):
        Z_grav[i, j] = score * (1 + (t/60)*0.3)

norm_g   = plt.Normalize(Z_grav.min(), Z_grav.max())
colors_g = cm.plasma(norm_g(Z_grav))
ax3.plot_surface(St_grid, T_grid, Z_grav,
                 facecolors=colors_g, alpha=0.85,
                 linewidth=0, antialiased=True)

ax3.set_xlabel('Gold ($)', fontsize=7, labelpad=6)
ax3.set_ylabel('Mins to open', fontsize=7, labelpad=6)
ax3.set_zlabel('Gravity score', fontsize=7, labelpad=6)
ax3.set_title('Surface 3 — Gravity well\n'
              'Bright = high institutional gravity   Price attracted to peaks',
              fontsize=9, color='#aaa', pad=6)
ax3.view_init(elev=32, azim=-50)

for gold_lv, lbl in [(FLIP_GOLD, f'Flip\n${FLIP_GOLD:,.0f}'),
                     (GOLD_SPOT,  f'Spot\n${GOLD_SPOT:,.0f}'),
                     (GOLD_SPOT*1.033, f'R1\n${GOLD_SPOT*1.033:,.0f}')]:
    idx = np.argmin(np.abs(all_s_gold - gold_lv))
    ax3.text(gold_lv, 5, Z_grav[idx, -1]+0.05, lbl,
             color=GOLD_C, fontsize=6.5, fontfamily='monospace')

# =============================================================================
# SURFACE 4 — CHARM DECAY  (bottom centre)
# =============================================================================
ax4 = fig.add_subplot(gs[1, 1], projection='3d')
style_3d(ax4)

charm_keys  = np.array(sorted(CHARM_BY_STRIKE.keys()), dtype=float)
charm_gold  = charm_keys * RATIO
charm_vals  = np.array([CHARM_BY_STRIKE[k] for k in sorted(CHARM_BY_STRIKE)]) / 1e3
days_range  = np.linspace(0, 3, 30)

D_grid, Cs_grid = np.meshgrid(days_range, charm_gold)
Z_charm = np.zeros_like(D_grid)
for i, cv in enumerate(charm_vals):
    for j, d in enumerate(days_range):
        Z_charm[i, j] = cv * (d / 3)

norm_c   = TwoSlopeNorm(vmin=min(Z_charm.min(),-0.01),
                        vcenter=0,
                        vmax=max(Z_charm.max(), 0.01))
colors_c = cm.RdYlGn(norm_c(Z_charm))
ax4.plot_surface(Cs_grid, D_grid, Z_charm,
                 facecolors=colors_c, alpha=0.85,
                 linewidth=0, antialiased=True)

ax4.set_xlabel('Gold ($)', fontsize=7, labelpad=6)
ax4.set_ylabel('Days (0=Fri → 3=Mon)', fontsize=7, labelpad=6)
ax4.set_zlabel('Charm flow (K delta)', fontsize=7, labelpad=6)
ax4.set_title('Surface 4 — Charm decay\n'
              'Red = sell pressure building    Green = buy\n'
              'Monday open = right edge',
              fontsize=9, color='#aaa', pad=6)
ax4.view_init(elev=28, azim=-45)

charm_dir = 'SELL' if NET_CHARM < 0 else 'BUY'
charm_col = RED_C if NET_CHARM < 0 else GREEN_C
ax4.text2D(0.05, 0.92,
           f'Total Monday charm: {NET_CHARM/1e3:+.1f}K delta\n'
           f'→ Net {charm_dir} pressure at open',
           transform=ax4.transAxes, fontsize=7,
           color=charm_col, fontfamily='monospace')

# =============================================================================
# PANEL 5 — MONDAY SCENARIO SUMMARY  (bottom right, 2D)
# =============================================================================
ax5 = fig.add_subplot(gs[1, 2])
ax5.axis('off')
ax5.set_facecolor(BG)

r1 = GOLD_SPOT * 1.033
r2 = GOLD_SPOT * 1.052
r3 = GOLD_SPOT * 1.064
s1 = GOLD_SPOT * 0.990
s2 = GOLD_SPOT * 0.972
s3 = GOLD_SPOT * 0.960
env = 'LONG γ — SUPPRESSED' if GOLD_SPOT > FLIP_GOLD else 'SHORT γ — AMPLIFIED'
env_col = GREEN_C if GOLD_SPOT > FLIP_GOLD else RED_C
bias_signals = 'BULLISH' if GOLD_SPOT > FLIP_GOLD else 'BEARISH'

lines = [
    ('MONDAY OPEN SCENARIO MAP',        GOLD_C,  9.5, True),
    ('',                                '#444',   8,   False),
    (f'Spot      : ${GOLD_SPOT:,.0f}',  '#c8d0e0', 8, False),
    (f'Flip      : ${FLIP_GOLD:,.0f}  ({((GOLD_SPOT-FLIP_GOLD)/GOLD_SPOT)*100:.1f}% below)', CYAN_C, 8, False),
    (f'Regime    : {env}',              env_col,  8,   False),
    (f'Bias      : {bias_signals}',     env_col,  8,   True),
    ('',                                '#444',   8,   False),
    ('── VIX SCENARIOS ──────────────', '#555',   7.5, False),
    ('VIX drops  → dealers BUY  → bull',  GREEN_C, 8, False),
    ('VIX flat   → neutral  → bull lean', '#c8d0e0',8,False),
    ('VIX +1-2   → mild sell → wait S1',  '#e67e22',8,False),
    ('VIX +2.5+  → SELL pressure → S2',   RED_C,   8,False),
    ('',                                '#444',   8,   False),
    ('── LEVELS ─────────────────────', '#555',   7.5, False),
    (f'R3  ${r3:,.0f}  resistance',      RED_C,    8,  False),
    (f'R2  ${r2:,.0f}  resistance',      RED_C,    8,  False),
    (f'R1  ${r1:,.0f}  resistance #1',   '#e67e22', 8, False),
    (f'S1  ${s1:,.0f}  ★ first support', GREEN_C,  8.5,True),
    (f'S2  ${s2:,.0f}  mid support',     '#c8d0e0', 8, False),
    (f'S3  ${s3:,.0f}  deep support',    '#c8d0e0', 8, False),
    ('',                                '#444',   8,   False),
    ('── OPEN MECHANICS ─────────────', '#555',   7.5, False),
    (f'Charm  : {NET_CHARM/1e3:+.1f}K delta at open', charm_col, 8, False),
    (f'Net GEX: {NET_GEX/1e6:+.1f}M (suppressed)',    GREEN_C,   8, False),
    ('',                                '#444',   8,   False),
    ('First 15 min : charm dip likely',  '#e67e22', 8, False),
    ('Then          : structural bid',   GREEN_C,   8, False),
]

y = 0.97
for text, color, size, bold in lines:
    ax5.text(0.03, y, text, transform=ax5.transAxes,
             fontsize=size, color=color,
             fontfamily='monospace',
             fontweight='bold' if bold else 'normal', va='top')
    y -= 0.033 if text == '' else 0.038

# ── Inline display ─────────────────────────────────────
import io as _io, IPython.display as _ipyd
_buf = _io.BytesIO()
plt.savefig(_buf, dpi=140, bbox_inches='tight', facecolor=BG)
plt.savefig('gold_3d_surfaces_monday.png', dpi=140,
            bbox_inches='tight', facecolor=BG)
_buf.seek(0)
_ipyd.display(_ipyd.Image(_buf.read()))
plt.close()
print("\n✓ Saved: gold_3d_surfaces_monday.png")
print(f"\n  Surface 1 — Vanna-VIX: what dealers do at every VIX move")
print(f"  Surface 2 — Gamma wall: resistance/support map in 3D")
print(f"  Surface 3 — Gravity well: institutional price attraction over time")
print(f"  Surface 4 — Charm decay: weekend theta unwinding into Monday open")
print(f"  Panel  5  — Monday scenario map: plain-language summary")


In [ ]:
# =============================================================================
# 3D QUANT SURFACE  |  Vanna-VIX Simulation + Gamma Wall + Gravity Well
# Three surfaces quants use to visualise Monday gold dealer flow
# Based on your cells 46-49 output — no refetch needed
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from matplotlib.colors import TwoSlopeNorm
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore'); unmute_convergence()

# ── Your data from cells 46-49 — update each Sunday ──────────────────────────

# ── Live spot price fetch (auto-updated every run) ───────────────────────
import yfinance as _yf_live
GLD_SPOT     = float(_yf_live.Ticker('GLD').fast_info['lastPrice'])
GOLD_SPOT    = float(_yf_live.Ticker('GC=F').fast_info['lastPrice'])
BASELINE_VIX = float(_yf_live.Ticker('^VIX').fast_info['lastPrice'])
RATIO        = GOLD_SPOT / GLD_SPOT
# FLIP_GOLD is model-derived; update manually or keep prior estimate
print(f"[LIVE] GLD=${GLD_SPOT:.2f}  Gold=${GOLD_SPOT:,.0f}  VIX={BASELINE_VIX:.2f}  Ratio={RATIO:.3f}")

try:
    BASELINE_GVZ = float(_yf_live.Ticker('^GVZ').fast_info['lastPrice'])
except Exception:
    BASELINE_GVZ = 20.0  # fallback if GVZ unavailable
print(f"[LIVE] BASELINE_GVZ={BASELINE_GVZ:.1f}")
RATIO       = LIVE_RATIO   # was hardcoded 10.906
FLIP_GLD    = 327.08
FLIP_GOLD   = 3567.0
NET_VANNA   = 12727.0
NET_GEX     = 261_255_337.0
NET_CHARM   = -817_220.0
R           = 0.036
CONTRACT    = 100

# GEX by strike — from your cell 46 output
GEX_BY_STRIKE = {
    400: -9_441_382,  405: -4_218_395,  410: -27_927_495,
    415: -20_571_392, 420:  5_000_000,  425: -48_868_483,
    430:  8_000_000,  432: 25_342_751,  433: 39_708_844,
    435: 40_606_377,  440: 33_943_835,  445: 12_000_000,
    450: 47_798_903,  455:  8_000_000,  460:  5_000_000,
}

# VEX by strike — from your cell 47 output
VEX_BY_STRIKE = {
    400:  617_500,  410: 1_227_100,  415:  789_800,
    420: -548_700,  425:  870_400,   445:  580_200,
    450: 1_690_400, 455:  549_300,   465:  585_700,
    475:  528_200,
}

# Charm by strike — from your cell 48 output
CHARM_BY_STRIKE = {
    410: -73_900,  415: -37_570,  420:  62_250,
    425: -101_620, 440: -50_080,  445: -46_690,
    450: -160_470, 455: -43_960,  460: -44_490,
    465: -36_390,
}

# ── Setup ─────────────────────────────────────────────────────────────────────
strikes_gld  = np.array(sorted(set(
    list(GEX_BY_STRIKE) + list(VEX_BY_STRIKE) + list(CHARM_BY_STRIKE)
)), dtype=float)
strikes_gold = strikes_gld * RATIO
vix_scenarios = np.linspace(-6, 6, 50)   # VIX delta range
n_s, n_v     = len(strikes_gld), len(vix_scenarios)

# ── Style ─────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#080e1a',
    'axes.facecolor':   '#080e1a',
    'text.color':       '#c8d0e0',
    'axes.labelcolor':  '#c8d0e0',
    'xtick.color':      '#6a7490',
    'ytick.color':      '#6a7490',
    'grid.color':       '#111827',
    'font.family':      'monospace',
    'font.size':        8,
})
GOLD_C  = '#C9A84C'
CYAN_C  = '#1abc9c'
RED_C   = '#e74c3c'
GREEN_C = '#2ecc71'
BLUE_C  = '#3498db'
PURP_C  = '#9b59b6'

fig = plt.figure(figsize=(22, 18))
fig.patch.set_facecolor('#080e1a')
gs  = gridspec.GridSpec(2, 3, figure=fig,
                         hspace=0.45, wspace=0.35,
                         left=0.05, right=0.97,
                         top=0.93, bottom=0.05)

fig.suptitle(
    f'3D QUANT SURFACES  //  GOLD MONDAY SCENARIO  |  '
    f'Spot ${GOLD_SPOT:,.0f}  GLD ${GLD_SPOT:.2f}  '
    f'Flip ${FLIP_GOLD:,.0f}  |  Bias: STRONGLY BULLISH',
    fontsize=11, color=GOLD_C, fontfamily='monospace', y=0.97
)

# =============================================================================
# SURFACE 1 — VANNA-VIX SIMULATION
# X = Gold strike  Y = VIX delta  Z = dealer flow (K delta)
# The key Monday surface — shows what dealers DO at every VIX move
# =============================================================================
ax1 = fig.add_subplot(gs[0, :2], projection='3d')
ax1.set_facecolor('#080e1a')

S_grid, V_grid = np.meshgrid(strikes_gold, vix_scenarios)
Z_vanna = np.zeros_like(S_grid)

for j, vdelta in enumerate(vix_scenarios):
    for i, sg in enumerate(strikes_gld):
        vex = VEX_BY_STRIKE.get(sg, 0)
        flow = vex * (-vdelta / BASELINE_GVZ) * 0.01 / 1e3
        Z_vanna[j, i] = flow

norm_v = TwoSlopeNorm(vmin=Z_vanna.min(), vcenter=0, vmax=Z_vanna.max())
colors_v = cm.RdYlGn(norm_v(Z_vanna))

surf1 = ax1.plot_surface(S_grid, V_grid, Z_vanna,
                          facecolors=colors_v, alpha=0.88,
                          linewidth=0, antialiased=True)

ax1.set_xlabel('Gold price ($)', fontsize=8, labelpad=8)
ax1.set_ylabel('VIX Δ from baseline', fontsize=8, labelpad=8)
ax1.set_zlabel('Dealer flow (K delta)', fontsize=8, labelpad=8)
ax1.set_title('Surface 1 — Vanna-VIX simulation\n'
              'Green=dealers BUY  Red=dealers SELL  '
              'Y-axis=VIX move from baseline (17.19)',
              fontsize=9, color='#aaa', pad=6)

# Monday VIX flat slice
flat_idx = np.argmin(np.abs(vix_scenarios))
ax1.plot(strikes_gold, [0]*n_s, Z_vanna[flat_idx],
         color=GOLD_C, lw=2, zorder=10, label='VIX flat (today)')

# Spot line
ax1.axvline(GOLD_SPOT, color=GOLD_C, lw=1, ls='--', alpha=0.5)

ax1.tick_params(colors='#6a7490', labelsize=7)
ax1.xaxis.pane.fill = False
ax1.yaxis.pane.fill = False
ax1.zaxis.pane.fill = False
ax1.xaxis.pane.set_edgecolor('#111827')
ax1.yaxis.pane.set_edgecolor('#111827')
ax1.zaxis.pane.set_edgecolor('#111827')
ax1.view_init(elev=28, azim=-55)

# Key level annotations on surface
for gold_lv, label, col in [
    (4628, '$4,628\nsupport ★', GREEN_C),
    (4900, '$4,900\nresistance', RED_C),
    (GOLD_SPOT, 'spot', GOLD_C),
]:
    idx = np.argmin(np.abs(strikes_gold - gold_lv))
    z   = Z_vanna[flat_idx, idx]
    ax1.text(gold_lv, 0, z + 2, label, color=col,
             fontsize=6.5, fontfamily='monospace', ha='center')

# =============================================================================
# SURFACE 2 — GAMMA WALL  (top right)
# X = Gold strike  Y = Distance from spot %  Z = GEX ($M)
# Shows the resistance/support walls in 3D — the "ceiling and floor" map
# =============================================================================
ax2 = fig.add_subplot(gs[0, 2], projection='3d')
ax2.set_facecolor('#080e1a')

gex_strikes = np.array(sorted(GEX_BY_STRIKE.keys()), dtype=float)
gex_gold    = gex_strikes * RATIO
gex_vals    = np.array([GEX_BY_STRIKE[k] for k in sorted(GEX_BY_STRIKE)]) / 1e6
dist_pct    = ((gex_gold - GOLD_SPOT) / GOLD_SPOT) * 100

# Build surface by expanding GEX bars into a mesh
width       = 0.4
X_gex, Y_gex, Z_gex = [], [], []
for i, (gp, gv, dp) in enumerate(zip(gex_gold, gex_vals, dist_pct)):
    xs = np.linspace(gp - 30, gp + 30, 8)
    ys = np.linspace(dp - 0.3, dp + 0.3, 8)
    Xg, Yg = np.meshgrid(xs, ys)
    Zg = np.full_like(Xg, gv)
    X_gex.append(Xg); Y_gex.append(Yg); Z_gex.append(Zg)

for Xg, Yg, Zg in zip(X_gex, Y_gex, Z_gex):
    color = GREEN_C if Zg[0,0] > 0 else RED_C
    ax2.plot_surface(Xg, Yg, Zg, color=color, alpha=0.75,
                     linewidth=0, antialiased=True)

ax2.axhline(0, color=GOLD_C, lw=1, ls='--', alpha=0.7)
ax2.set_xlabel('Gold ($)', fontsize=7, labelpad=6)
ax2.set_ylabel('Dist from spot %', fontsize=7, labelpad=6)
ax2.set_zlabel('GEX ($M)', fontsize=7, labelpad=6)
ax2.set_title('Surface 2 — Gamma wall\n'
              'Green=dealer long γ (suppresses)  Red=short γ (amplifies)',
              fontsize=9, color='#aaa', pad=6)
ax2.tick_params(colors='#6a7490', labelsize=6)
ax2.xaxis.pane.fill = False
ax2.yaxis.pane.fill = False
ax2.zaxis.pane.fill = False
ax2.xaxis.pane.set_edgecolor('#111827')
ax2.yaxis.pane.set_edgecolor('#111827')
ax2.zaxis.pane.set_edgecolor('#111827')
ax2.view_init(elev=30, azim=-40)

# =============================================================================
# SURFACE 3 — GRAVITY WELL  (bottom left)
# X = Gold strike  Y = Time to open (mins)  Z = Combined gravity score
# Shows price being "pulled" toward high-gravity nodes as open approaches
# =============================================================================
ax3 = fig.add_subplot(gs[1, 0], projection='3d')
ax3.set_facecolor('#080e1a')

all_strikes_gld  = np.array(sorted(set(
    list(GEX_BY_STRIKE) + list(VEX_BY_STRIKE) + list(CHARM_BY_STRIKE)
)), dtype=float)
all_strikes_gold = all_strikes_gld * RATIO
time_to_open     = np.linspace(60, 0, 40)   # 60 min to open → 0

T_grid, St_grid = np.meshgrid(time_to_open, all_strikes_gold)
Z_grav          = np.zeros_like(T_grid)

gex_max  = max(abs(v) for v in GEX_BY_STRIKE.values())
vex_max  = max(abs(v) for v in VEX_BY_STRIKE.values())
charm_max = max(abs(v) for v in CHARM_BY_STRIKE.values()) if CHARM_BY_STRIKE else 1

for i, sg in enumerate(all_strikes_gld):
    gex_n   = abs(GEX_BY_STRIKE.get(sg, 0))   / gex_max
    vex_n   = abs(VEX_BY_STRIKE.get(sg, 0))   / vex_max
    charm_n = abs(CHARM_BY_STRIKE.get(sg, 0)) / charm_max
    score   = gex_n * 0.40 + vex_n * 0.35 + charm_n * 0.25
    for j, t in enumerate(time_to_open):
        decay = 1 + (t / 60) * 0.3
        Z_grav[i, j] = score * decay

norm_g   = plt.Normalize(Z_grav.min(), Z_grav.max())
colors_g = cm.plasma(norm_g(Z_grav))
ax3.plot_surface(St_grid, T_grid, Z_grav,
                 facecolors=colors_g, alpha=0.85,
                 linewidth=0, antialiased=True)

ax3.set_xlabel('Gold ($)', fontsize=7, labelpad=6)
ax3.set_ylabel('Mins to open', fontsize=7, labelpad=6)
ax3.set_zlabel('Gravity score', fontsize=7, labelpad=6)
ax3.set_title('Surface 3 — Gravity well\n'
              'Bright=high institutional gravity  Price attracted to peaks',
              fontsize=9, color='#aaa', pad=6)
ax3.tick_params(colors='#6a7490', labelsize=6)
ax3.xaxis.pane.fill = False; ax3.yaxis.pane.fill = False; ax3.zaxis.pane.fill = False
ax3.xaxis.pane.set_edgecolor('#111827')
ax3.yaxis.pane.set_edgecolor('#111827')
ax3.zaxis.pane.set_edgecolor('#111827')
ax3.view_init(elev=32, azim=-50)

for gold_lv, label in [(4628, '★ $4,628'), (4900, '$4,900'), (4465, '$4,465')]:
    idx = np.argmin(np.abs(all_strikes_gold - gold_lv))
    ax3.text(gold_lv, 5, Z_grav[idx, -1] + 0.05, label,
             color=GOLD_C, fontsize=6.5, fontfamily='monospace')

# =============================================================================
# SURFACE 4 — CHARM DECAY CURVE  (bottom centre)
# X = Gold strike  Y = Days elapsed  Z = Cumulative charm flow
# Shows how weekend theta unwinds and creates Monday forced selling
# =============================================================================
ax4 = fig.add_subplot(gs[1, 1], projection='3d')
ax4.set_facecolor('#080e1a')

charm_strikes_gld  = np.array(sorted(CHARM_BY_STRIKE.keys()), dtype=float)
charm_strikes_gold = charm_strikes_gld * RATIO
charm_vals         = np.array([CHARM_BY_STRIKE[k] for k in sorted(CHARM_BY_STRIKE)]) / 1e3
days_range         = np.linspace(0, 3, 30)  # 0 = Friday close, 3 = Monday open

D_grid, Cs_grid = np.meshgrid(days_range, charm_strikes_gold)
Z_charm = np.zeros_like(D_grid)

for i, cv in enumerate(charm_vals):
    for j, d in enumerate(days_range):
        Z_charm[i, j] = cv * (d / 3)

norm_c   = TwoSlopeNorm(
    vmin=min(Z_charm.min(), -0.01),
    vcenter=0,
    vmax=max(Z_charm.max(), 0.01)
)
colors_c = cm.RdYlGn(norm_c(Z_charm))
ax4.plot_surface(Cs_grid, D_grid, Z_charm,
                 facecolors=colors_c, alpha=0.85,
                 linewidth=0, antialiased=True)

ax4.axhline(0, color='#444', lw=0.8)
ax4.set_xlabel('Gold ($)', fontsize=7, labelpad=6)
ax4.set_ylabel('Days (0=Fri → 3=Mon)', fontsize=7, labelpad=6)
ax4.set_zlabel('Charm flow (K delta)', fontsize=7, labelpad=6)
ax4.set_title('Surface 4 — Charm decay\n'
              'Red=sell pressure building  Green=buy  '
              'Monday open = right edge',
              fontsize=9, color='#aaa', pad=6)
ax4.tick_params(colors='#6a7490', labelsize=6)
ax4.xaxis.pane.fill = False; ax4.yaxis.pane.fill = False; ax4.zaxis.pane.fill = False
ax4.xaxis.pane.set_edgecolor('#111827')
ax4.yaxis.pane.set_edgecolor('#111827')
ax4.zaxis.pane.set_edgecolor('#111827')
ax4.view_init(elev=28, azim=-45)

# Total charm annotation
ax4.text2D(0.05, 0.92,
           f'Total Monday charm: {NET_CHARM/1e3:+.1f}K delta\n'
           f'→ Net SELL pressure at open',
           transform=ax4.transAxes, fontsize=7,
           color=RED_C, fontfamily='monospace')

# =============================================================================
# PANEL 5 — MONDAY SCENARIO SUMMARY  (bottom right — 2D text panel)
# =============================================================================
ax5 = fig.add_subplot(gs[1, 2])
ax5.axis('off')
ax5.set_facecolor('#080e1a')

lines = [
    ('MONDAY OPEN SCENARIO MAP', GOLD_C, 9.5, True),
    ('', '#444', 8, False),
    (f'Spot      : Gold ${GOLD_SPOT:,.0f}  GLD ${GLD_SPOT}', '#c8d0e0', 8, False),
    (f'Flip lvl  : ${FLIP_GOLD:,.0f}  ({((GOLD_SPOT-FLIP_GOLD)/GOLD_SPOT)*100:.0f}% below spot)', CYAN_C, 8, False),
    (f'Regime    : SUPPRESSED — dealers long γ', GREEN_C, 8, False),
    (f'Bias      : STRONGLY BULLISH (4/5 signals)', GREEN_C, 8, False),
    ('', '#444', 8, False),
    ('── VIX SCENARIOS ────────────────', '#555', 7.5, False),
    ('VIX drops  → dealers BUY  → bull', GREEN_C, 8, False),
    ('VIX flat   → neutral flow → bull lean', '#c8d0e0', 8, False),
    ('VIX +1-2   → mild sell → wait $4,628', '#e67e22', 8, False),
    ('VIX +2.5+  → SELL PRESSURE → $4,465', RED_C, 8, False),
    ('', '#444', 8, False),
    ('── LEVELS ───────────────────────', '#555', 7.5, False),
    ('$4,900  GLD $450  resistance #1', RED_C, 8, False),
    ('$4,792  GLD $440  resistance #2', RED_C, 8, False),
    ('$4,737  GLD $435  resistance #3', '#e67e22', 8, False),
    ('$4,628  GLD $425  support ★ entry', GREEN_C, 8.5, True),
    ('$4,519  GLD $415  mid support', '#c8d0e0', 8, False),
    ('$4,465  GLD $410  deep support', '#c8d0e0', 8, False),
    ('', '#444', 8, False),
    ('── OPEN MECHANICS ───────────────', '#555', 7.5, False),
    (f'Charm flow  : {NET_CHARM/1e3:+.1f}K delta at open', RED_C, 8, False),
    (f'Vanna       : {NET_VANNA/1e3:+.1f}K (VIX-sensitive)', GREEN_C, 8, False),
    (f'Net GEX     : {NET_GEX/1e6:+.1f}M (suppressed)', GREEN_C, 8, False),
    ('', '#444', 8, False),
    ('First 15 min: charm dip expected', '#e67e22', 8, False),
    ('Then:  structural bid reasserts', GREEN_C, 8, False),
]

y = 0.97
for text, color, size, bold in lines:
    weight = 'bold' if bold else 'normal'
    ax5.text(0.03, y, text, transform=ax5.transAxes,
             fontsize=size, color=color,
             fontfamily='monospace', fontweight=weight, va='top')
    y -= 0.043 if text == '' else 0.048

# ── Inline display ─────────────────────────────────────
import io as _io, IPython.display as _ipyd
_buf = _io.BytesIO()
plt.savefig(_buf, dpi=150, bbox_inches='tight', facecolor='#080e1a')
plt.savefig('gold_3d_surfaces_monday.png', dpi=150,
            bbox_inches='tight', facecolor='#080e1a')
_buf.seek(0)
_ipyd.display(_ipyd.Image(_buf.read()))
plt.close()
print("\n✓ Saved: gold_3d_surfaces_monday.png")
print("\nSurfaces built:")
print("  1. Vanna-VIX simulation — how dealer flow changes with every VIX move")
print("  2. Gamma wall           — resistance/support walls in 3D")
print("  3. Gravity well         — institutional price attraction over time")
print("  4. Charm decay          — weekend theta unwinding into Monday open")
print("  5. Monday scenario map  — plain-language summary of all four")
